# Stress tests of the proof chain "Theorems A, B, C ⇒ GRH ⇒ RH"

## What this notebook is and is not

The paper under examination claims a proof of the Generalized Riemann Hypothesis (and hence of the Riemann Hypothesis) through the chain
Theorems A, B, C ⇒ main inequality ⇒ one-sided bound ⇒ one-sided criterion ⇒ GRH ⇒ RH.

This notebook collects, in executable form, the **numerical consistency checks and falsification attempts** that were applied to individual statements of that proof, in small cases.

* Every check here is a test of **one statement** (a lemma, a proposition, a formula) in a **small instance**: small primes, small ranks, small degrees, small fields.
* Passing such a test is evidence that the statement is **not wrong in the tested cases**. It is nothing more.
* This is **not a proof**, and it is **not a verification of Theorems A, B, C in their actual parameter range**. Several hypotheses of the paper require astronomically large parameters (ranks of order $x^5$, moduli larger than $b^{20}$, degrees beyond the range `ph:eq-large`); those ranges cannot be reached numerically, and most load-bearing statements have no small instance at all.
* Every test that could have refuted a statement is listed below together with its outcome, its parameters and its limits, including the statements for which **no** numerical test exists or none was run (see the Results table, Section 5, and Section 6).
* Labels such as `ph:thm` or `au:moment` are the labels of statements in the paper; each is described in one sentence where it is tested.

**Runtime.** With the default parameters the whole notebook runs in roughly 10 minutes on a standard Colab CPU runtime. Only `numpy`, `sympy`, `mpmath` (all preinstalled on Colab) are needed. Where a script was run at a larger size than the default, the full-size setting and the recorded result are stated.

**How the code is organised.** Each test script is written to disk with `%%writefile <node>/<script>.py` exactly as it was used (unchanged except where noted), then executed in a subprocess by the helper `run(...)`, which prints the script's output. A separate cell then checks the printed output with `assert` statements and records the outcome in the results table. A `PASS` in the table therefore means: the script ran in this notebook and the assertions on its output held.

In [1]:
# Setup: dependencies (preinstalled on Colab; the pip line is a no-op there)
import importlib, subprocess, sys
for pkg in ("numpy", "sympy", "mpmath"):
    try: importlib.import_module(pkg)
    except ImportError: subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
import os, re, time, numpy, sympy, mpmath
print("python", sys.version.split()[0], "| numpy", numpy.__version__, "| sympy", sympy.__version__, "| mpmath", mpmath.__version__)
for d in ["N1","N2b","N3","N4a","N4b","N4c","N5","N6b","N7c","N8"]:
    os.makedirs(d, exist_ok=True)
T0 = time.time()
RESULTS = []   # rows of the results table, appended only by executed check cells

def run(node, *args, timeout=1500):
    '''Run a test script of a node in a fresh Python subprocess; print and return its output.'''
    t = time.time()
    p = subprocess.run([sys.executable, *map(str, args)], cwd=node, capture_output=True, text=True, timeout=timeout)
    print(p.stdout, end="")
    if p.stderr.strip(): print("[stderr]\n" + p.stderr)
    print(f"[{node}/{' '.join(map(str,args))}: exit code {p.returncode}, {time.time()-t:.1f} s]")
    assert p.returncode == 0, "script failed"
    return p.stdout

def record(node, label, what, params, outcome, limit):
    RESULTS.append(dict(node=node, label=label, what=what, params=params, outcome=outcome, limit=limit))
    print(f"RECORDED: {node} | {label} | {outcome}")

python 3.13.0 | numpy 2.5.5 | sympy 1.14.0 | mpmath 1.3.0


## 2. Map of the chain

Notation: $R_{d,q}(x)=\sum_{n\le x,\ n\equiv 1\ (q)}\Lambda(n)(1-\chi_d(n))(1/n-1/x)-\log x/\varphi(q)-c_q$.

**The top of the chain** (each link was checked by hand by the referee of the global chain; no break was found):

1. RH ⇐ GRH for $\mathrm{Re}\,s>1/2$ (`thm:GRH-half` with $q=1$, plus $\xi(s)=\xi(1-s)$).
2. GRH ⇐ one-sided criterion + one-sided bound: the criterion is applied to the modulus $q_0=18480q$ and needs $R_{d,q_0}(x)\le x^{\beta-1}$ for every prime $d$ in a fixed class.
3. The criterion (`an:thm-criterion`): Landau's lemma applied to a transform whose residues are signed orders of zeros of $L$-functions.
4. Choice of $d$ (`an:prop-density`): Heath-Brown's large sieve and Pólya–Vinogradov give a $d$ with no zeros of the twisted $L$-functions in a box.
5. One-sided bound ⇐ main inequality (`an:thm-main`) for admissible data.
6. Admissible data exist (`an:thm-data`): Siegel–Walfisz in windows, a sparse set, pigeonhole in a class group.
7. Main inequality ⇐ Theorems A, B, C (statements used and proved are identical).
8. A ⇐ inputs (I1)–(I6); B ⇐ (I6), (I7); C ⇐ slope inequality + inputs.

**The eight load-bearing nodes** (new statements on which everything rests; the referee of the global chain did *not* verify them end to end):

| Node | Statement(s) | In one line |
|---|---|---|
| N1 | `ph:thm` (+ `ph:root`, `ph:count`, `ph:av`, `pa:hecke`) | Elementary-divisor theorem: a minimally normalised Hecke operator at an inert prime is unimodular on integral forms up to a small mean exponent. |
| N2 | `au:seesaw` (+ `pz:types`, `pl:direct`) | Seesaw identity: the lower chain of theta lifts applied to a Bessel–Poincaré series equals a constant times the restriction of the upper chain, with a twist-independent constant. |
| N3 | `ps:conj` = `pk:conj`, `pf:recip`, `pf:CS` | Conjugates of jets of CM theta functions are jets at a scale $t_\sigma$ with $\lvert t_\sigma\rvert^2=1/\Omega$ exactly (a Chowla–Selberg type constant). |
| N4 | `au:moment` | Near-isometry and second moment of the Poincaré map, uniformly in rank $r\approx x^5$. |
| N5 | `au:factor`, `au:inj` | Exchange identities of Siegel–Weil type with constants given by products of $L$-values. |
| N6 | `au:onelift` | The compressed comparison operator has determinant governed by three Hecke $L$-values, independently of the eigenform. |
| N7 | `pg:I`, `pg:II`, `pg:III`, `pg:loss`, `pg:odd` | For every prime ℓ, one anticyclotomic twist with small Selmer groups, Ribet's method bounding the congruence module, local loss controlled. |
| N8 | `psu:small`, `psu:arch` | Existence of an integral test form of small valuation at every ℓ, and a lower bound for all conjugates of a norm. |

Sub-nodes with their own reports: N2b (archimedean part of N2), X1 (junction of A and B), N4a/N4b/N4c (parts of B), N6b (grid files of C), N7b/N7c (integral p-adic Hodge theory for N7).

## 3. Falsifiers of the whole argument (logical tests)

A proof of RH that does not use the Euler product would also "prove" false statements, because there are Dirichlet series with a functional equation and zeros off the critical line. Section 3(a) records where the argument uses the Euler product; 3(b) records variations of the argument that would yield false statements if the proof were insensitive to them; 3(c) is the circularity check; 3(d) the junction of A and B.

### 3(a) Functions without Euler product (Davenport–Heilbronn, Epstein)

**What the global-chain report says (analytic, not a computation).** The Euler product enters in exactly two places:

1. *The criterion.* The quantity $R_q$ and the transform $D_q(s)$ are built from logarithmic derivatives $L'/L$, i.e. from **prime sums**; the residues of the transform are (signed) orders of zeros. For a Davenport–Heilbronn or Epstein function there is no such prime sum, and $R_q$ is **not defined**.
2. *The automorphic side.* Primes enter only through `au:gains`(a): $s_{\mathrm{fin}}=\sum_p 2\log p\cdot s_p$, a sum over rational primes of $p$-adic valuations of jets of CM theta functions, and the inequality itself is the **product formula** over $\mathbb Q$ (`au:product`).

There is therefore **no formal analogue** of the argument for a function without Euler product. This falsifier is **vacuous** — the argument cannot be run on such functions — and it is **not "passed"**. It shows only that the obvious objection does not apply formally; it provides no evidence that the argument is correct.

**Illustration of why the falsifier matters.** The cells below construct the Davenport–Heilbronn function
$$f(s)=\tfrac{1-i\kappa}{2}L(s,\chi)+\tfrac{1+i\kappa}{2}L(s,\bar\chi),\qquad \kappa=\frac{\sqrt{10-2\sqrt5}-2}{\sqrt5-1},$$
with $\chi$ the character mod 5 with $\chi(2)=i$, verify its functional equation $\Lambda(s)=\Lambda(1-s)$ with $\Lambda(s)=(5/\pi)^{(s+1)/2}\Gamma(\tfrac{s+1}{2})f(s)$, locate a zero off the critical line, and show that its coefficients are not multiplicative.

In [2]:
import mpmath as mp
mp.mp.dps = 30
chi  = {0: 0, 1: 1, 2: 1j, 3: -1j, 4: -1}          # chi mod 5, chi(2)=i  (odd character)
chib = {a: complex(v).conjugate() for a, v in chi.items()}
def L(s, c):   # Dirichlet L-function via Hurwitz zeta: L(s,c) = 5^{-s} sum_a c(a) zeta(s, a/5)
    return mp.mpf(5)**(-s) * sum(c[a] * mp.zeta(s, mp.mpf(a)/5) for a in range(1, 5))
kappa = (mp.sqrt(10 - 2*mp.sqrt(5)) - 2) / (mp.sqrt(5) - 1)
def f(s):   return (1 - 1j*kappa)/2 * L(s, chi) + (1 + 1j*kappa)/2 * L(s, chib)
def Lam(s): return (mp.mpf(5)/mp.pi)**((s + 1)/2) * mp.gamma((s + 1)/2) * f(s)
print("kappa =", kappa)
worst = 0
for s in [mp.mpc(0.3, 2), mp.mpc(2, 5), mp.mpc(-1.5, 11), mp.mpc(0.8, 40)]:
    a, b = Lam(s), Lam(1 - s)
    rel = abs(a - b) / abs(a); worst = max(worst, rel)
    print(f"s = {mp.nstr(s, 6):>14}:  Lambda(s) = {mp.nstr(a, 12)},  Lambda(1-s) = {mp.nstr(b, 12)},  rel.diff = {mp.nstr(rel, 3)}")
assert worst < mp.mpf(10)**-20
print("functional equation Lambda(s) = Lambda(1-s) holds to relative precision", mp.nstr(worst, 3))

kappa = 0.284079043840412296028291832393
s =   (0.3 + 2.0j):  Lambda(s) = (0.954655003763 - 0.081205065304j),  Lambda(1-s) = (0.954655003763 - 0.081205065304j),  rel.diff = 2.53e-31
s =   (2.0 + 5.0j):  Lambda(s) = (-0.150730680927 + 0.148782931607j),  Lambda(1-s) = (-0.150730680927 + 0.148782931607j),  rel.diff = 9.09e-31
s = (-1.5 + 11.0j):  Lambda(s) = (-0.00299973910236 - 0.00824159566959j),  Lambda(1-s) = (-0.00299973910236 - 0.00824159566959j),  rel.diff = 1.69e-30
s =  (0.8 + 40.0j):  Lambda(s) = (1.01931989887e-14 - 1.12602487771e-13j),  Lambda(1-s) = (1.01931989887e-14 - 1.12602487771e-13j),  rel.diff = 5.78e-30
functional equation Lambda(s) = Lambda(1-s) holds to relative precision 5.78e-30


In [3]:
rho = mp.findroot(f, mp.mpc(0.808, 85.699))
print("zero found by findroot:", rho)
print("|f(rho)| =", mp.nstr(abs(f(rho)), 5))
print("Re(rho) - 1/2 =", mp.nstr(rho.real - 0.5, 10))
assert abs(f(rho)) < mp.mpf(10)**-20 and abs(rho.real - 0.5) > 0.3
# the functional equation forces a second zero at 1 - conj(rho), also off the line
rho2 = 1 - mp.conj(rho)
print("mirror zero 1 - conj(rho) =", mp.nstr(rho2, 15), " |f| =", mp.nstr(abs(f(rho2)), 5))
assert abs(f(rho2)) < mp.mpf(10)**-15

zero found by findroot: (0.808517182456637385553351960607 + 85.6993484853775921719292677089j)
|f(rho)| = 2.1076e-30
Re(rho) - 1/2 = 0.3085171825


mirror zero 1 - conj(rho) = (0.191482817543363 + 85.6993484853776j)  |f| = 2.9994e-29


In [4]:
# No Euler product: the Dirichlet coefficients a(n) = (1-i k)/2 chi(n) + (1+i k)/2 conj(chi(n)) are not multiplicative.
a = lambda n: (1 - 1j*kappa)/2 * chi[n % 5] + (1 + 1j*kappa)/2 * chib[n % 5]
print("a(2) =", mp.nstr(a(2), 12), "  a(3) =", mp.nstr(a(3), 12), "  a(6) =", mp.nstr(a(6), 12))
print("a(2)a(3) =", mp.nstr(a(2)*a(3), 12), " (= -kappa^2)  versus  a(6) =", mp.nstr(a(6), 12))
assert abs(a(2)*a(3) - a(6)) > 0.5
print("a(2)a(3) != a(6): f has no Euler product.")
record("global", "Euler-product falsifier", "Davenport–Heilbronn f: functional equation, zero off the line, non-multiplicative coefficients",
       "f mod 5, zero near 0.8085+85.6993i", "illustration (computed)",
       "Illustrates the falsifier only; the proof's mechanism (L'/L prime sums, product formula) has no analogue for f, so the falsifier is VACUOUS, not passed")

a(2) = (0.28407904384 + 0.0j)   a(3) = (-0.28407904384 + 0.0j)   a(6) = (1.0 + 0.0j)
a(2)a(3) = (-0.0807009031493 + 0.0j)  (= -kappa^2)  versus  a(6) = (1.0 + 0.0j)
a(2)a(3) != a(6): f has no Euler product.
RECORDED: global | Euler-product falsifier | illustration (computed)


**Conclusion of 3(a).** $f$ satisfies a functional equation of the same shape as a Dirichlet $L$-function and has zeros off the critical line, but it has no Euler product. The argument of the paper cannot be applied to $f$: $R_q$ is defined through $L'/L$ (prime sums), and on the automorphic side primes enter only through $p$-adic valuations and the product formula. So the proof's mechanism **does not apply** to $f$; this is a formal observation, not a successful numerical test of the proof.

### 3(b) Knob tests K1–K4 (analytic arguments, not computations)

Each "knob test" asks: if Theorems A, B, C held with one parameter moved, would the same deduction prove something false? If so, the proof must use that parameter at the corresponding precision somewhere, and one checks that it does. These are **analytic arguments by the referee, not computations**.

**K1 — shift of the level $N$ (analytic).**
*If* A, B, C held verbatim with $\log N$ raised by $T\approx(\log b)^2$, the excess inequality `eq:au-excess` would improve by $a\cdot T$ and the main inequality would give $R_{d,q_0}(x)\le -x^{-1/2+\varepsilon}$ for all large $x$ and every $d$ of the class.
*Why that is false:* by Landau's lemma applied to $-R$, this forces a real zero $>1/2$ of some $L(s,\psi\chi_d)$ for every such $d$, contradicting the choice of $d$ (`an:cor-choice`, unconditional via Heath-Brown).
*What blocks it:* the proofs must use the upper end of the window for $N$ at precision $\ll(\log b)^2$. They do, in one place: Theorem B (`pb:sizes`), where the admissible slack in $\log N$ is about $3\log b$ (off-identity terms $\approx(b^3/M)^{kf}D_{\mathrm{main}}$ must be $\ll D_{\mathrm{id}}$). A needs $\log N\ge m(w+G)-O(\log b)$ and B allows $\log N\le mw+(m-1)G-O(\log b)$: compatible, with a loss inside the allowance. The consistency rests on a leading-order coincidence between `pb:sizes` and `au:gains`(c) (both equal $\log(M\cdot rd_E\cdot\bar\kappa/2\pi)$), which the referee checked at leading order only.

**K2 — $m\gg\sqrt x$ (analytic).**
*If* the theorems held with $m\gg\sqrt x$, one would get $R\le x^{-1/2-\delta}$, hence central vanishing of $\prod_\psi L(1/2,\psi\chi_d)$ for every $d$ of the class — false.
*What blocks it:* two errors quadratic in $m$: $2mfb\log N$ (`au:errsum`) and $3fmb\log^2 b$ per lift (`au:onelift`(a), proof of `au:Zj`). Both are genuinely used.

**K3 — other residue class for $\Phi_a$, or another source weight with the same target (analytic).**
Gives $R_{q_0,c}(x')\le$ small for other classes and cut-offs; all such consequences are consistent with GRH, so no contradiction arises (and no evidence either way).

**K4 — internal product-formula checks (analytic).**
The A-side vectors alone, and the C-side numbers alone, are compatible with the product formula because of the factor $M^\kappa$ in $s_\infty$ ($h_{\mathrm{dir}}\ge 19\sum\kappa\log b>0$); information on the remainder arises only from the comparison of the two sides, as the paper states.

No variation produced a false statement. As the referee notes, this is evidence of careful design (the errors are such that $m\approx\sqrt x$ is optimal), **not evidence of correctness**.

### 3(c) Circularity check (from the global report; analytic)

No zero-free region, density hypothesis, or GRH is used upstream. The analytic inputs on the path are:

* Heath-Brown's large sieve for quadratic characters (1995, Cor. 2), applied as stated;
* Pólya–Vinogradov;
* de la Vallée Poussin (zero-free region of the classical, unconditional type, used only for the prime number theorem in progressions at a fixed modulus, `an:prop-pnt`);
* Siegel–Walfisz (non-effective; harmless for the argument);
* Hecke's analytic continuation, and non-vanishing on $\mathrm{Re}\,s=1$ (`po:Hpos`, proved in the text).

The ramified ratios (R2) are purely ℓ-adic statements with multiplier 1, so no lower bound for $L(1,\cdot)$ and no Siegel-zero issue enters. Siegel zeros are excluded by the conclusion (the one-sided bound for fixed $(q_0,d)$ is incompatible with a real zero in $(1/2,1)$), and the zeros of the twists on $[\beta,1)$ are removed beforehand, unconditionally, by the choice of $d$.

### 3(d) The A/B junction (node X1; analytic, no computation)

**What was found.** Theorem A and Theorem B are formulated with **two different lattices** in the hermitian space $Y_0$: A uses a lattice $\mathsf M_0$ in which the string vectors are scaled by a norm-one idele $u_0=(\mathsf a/\bar{\mathsf a})^{m/2}$ (non-trivial at the chosen place above $M$ and above $q_1$), while B is written with the lattice $L$ in which $u_0$ is replaced by 1. The two lattices have different stabilisers and different spaces of forms, and in general $u_0$ cannot be replaced by a rational norm-one element (its ideal class has no reason to vanish).

**Why it does not affect the argument.** The scaling factorises as $u_0\cdot\mathsf h$, where $u_0$ acting on all of $Y_0$ is central in the group and $\mathsf h$ ($u_0^{-1}$ on $X_0$, 1 on the string) is central in $H(\mathbb A_f)=U(X_0)(\mathbb A_f)$. Hence $\mathsf M_0=u_0\mathsf hL$, the stabilisers are conjugate by $\mathsf h$, and the two Poincaré maps correspond exactly under a unitary isomorphism of the targets commuting with the group action and a unitary automorphism of the source: $\mathcal P_{\mathsf M_0}=\mathsf j^{-1}\circ\mathcal P_L\circ R$. Statements (i)–(iii) of `au:moment` are invariant under such pre- and post-composition, so whatever B proves for $L$ holds verbatim for $\mathsf M_0$. The identification is now a lemma in the text (`pb:lattice`). This settles the interface only; it does not verify Theorem B itself.

## 4. Numerical tests, node by node

Conventions for each test: a markdown cell states the statement tested and **what outcome would have refuted it**; then the scripts are written to disk (unchanged unless noted), run, and their output checked.

### Node N1 — elementary divisors of the scaled Hecke operator (`ph:thm` and its inputs)

`ph:thm` asserts that the minimally normalised Hecke operator $p^{A}T_{\mathfrak l}$ at a prime inert in $K$ has mean elementary-divisor exponent at most a small fraction of the scaling, in the range `ph:eq-large`. **That range cannot be reached numerically** (it requires degrees $n\ge 24Nk^2cP^{2r}/\theta$ and representations of enormous dimension). The tests below check the inputs `ph:root`, `ph:depth`, `pe:vol`, `ph:count` in small cases, and the shape of the theorem's conclusion far below its range.

Helper modules: arithmetic in $\mathbb F_{p^2}$ (`fq.py`) and the representation of $\mathrm{GL}_2(\mathbb F_{p^2})$ on bihomogeneous polynomials (`rep.py`).

In [5]:
%%writefile N1/fq.py
# F_{p^2} = F_p[s]/(s^2-ns) arithmetic with tables; linear algebra (numpy, table lookups)
import numpy as np, itertools
class Fq2:
    def __init__(self,p):
        self.p=p
        ns=[a for a in range(2,p) if all((x*x-a)%p for x in range(p))][0]
        self.ns=ns; q=p*p; self.q=q
        A=np.zeros((q,q),dtype=np.int16); M=np.zeros((q,q),dtype=np.int16)
        for x in range(q):
            a,b=x%p,x//p
            for y in range(q):
                c,d=y%p,y//p
                A[x,y]=(a+c)%p+p*((b+d)%p)
                M[x,y]=(a*c+ns*b*d)%p+p*((a*d+b*c)%p)
        self.A=A;self.M=M
        self.neg=np.array([(-x%p)%p+p*((-(x//p))%p) for x in range(q)],dtype=np.int16)
        inv=np.zeros(q,dtype=np.int16)
        for x in range(1,q):
            for y in range(1,q):
                if M[x,y]==1: inv[x]=y;break
        self.inv=inv
        self.frob=np.array([x%p+p*((-(x//p))%p) for x in range(q)],dtype=np.int16)
    def add(s,x,y): return s.A[x,y]
    def mul(s,x,y): return s.M[x,y]
    def pw(s,x,e):
        r=1
        for _ in range(e): r=int(s.M[r,x])
        return r
    def matmul(s,X,Y):
        X=np.asarray(X);Y=np.asarray(Y)
        R=np.zeros((X.shape[0],Y.shape[1]),dtype=np.int16)
        for k in range(X.shape[1]):
            R=s.A[R,s.M[X[:,k][:,None],Y[k,:][None,:]]]
        return R
    def rref(s,Mx):
        Mx=np.array(Mx,dtype=np.int16);r=0;piv=[]
        nr,nc=Mx.shape
        for c in range(nc):
            if r>=nr:break
            nz=np.nonzero(Mx[r:,c])[0]
            if len(nz)==0: continue
            i=r+nz[0]
            if i!=r: Mx[[r,i]]=Mx[[i,r]]
            Mx[r]=s.M[s.inv[Mx[r,c]],Mx[r]]
            for i2 in np.nonzero(Mx[:,c])[0]:
                if i2!=r:
                    Mx[i2]=s.A[Mx[i2],s.M[s.neg[Mx[i2,c]],Mx[r]]]
            piv.append(c);r+=1
        return Mx[:r],piv
    def nullspace(s,Mx):
        # columns basis of {v: Mx v=0}
        Mx=np.asarray(Mx,dtype=np.int16)
        nc=Mx.shape[1]
        if Mx.shape[0]==0: return np.eye(nc,dtype=np.int16)
        R,piv=s.rref(Mx)
        free=[c for c in range(nc) if c not in piv]
        B=np.zeros((nc,len(free)),dtype=np.int16)
        for j,f in enumerate(free):
            B[f,j]=1
            for i,pc in enumerate(piv):
                B[pc,j]=s.neg[R[i,f]]
        return B

Overwriting N1/fq.py


In [6]:
%%writefile N1/rep.py
# k=2, N=2 representation over F_{p^2}: letters 0 (g) and 1 (Frob(g^{-T})).
import numpy as np, itertools
from fq import Fq2
def monos(n0,n1):
    return [(a,n0-a,b,n1-b) for a in range(n0+1) for b in range(n1+1)]  # exps of x10,x20,x11,x21
def polymul(F,f,g):
    r={}
    for m1,c1 in f.items():
        for m2,c2 in g.items():
            m=tuple(a+b for a,b in zip(m1,m2)); r[m]=int(F.A[r.get(m,0),F.M[c1,c2]])
    return {m:c for m,c in r.items() if c}
def polypow(F,f,e):
    r={(0,0,0,0):1}
    for _ in range(e): r=polymul(F,r,f)
    return r
def rho(F,g,n0,n1,mons=None):
    # g 2x2 over F_q (list of lists). substitution x_{c,s} -> sum_b (g_s)_{bc} x_{b,s}
    if mons is None: mons=monos(n0,n1)
    idx={m:i for i,m in enumerate(mons)}
    a,b,c,d=g[0][0],g[0][1],g[1][0],g[1][1]
    det=int(F.A[F.M[a,d],F.neg[F.M[b,c]]]); di=int(F.inv[det])
    # g^{-T} = (1/det) [[d,-c],[-b,a]]
    h=[[int(F.M[di,d]),int(F.M[di,F.neg[c]])],[int(F.M[di,F.neg[b]]),int(F.M[di,a])]]
    h=[[int(F.frob[x]) for x in row] for row in h]
    def lin(M,c,s):
        f={}
        for bb in range(2):
            if M[bb][c]:
                e=[0,0,0,0]; e[2*s+bb]=1; f[tuple(e)]=M[bb][c]
        return f
    im=[lin(g,0,0),lin(g,1,0),lin(h,0,1),lin(h,1,1)]
    cache={}
    R=np.zeros((len(mons),len(mons)),dtype=np.int16)
    for j,m in enumerate(mons):
        r={(0,0,0,0):1}
        for v in range(4):
            key=(v,m[v])
            if key not in cache: cache[key]=polypow(F,im[v],m[v])
            r=polymul(F,r,cache[key])
        for mm,cc in r.items(): R[idx[mm],j]=cc
    return R
def stable(F,gens,U):
    # largest subspace of col-span(U) stable under gens (matrices). U: columns basis
    n=U.shape[0]
    while True:
        if U.shape[1]==0: return U
        # constraints: C rows spanning annihilator of U
        C=F.nullspace(U.T).T  # rows c with c.U=0
        if C.shape[0]==0: return U
        cons=[F.matmul(F.matmul(C,g),U) for g in gens]
        K=F.nullspace(np.vstack(cons))
        if K.shape[1]==U.shape[1]: return U
        U=F.matmul(U,K)

Overwriting N1/rep.py


**T1 + T2 (k = 2): `ph:root` and `ph:depth`.** `ph:root` (root lemma over the residue field): in the largest $\mathrm{SL}_2(\mathbb F_{p^2})$-stable subspace spanned by monomials of weight $\ge t$, the monomials of least weight all have both defects $\mathrm{def}_0,\mathrm{def}_1\le 0$. `ph:depth`: $\sum_\delta\dim D_\delta\le (n/k)(N/2)((k-2r)/P^{2r}+2r/P)\lvert\mathfrak B\rvert$.
Default sizes here: bidegrees $\\le 8$ ($p=5$) and $\\le 7$ ($p=3$), about 1 minute; the report's full size ($\\le11$, $\\le9$) is available with `N1_T1_FULL = True` (5–8 minutes; recorded result: 2014 + 2077 lowest monomials, 0 violations, every depth sum within the bound).
*Refutation would be:* a printed `VIOLATION` line (a least-weight monomial with a positive defect), or a degree $n$ printed with `FAIL` (depth sum above the bound). Parameters: $k=2$, $N=2$, $r=1$; $p=5$ with bidegrees $\le 11$, and $p=3$ with bidegrees $\le 9$; five weight systems; every threshold.

In [7]:
N1_T1_FULL = False   # True: bidegrees <= 11 (p=5) and <= 9 (p=3), as in the report (about 5-8 minutes)
N1_T1_DEG = (11, 9) if N1_T1_FULL else (8, 7)

In [8]:
%%writefile N1/t1_root.py
# Test root lemma (R=k) and depth count for k=2,N=2,r=1.
import sys, numpy as np
from fractions import Fraction as Fr
from fq import Fq2
from rep import *
p=int(sys.argv[1]); nmax=int(sys.argv[2])
F=Fq2(p); P=p; q=p*p
# generator of F_q^*
for z in range(2,q):
    if len({F.pw(z,e) for e in range(1,q)})==q-1: break
gensGL=[[[z,0],[0,1]],[[1,1],[0,1]],[[0,1],[1,0]]]
zi=int(F.inv[z])
gensSL=[[[z,0],[0,zi]],[[1,1],[0,1]],[[0,1],[F.neg[1],0]],[[1,0],[1,1]]]
def defect(m,i):
    # letters: s=0: X=x1,Y=x2 ; s=1: X=x2,Y=x1.  m=(x10,x20,x11,x21)
    Y=[m[1],m[2]]; X=[m[0],m[3]]
    tot=sum(Fr(P**((s-i)%2))*(Y[s]-X[s]) for s in range(2))
    return Y[i]+tot/(q-1)
bad=0; tot=0
for n0 in range(0,nmax+1):
  for n1 in range(0,nmax+1):
    mons=monos(n0,n1)
    G=[rho(F,g,n0,n1,mons) for g in gensSL]
    for wts in [(1,1),(Fr(1,2),1),(1,Fr(1,2)),(Fr(1,2),Fr(1,2)),(Fr(3,4),Fr(2,3))]:
        cw=[wts[0]*m[1]+wts[1]*m[2] for m in mons]
        for t in sorted(set(cw)):
            sel=[i for i in range(len(mons)) if cw[i]>=t]
            U=np.zeros((len(mons),len(sel)),dtype=np.int16)
            for j,i in enumerate(sel): U[i,j]=1
            S=stable(F,G,U)
            if S.shape[1]==0: continue
            for i in range(len(mons)):
                if cw[i]==t and S[i].any():
                    tot+=1
                    d0,d1=defect(mons[i],0),defect(mons[i],1)
                    if d0>0 or d1>0:
                        bad+=1; print("VIOLATION",n0,n1,wts,t,mons[i],d0,d1)
print("root lemma: lowest monomials checked",tot,"violations",bad)
# depth count: D_delta for GL2, degrees (n,n)
for n in range(1,nmax+1):
    mons=monos(n,n)
    G=[rho(F,g,n,n,mons) for g in gensGL]
    dp=[m[1]+m[2] for m in mons]
    s=0; dims=[]
    for dl in range(1,2*n+1):
        sel=[i for i in range(len(mons)) if dp[i]>=dl]
        U=np.zeros((len(mons),len(sel)),dtype=np.int16)
        for j,i in enumerate(sel): U[i,j]=1
        S=stable(F,G,U); dims.append(S.shape[1]); s+=S.shape[1]
        if S.shape[1]==0: break
    bound=Fr(n,2)*Fr(2,P)*len(mons)   # (n/k)*(N/2)*((k-2r)/P^{2r}+2r/P)|B| with k=2,r=1,N=2
    cdiv=2*n*(Fr(1,P*P)+Fr(1,2*P))*len(mons)
    print("n",n,"|B|",len(mons),"dims D_delta",dims,"sum",s,"sharp bound",float(bound),"cdiv|B|",float(cdiv),"OK" if s<=bound else "FAIL")

Overwriting N1/t1_root.py


In [9]:
o1 = run("N1", "t1_root.py", 5, N1_T1_DEG[0])
o2 = run("N1", "t1_root.py", 3, N1_T1_DEG[1])
for o in (o1, o2):
    assert "VIOLATION" not in o and "FAIL" not in o
    assert re.search(r"violations 0\b", o)
tot = [int(re.search(r"checked (\d+)", o).group(1)) for o in (o1, o2)]
print("lowest monomials checked:", tot)
prm = f"k=2,N=2; p=5 deg<={N1_T1_DEG[0]}, p=3 deg<={N1_T1_DEG[1]}"
record("N1", "ph:root", "least-weight monomials of largest stable subspace have def<=0", prm, f"PASS ({tot[0]}+{tot[1]} cases, 0 violations)", "residue field only, k=2")
record("N1", "ph:depth (k=2)", "sum of dims of depth subspaces <= sharp bound", prm, "PASS", "small n, far below ph:eq-large")
if not N1_T1_FULL:
    record("N1", "ph:root, ph:depth (full size)", "same", "p=5 deg<=11, p=3 deg<=9", "recorded only (2014+2077 cases, 0 violations; all depth sums OK; also re-run at full size when this notebook was prepared: same)", "set N1_T1_FULL=True to re-run")

root lemma: lowest monomials checked 791 violations 0
n 1 |B| 4 dims D_delta [0] sum 0 sharp bound 0.8 cdiv|B| 1.12 OK
n 2 |B| 9 dims D_delta [0] sum 0 sharp bound 3.6 cdiv|B| 5.04 OK
n 3 |B| 16 dims D_delta [0] sum 0 sharp bound 9.6 cdiv|B| 13.44 OK
n 4 |B| 25 dims D_delta [0] sum 0 sharp bound 20.0 cdiv|B| 28.0 OK
n 5 |B| 36 dims D_delta [10, 0] sum 10 sharp bound 36.0 cdiv|B| 50.4 OK
n 6 |B| 49 dims D_delta [23, 1, 0] sum 24 sharp bound 58.8 cdiv|B| 82.32 OK
n 7 |B| 64 dims D_delta [38, 4, 0] sum 42 sharp bound 89.6 cdiv|B| 125.44 OK
n 8 |B| 81 dims D_delta [55, 9, 0] sum 64 sharp bound 129.6 cdiv|B| 181.44 OK
[N1/t1_root.py 5 8: exit code 0, 35.8 s]


root lemma: lowest monomials checked 966 violations 0
n 1 |B| 4 dims D_delta [0] sum 0 sharp bound 1.3333333333333333 cdiv|B| 2.2222222222222223 OK
n 2 |B| 9 dims D_delta [0] sum 0 sharp bound 6.0 cdiv|B| 10.0 OK
n 3 |B| 16 dims D_delta [6, 0] sum 6 sharp bound 16.0 cdiv|B| 26.666666666666668 OK
n 4 |B| 25 dims D_delta [15, 1, 0] sum 16 sharp bound 33.333333333333336 cdiv|B| 55.55555555555556 OK
n 5 |B| 36 dims D_delta [26, 6, 0] sum 32 sharp bound 60.0 cdiv|B| 100.0 OK
n 6 |B| 49 dims D_delta [39, 19, 0] sum 58 sharp bound 98.0 cdiv|B| 163.33333333333334 OK
n 7 |B| 64 dims D_delta [54, 34, 6, 0] sum 94 sharp bound 149.33333333333334 cdiv|B| 248.88888888888889 OK
[N1/t1_root.py 3 7: exit code 0, 14.5 s]
lowest monomials checked: [791, 966]
RECORDED: N1 | ph:root | PASS (791+966 cases, 0 violations)
RECORDED: N1 | ph:depth (k=2) | PASS
RECORDED: N1 | ph:root, ph:depth (full size) | recorded only (2014+2077 cases, 0 violations; all depth sums OK; also re-run at full size when this notebo

**T2 (k = 3, 4): `ph:depth` for larger $k$.** Same bound, exact computation of the largest $\mathrm{GL}_k(\mathbb F_{p^2})$-stable subspaces. Cases: $(p,k,r,n)=(3,3,1,4..5)$, $(5,3,1,5)$, $(3,4,2,3)$ (the case $k=2r$), $(3,4,1,3)$. *Refutation would be:* a line ending in `FAIL`. Each run takes about 30–60 s.

In [10]:
%%writefile N1/t2_depth_k3.py
# Depth count (Proposition ph:depth) for general k, N=2, over F_{p^2}: sum_delta dim D_delta <= (n/k)(N/2)((k-2r)/P^{2r}+2r/P)|B| <= c_div|B|.
import sys, itertools, numpy as np
from fractions import Fraction as Fr
from fq import Fq2
from rep import stable
p=int(sys.argv[1]); k=int(sys.argv[2]); r=int(sys.argv[3]); ns=[int(x) for x in sys.argv[4].split(',')]
F=Fq2(p); P=p; q=p*p
def comps(n,k):
    if k==1: yield (n,); return
    for a in range(n+1):
        for rest in comps(n-a,k-1): yield (a,)+rest
def pmul(f,g):
    r_={}
    for m1,c1 in f.items():
        for m2,c2 in g.items():
            m=tuple(a+b for a,b in zip(m1,m2)); v=int(F.A[r_.get(m,0),F.M[c1,c2]])
            if v: r_[m]=v
            elif m in r_: del r_[m]
    return r_
def matinv(g):
    n=len(g); M=np.array([list(g[i])+[1 if i==j else 0 for j in range(n)] for i in range(n)],dtype=np.int16)
    R,piv=F.rref(M); return [[int(R[i,n+j]) for j in range(n)] for i in range(n)]
def rho(g,n,mons,idx):
    gi=matinv(g)
    g1=[[int(F.frob[gi[j][i]]) for j in range(k)] for i in range(k)]   # Frob(g^{-T})
    zero=tuple([0]*(2*k))
    def lin(M,c,s):
        f={}
        for b in range(k):
            if M[b][c]:
                e=[0]*(2*k); e[s*k+b]=1; f[tuple(e)]=M[b][c]
        return f
    im=[lin(g,c,0) for c in range(k)]+[lin(g1,c,1) for c in range(k)]
    cache={}
    def pw(v,e):
        if (v,e) not in cache:
            cache[(v,e)]={zero:1} if e==0 else pmul(pw(v,e-1),im[v])
        return cache[(v,e)]
    R=np.zeros((len(mons),len(mons)),dtype=np.int16)
    for j,m in enumerate(mons):
        r_={zero:1}
        for v in range(2*k):
            if m[v]: r_=pmul(r_,pw(v,m[v]))
        for mm,cc in r_.items(): R[idx[mm],j]=cc
    return R
for z in range(2,q):
    if len({F.pw(z,e) for e in range(1,q)})==q-1: break
I=[[1 if i==j else 0 for j in range(k)] for i in range(k)]
g1=[row[:] for row in I]; g1[0][0]=z
g2=[row[:] for row in I]; g2[0][1]=1
g3=[[1 if j==(i+1)%k else 0 for j in range(k)] for i in range(k)]
g4=[row[:] for row in I]; g4[0],g4[1]=g4[1],g4[0]
for n in ns:
    mons=[a+b for a in comps(n,k) for b in comps(n,k)]; idx={m:i for i,m in enumerate(mons)}
    G=[rho(g,n,mons,idx) for g in (g1,g2,g3,g4)]
    dp=[sum(m[b] for b in range(r,k))+sum(m[k+b] for b in range(r)) for m in mons]
    dims=[];s=0
    for dl in range(1,2*n+1):
        sel=[i for i in range(len(mons)) if dp[i]>=dl]
        U=np.zeros((len(mons),len(sel)),dtype=np.int16)
        for j,i in enumerate(sel): U[i,j]=1
        S=stable(F,G,U); dims.append(S.shape[1]); s+=S.shape[1]
        if S.shape[1]==0: break
    bound=Fr(n,k)*(Fr(k-2*r,P**(2*r))+Fr(2*r,P))*len(mons)
    cdiv=2*n*(Fr(1,P**(2*r))+Fr(r*r,k*P))*len(mons)
    print(f"p={p} k={k} r={r} n={n} |B|={len(mons)} dims D_delta={dims} sum={s} sharp bound={float(bound):.1f} c_div|B|={float(cdiv):.1f}", "OK" if s<=bound else "FAIL",flush=True)

Overwriting N1/t2_depth_k3.py


In [11]:
outs = [run("N1", "t2_depth_k3.py", *a) for a in [(3,3,1,"4,5"), (5,3,1,"5"), (3,4,2,"3"), (3,4,1,"3")]]
lines = [l for o in outs for l in o.splitlines() if l.startswith("p=")]
assert len(lines) == 5 and all(l.rstrip().endswith("OK") for l in lines)
record("N1", "ph:depth (k=3,4)", "depth sum <= sharp bound", "(p,k,r,n)=(3,3,1,4-5),(5,3,1,5),(3,4,2,3),(3,4,1,3)", "PASS (5/5)", "n<=5, far below ph:eq-large")

p=3 k=3 r=1 n=4 |B|=225 dims D_delta=[59, 1, 0] sum=60 sharp bound=233.3 c_div|B|=400.0 OK
p=3 k=3 r=1 n=5 |B|=441 dims D_delta=[171, 9, 0] sum=180 sharp bound=571.7 c_div|B|=980.0 OK
[N1/t2_depth_k3.py 3 3 1 4,5: exit code 0, 34.0 s]


p=5 k=3 r=1 n=5 |B|=441 dims D_delta=[30, 0] sum=30 sharp bound=323.4 c_div|B|=470.4 OK
[N1/t2_depth_k3.py 5 3 1 5: exit code 0, 17.4 s]


p=3 k=4 r=2 n=3 |B|=400 dims D_delta=[20, 0] sum=20 sharp bound=400.0 c_div|B|=829.6 OK
[N1/t2_depth_k3.py 3 4 2 3: exit code 0, 17.1 s]


p=3 k=4 r=1 n=3 |B|=400 dims D_delta=[20, 0] sum=20 sharp bound=266.7 c_div|B|=466.7 OK
[N1/t2_depth_k3.py 3 4 1 3: exit code 0, 15.0 s]
RECORDED: N1 | ph:depth (k=3,4) | PASS (5/5)


**T3: `pe:vol` by Monte Carlo.** `pe:vol`: the proportion of the simplex $\Sigma_D$ lying in the cone $C$ (all defects $\le 0$) equals $\prod_{l=1}^k (q-1)/(q^l-1)$, for any degrees $D$. *Refutation would be:* a Monte-Carlo share differing from the prediction by many standard errors. The script prints the share, its standard error, and the prediction; the check below requires agreement within 4 standard errors.

In [12]:
%%writefile N1/t3_vol.py
# Monte Carlo test of Lemma pe:vol: vol(C cap Sigma_D)/vol(Sigma_D) = prod_{l=1}^k (q-1)/(q^l-1), any degrees D.
import numpy as np
rng=np.random.default_rng(1)
def test(k,N,P,D,M=4_000_000):
    q=P**N
    # beta[s][b]: uniform on simplex of size D_s (thin shell limit)
    beta=np.stack([rng.dirichlet(np.ones(k),size=M)*D[s] for s in range(N)],axis=1)  # M x N x k
    ok=np.ones(M,dtype=bool)
    for b in range(k-1):
        a,a2=b,b+1
        X=np.stack([beta[:,s,a] if s%2==0 else beta[:,s,a2] for s in range(N)],axis=1)
        Y=np.stack([beta[:,s,a2] if s%2==0 else beta[:,s,a] for s in range(N)],axis=1)
        for i in range(N):
            tot=sum(P**((s-i)%N)*(Y[:,s]-X[:,s]) for s in range(N))
            ok&=(Y[:,i]+tot/(q-1)<=0)
    share=ok.mean(); pred=np.prod([(q-1)/(q**l-1) for l in range(1,k+1)])
    print(f"k={k} N={N} P={P} D={D}: MC share={share:.6f} +- {np.sqrt(pred/M):.6f}  predicted={pred:.6f}")
test(2,2,5,[1,1]); test(2,2,5,[1,3.7]); test(2,4,2,[1,1,1,1]); test(2,4,2,[1,2,0.5,3])
test(3,2,2,[1,1]); test(3,2,2,[2,0.7]); test(3,4,2,[1,1,1,1],M=6_000_000); test(4,2,2,[1,1.5],M=8_000_000); test(3,2,3,[1,1],M=8_000_000)

Overwriting N1/t3_vol.py


In [13]:
o = run("N1", "t3_vol.py")
rows = re.findall(r"share=([\d.]+) \+- ([\d.]+)\s+predicted=([\d.]+)", o)
devs = [abs(float(s)-float(p))/float(e) for s, e, p in rows]
print("deviations in standard errors:", [round(d, 2) for d in devs])
assert len(rows) == 9 and max(devs) < 4
record("N1", "pe:vol", "Monte Carlo cone share = prod (q-1)/(q^l-1)", "9 cases (k,N,P) up to (4,2,2), 4-8e6 samples", f"PASS (max dev {max(devs):.1f} s.e.)", "statistical; small k")

k=2 N=2 P=5 D=[1, 1]: MC share=0.038658 +- 0.000098  predicted=0.038462
k=2 N=2 P=5 D=[1, 3.7]: MC share=0.038371 +- 0.000098  predicted=0.038462
k=2 N=4 P=2 D=[1, 1, 1, 1]: MC share=0.058852 +- 0.000121  predicted=0.058824
k=2 N=4 P=2 D=[1, 2, 0.5, 3]: MC share=0.058825 +- 0.000121  predicted=0.058824
k=3 N=2 P=2 D=[1, 1]: MC share=0.009536 +- 0.000049  predicted=0.009524
k=3 N=2 P=2 D=[2, 0.7]: MC share=0.009549 +- 0.000049  predicted=0.009524
k=3 N=4 P=2 D=[1, 1, 1, 1]: MC share=0.000224 +- 0.000006  predicted=0.000215
k=4 N=2 P=2 D=[1, 1.5]: MC share=0.000116 +- 0.000004  predicted=0.000112
k=3 N=2 P=3 D=[1, 1]: MC share=0.001112 +- 0.000012  predicted=0.001099
[N1/t3_vol.py: exit code 0, 6.5 s]
deviations in standard errors: [2.0, 0.93, 0.23, 0.01, 0.24, 0.51, 1.5, 1.0, 1.08]
RECORDED: N1 | pe:vol | PASS (max dev 2.0 s.e.)


**T4: Smith normal form of the edge Hecke operator (`ph:eq-A`), the shape of `ph:thm`.** The edge operator $A$ for $k=2$, $r=1$, $N=2$, one vertex, 26 edges, over $\mathbb Z_{25}$ (symmetric edge data, trial 0, and randomised edge data, trial 1). The theorem's claim is that the scaled operator is unimodular up to a mean exponent $O(nN/P)$, against the scaling $nN/2=n$; the script prints the mean exponent next to $n/P$ and $3c_{\mathrm{div}}=0.84n$.
*Refutation (of the shape of the claim) would be:* mean exponents growing like the scaling $n$ rather than like $n/P$. **These $n$ are far below the range `ph:eq-large`, where the theorem is not asserted**; the single case $n=1$ with symmetric data has mean $1.0>0.84$, which is outside every hypothesis and not a counterexample.

Default sizes: $n\in\{1,2,4,8,12\}$ (about 20 s). Full size used in the report: $n\le 24$ (matrices up to $625\times625$, several minutes each); set `N1_T4_FULL = True` to run $n=16,20,24$ as well. Recorded full-size result (not re-run by default): mean exponents $n=16$: 2.64/3.08; $n=20$: 3.14/3.83; $n=24$: 3.81/4.93.

In [14]:
N1_T4_FULL = False      # True: also n = 16, 20, 24 (several minutes each)
N1_T4_NS = "1,2,4,8,12" + (",16,20,24" if N1_T4_FULL else "")

In [15]:
%%writefile N1/t4_snf.py
# Elementary divisors of the edge Hecke operator A (ph:eq-A) for k=2, r=1, N=2, |X|=1, H trivial,
# W = Z_{p^2} (unramified), iota_0 = id, iota_1 = Frobenius. A = sum_kappa ^t( rho(gamma_k) Delta^2 rho(t_k) ).
import sys, random
from fractions import Fraction as Fr
from math import comb
p=int(sys.argv[1]); nlist=[int(x) for x in sys.argv[2].split(',')]; seed=int(sys.argv[3]) if len(sys.argv)>3 else 0
random.seed(seed)
ns=[a for a in range(2,p) if all((x*x-a)%p for x in range(p))][0]
class R:
    J=40; MOD=p**40
    __slots__=('a','b')
    def __init__(s,a,b=0): s.a=a%R.MOD; s.b=b%R.MOD
    def __add__(s,o): return R(s.a+o.a,s.b+o.b)
    def __sub__(s,o): return R(s.a-o.a,s.b-o.b)
    def __neg__(s): return R(-s.a,-s.b)
    def __mul__(s,o): return R(s.a*o.a+ns*s.b*o.b, s.a*o.b+s.b*o.a)
    def conj(s): return R(s.a,-s.b)
    def iszero(s): return s.a==0 and s.b==0
    def val(s):
        if s.iszero(): return R.J
        v=0;a,b=s.a,s.b
        while a%p==0 and b%p==0: a//=p;b//=p;v+=1
        return v
    def inv(s):
        nrm=(s.a*s.a-ns*s.b*s.b)%R.MOD
        ni=pow(nrm,-1,R.MOD)
        return R(s.a*ni,-s.b*ni)
    def divp(s,v): return R(s.a//p**v, s.b//p**v) if v else s
ONE=R(1);ZERO=R(0)
def polymul(f,g):
    r=[ZERO]*(len(f)+len(g)-1)
    for i,x in enumerate(f):
        for j,y in enumerate(g): r[i+j]=r[i+j]+x*y
    return r
def sym(M,n):
    # matrix on coefficient vectors (index beta = exponent of x1) of x_c -> sum_b M[b][c] x_b
    # x1 -> M11 x1 + M21 x2 ; x2 -> M12 x1 + M22 x2 ; poly in (x1,x2) homogeneous: list by power of x1
    l1=[M[1][0],M[0][0]]; l2=[M[1][1],M[0][1]]   # index = power of x1
    pw1=[[ONE]]; pw2=[[ONE]]
    for i in range(n): pw1.append(polymul(pw1[-1],l1)); pw2.append(polymul(pw2[-1],l2))
    S=[[ZERO]*(n+1) for _ in range(n+1)]
    for g in range(n+1):
        col=polymul(pw1[g],pw2[n-g])
        for b in range(n+1): S[b][g]=col[b]
    return S
def kron(A,B):
    n=len(A);m=len(B)
    return [[A[i][j]*B[k][l] for j in range(n) for l in range(m)] for i in range(n) for k in range(m)]
def mm(A,B):
    n=len(A);m=len(B[0]);k=len(B)
    return [[sum((A[i][t]*B[t][j] for t in range(k)),ZERO) for j in range(m)] for i in range(n)]
def tr(A): return [list(r) for r in zip(*A)]
def inv2(g):
    a,b,c,d=g[0][0],g[0][1],g[1][0],g[1][1]
    di=(a*d-b*c).inv()
    return [[d*di,-(b*di)],[-(c*di),a*di]]
def rho2(g,n):
    g0=g
    gi=inv2(g); g1=[[gi[0][0].conj(),gi[1][0].conj()],[gi[0][1].conj(),gi[1][1].conj()]]  # Frob(g^{-T})
    return sym(g0,n),sym(g1,n)
def snf_vals(A):
    A=[row[:] for row in A]; n=len(A); vals=[]
    for s in range(n):
        best=None;bv=R.J+1
        for i in range(s,n):
            for j in range(s,n):
                v=A[i][j].val()
                if v<bv: bv=v;best=(i,j)
                if bv==0: break
            if bv==0: break
        vals.append(bv)
        if bv>=R.J: vals+= [R.J]*(n-s-1); break
        i,j=best
        A[s],A[i]=A[i],A[s]
        for row in A: row[s],row[j]=row[j],row[s]
        pinv=A[s][s].divp(bv).inv()
        for i in range(s+1,n):
            if not A[i][s].iszero():
                f=A[i][s].divp(bv)*pinv
                Ai=A[i];As=A[s]
                for j in range(s,n): Ai[j]=Ai[j]-f*As[j]
    return vals
# residue reps of F_{p^2}
res=[R(a,b) for a in range(p) for b in range(p)]
def rnd_lift(x): return R(x.a+p*random.randrange(p**3), x.b+p*random.randrange(p**3))
# G/K_r, K_r = {g: g_12 in m}: cosets <-> lines g e_2: reps [[1,a],[0,1]] and [[0,1],[1,0]]
betas=[[[ONE,rnd_lift(x)],[ZERO,ONE]] for x in res]+[[[R(p*random.randrange(p)),ONE],[ONE,ZERO]]]
for n in nlist:
    d=(n+1)**2
    # exponent index: (beta0,beta1) = powers of x_{1,0}, x_{1,1}. deep: x_{2,0} and x_{1,1}
    dp=[(n-b0)+b1 for b0 in range(n+1) for b1 in range(n+1)]
    out=[]
    for trial in range(2):
        perm=list(range(len(betas)))
        if trial==1: random.shuffle(perm)
        A=[[ZERO]*d for _ in range(d)]
        for kk,be in enumerate(betas):
            t=inv2(be)                       # t(kappa)=beta^{-1}
            gam=tr(inv2(betas[perm[kk]]))    # gamma with ^t gamma^{-1} K = distinct cosets
            if trial==1:                      # multiply gamma on the right by v with ^t v in K_r
                v=[[ONE,ZERO],[R(random.randrange(p**3),random.randrange(p**3)),ONE]]
                gam=mm(gam,v)
            a0,a1=rho2(gam,n); b0,b1=rho2(t,n)
            M=kron(mm(a0,[[x*R(p**(n-i)) for x in row] for i,row in enumerate(b0)]),mm(a1,[[x*R(p**i) for x in row] for i,row in enumerate(b1)]))
            Mt=tr(M)
            for i in range(d):
                Ai=A[i];Mi=Mt[i]
                for j in range(d): Ai[j]=Ai[j]+Mi[j]
        vals=snf_vals(A)
        out.append(sum(vals))
        hist={}
        for v in vals: hist[v]=hist.get(v,0)+1
        cdiv=2*n*(Fr(1,p*p)+Fr(1,2*p))
        print(f"p={p} n={n} |B|={d} trial={trial} sum exps={sum(vals)} mean={sum(vals)/d:.4f} hist={sorted(hist.items())}  n/P={n/p:.3f} 3cdiv={3*float(cdiv):.3f} scaling nN/2={n}",flush=True)

Overwriting N1/t4_snf.py


In [16]:
o = run("N1", "t4_snf.py", 5, N1_T4_NS)
rows = [(int(n), float(m), float(c)) for n, m, c in re.findall(r"n=(\d+) \|B\|=\d+ trial=\d sum exps=\d+ mean=([\d.]+).*?3cdiv=([\d.]+)", o)]
for n, m, c in rows: print(f"n={n:2d}  mean exponent {m:.3f}   n/P={n/5:.2f}   3c_div={c:.2f}   scaling n={n}")
assert all(m <= c for n, m, c in rows if n >= 2)          # below 3 c_div for every n >= 2
assert all(m <= n/4 + 1 for n, m, c in rows)               # grows like n/P, not like n
record("N1", "ph:thm (shape)", "mean elementary-divisor exponent of edge Hecke operator ~ n/P, below 3c_div", f"p=5,k=2,r=1,N=2, n in {{{N1_T4_NS}}}", "PASS",
       "far below ph:eq-large where the theorem is asserted; one vertex; ph:thm itself is NOT testable in its range")
record("N1", "ph:thm (shape), large n", "same, n=16,20,24", "p=5, matrices up to 625x625", "recorded only (means 2.64/3.08, 3.14/3.83, 3.81/4.93)" if not N1_T4_FULL else "PASS", "far below ph:eq-large")

p=5 n=1 |B|=4 trial=0 sum exps=4 mean=1.0000 hist=[(0, 1), (1, 2), (2, 1)]  n/P=0.200 3cdiv=0.840 scaling nN/2=1
p=5 n=1 |B|=4 trial=1 sum exps=0 mean=0.0000 hist=[(0, 4)]  n/P=0.200 3cdiv=0.840 scaling nN/2=1
p=5 n=2 |B|=9 trial=0 sum exps=9 mean=1.0000 hist=[(1, 9)]  n/P=0.400 3cdiv=1.680 scaling nN/2=2
p=5 n=2 |B|=9 trial=1 sum exps=0 mean=0.0000 hist=[(0, 9)]  n/P=0.400 3cdiv=1.680 scaling nN/2=2
p=5 n=4 |B|=25 trial=0 sum exps=0 mean=0.0000 hist=[(0, 25)]  n/P=0.800 3cdiv=3.360 scaling nN/2=4
p=5 n=4 |B|=25 trial=1 sum exps=11 mean=0.4400 hist=[(0, 15), (1, 9), (2, 1)]  n/P=0.800 3cdiv=3.360 scaling nN/2=4
p=5 n=8 |B|=81 trial=0 sum exps=91 mean=1.1235 hist=[(0, 25), (1, 27), (2, 23), (3, 6)]  n/P=1.600 3cdiv=6.720 scaling nN/2=8
p=5 n=8 |B|=81 trial=1 sum exps=112 mean=1.3827 hist=[(0, 17), (1, 34), (2, 13), (3, 16), (4, 1)]  n/P=1.600 3cdiv=6.720 scaling nN/2=8
p=5 n=12 |B|=169 trial=0 sum exps=380 mean=2.2485 hist=[(0, 25), (1, 30), (2, 36), (3, 44), (4, 24), (5, 10)]  n/P=2.40

**T5: `ph:count` exactly for $k=2$.** `ph:count`: $[G:K_r]\dim S_c\le(1+\theta)\lvert\mathfrak B\rvert$ whenever $24Nk^2cP^{2r}\le\theta n$. Computed at the smallest $n$ allowed by the hypothesis. *Refutation would be:* a `FAIL` line (ratio above $1+\theta$). The ratio also tends to 1, i.e. the count is tight, as the argument needs.

In [17]:
%%writefile N1/t5_count.py
# Exact test of Proposition ph:count for k=2,r=1 (L = diagonal torus, so S_c is spanned by the monomials
# beta with max_i def_i(beta) <= 3c* = 18c): [G:K_r] dim S_c <= (1+theta)|B| when 24 N k^2 c P^{2r} <= theta n.
from fractions import Fraction as Fr
import itertools, math
import numpy as np
def count(N,P,n,c):
    # N=2 only (vectorised over y_1 for each y_0). (q-1)*def_i = (q-1) y_i + sum_s P^{[s-i]} (2 y_s - n)
    assert N==2
    q=P**N; thr=18*c*(q-1); cnt=0
    y1=np.arange(n+1,dtype=np.int64)
    for y0 in range(n+1):
        d0=(q-1)*y0+(2*y0-n)+P*(2*y1-n)
        d1=(q-1)*y1+(2*y1-n)+P*(2*y0-n)
        cnt+=int(((d0<=thr)&(d1<=thr)).sum())
    return cnt
for (N,P,c,theta) in [(2,5,1,Fr(1)),(2,5,1,Fr(1,2)),(2,5,1,Fr(1,10)),(2,7,1,Fr(1)),(2,5,3,Fr(1)),(2,13,1,Fr(1))]:
    n=math.ceil(24*N*4*c*P*P/theta); q=P**N
    S=count(N,P,n,c); lhs=(q+1)*S; B=(n+1)**N
    print(f"N={N} P={P} c={c} theta={theta} n={n}: [G:K]dim S_c={lhs}  (1+theta)|B|={float((1+theta)*B):.0f}  ratio={lhs/B:.5f}  {'OK' if lhs<=(1+theta)*B else 'FAIL'}")

Overwriting N1/t5_count.py


In [18]:
o = run("N1", "t5_count.py")
lines = [l for l in o.splitlines() if l.startswith("N=")]
assert len(lines) == 6 and all(l.rstrip().endswith("OK") for l in lines)
record("N1", "ph:count", "[G:K_r] dim S_c <= (1+theta)|B| at smallest admissible n", "k=2,N=2; P in {5,7,13}, theta in {1,1/2,1/10}, n up to 48000", "PASS (6/6; ratios 1.003-1.031)", "k=2 only (L = diagonal torus)")

N=2 P=5 c=1 theta=1 n=4800: [G:K]dim S_c=23767250  (1+theta)|B|=46099202  ratio=1.03113  OK
N=2 P=5 c=1 theta=1/2 n=9600: [G:K]dim S_c=93608866  (1+theta)|B|=138268802  ratio=1.01551  OK
N=2 P=5 c=1 theta=1/10 n=48000: [G:K]dim S_c=2311221666  (1+theta)|B|=2534505601  ratio=1.00309  OK
N=2 P=7 c=1 theta=1 n=9408: [G:K]dim S_c=90632800  (1+theta)|B|=177058562  ratio=1.02376  OK
N=2 P=5 c=3 theta=1 n=14400: [G:K]dim S_c=213720026  (1+theta)|B|=414777602  ratio=1.03053  OK
N=2 P=13 c=1 theta=1 n=32448: [G:K]dim S_c=1067398720  (1+theta)|B|=2105875202  ratio=1.01373  OK
[N1/t5_count.py: exit code 0, 10.8 s]
RECORDED: N1 | ph:count | PASS (6/6; ratios 1.003-1.031)


### Node N2 — the seesaw identity (`au:seesaw`), and N2b (archimedean part)

**N2:** no numerical scripts were run; all checks were by hand (the statement concerns theta lifts on unitary groups of large rank and has no small instance that was computed).

**N2b** checked the archimedean ingredients (`par:*`, `pco:*` in the archimedean files) and the level-$N$ algebra (`pl:*`) in small signatures.

**`t1_fock.py`: oscillator operators, `par:eq-osc`, `par:casimir`, `par:relw`, `par:exist`.** On polynomials in the Fock model (sympy, exact): the $\mathfrak{gl}_n$ and $\mathfrak{gl}_{n'}$ relations, commutation of the two actions, the Casimir operator identity (4 signatures); and for 60 random matching pairs in 5 signature pairs up to $U(2,1)\times U(3,2)$: the weight relations `par:relw`(i),(ii), the conditions of `par:def-harm` for the product of minors of `par:exist`, and the scalar Casimir identity.
*Refutation would be:* a `False` in the algebra/Casimir lines, an `AssertionError`, or a `FAIL` line (the script stops at the first failure).

In [19]:
%%writefile N2b/t1_fock.py
# Tests for A-archimedean: par:eq-osc (commutation), par:casimir, par:relw, par:exist, par:unique (small cases)
import sympy as sp, itertools, random
from sympy import Rational as Q
random.seed(1)

def setup(eps, epsp):
    n, n1 = len(eps), len(epsp)
    z = [[sp.Symbol(f'z{r}_{c}') for c in range(n1)] for r in range(n)]
    eta = [[-eps[r]*epsp[c] for c in range(n1)] for r in range(n)]
    def ap(r,c,P):   # a^+
        return sp.expand(z[r][c]*P) if eta[r][c]==1 else sp.expand(-sp.diff(P,z[r][c]))
    def am(r,c,P):   # a^-
        return sp.diff(P,z[r][c]) if eta[r][c]==1 else sp.expand(z[r][c]*P)
    return n,n1,z,eta,ap,am

def ops(eps, epsp, gam, gamp):
    n,n1,z,eta,ap,am = setup(eps,epsp)
    def E(a,b,P):
        s = 1 if eps[a]==eps[b] else -1
        return sp.expand(s*sum(ap(a,c,am(b,c,P)) for c in range(n1)) + (gam*P if a==b else 0))
    def Ep(c,d,P):
        s = 1 if epsp[c]==epsp[d] else -1
        return sp.expand(-s*sum(am(r,c,ap(r,d,P)) for r in range(n)) + (gamp*P if c==d else 0))
    return n,n1,z,eta,E,Ep

def randpoly(z,deg=3,terms=6):
    vs=[v for row in z for v in row]
    P=0
    for _ in range(terms):
        m=1
        for _ in range(random.randint(0,deg)): m*=random.choice(vs)
        P+=random.randint(-3,3)*m
    return sp.expand(P)

def test_algebra(eps,epsp,gam,gamp):
    n,n1,z,eta,E,Ep=ops(eps,epsp,gam,gamp)
    P=randpoly(z)
    ok=True
    for a,b,c,d in itertools.product(range(n),repeat=4):
        lhs=sp.expand(E(a,b,E(c,d,P))-E(c,d,E(a,b,P)))
        rhs=sp.expand((E(a,d,P) if b==c else 0)-(E(c,b,P) if a==d else 0))
        ok&= (lhs-rhs==0)
    for a,b,c,d in itertools.product(range(n1),repeat=4):
        lhs=sp.expand(Ep(a,b,Ep(c,d,P))-Ep(c,d,Ep(a,b,P)))
        rhs=sp.expand((Ep(a,d,P) if b==c else 0)-(Ep(c,b,P) if a==d else 0))
        ok&= (lhs-rhs==0)
    for a,b in itertools.product(range(n),repeat=2):
        for c,d in itertools.product(range(n1),repeat=2):
            ok&= (sp.expand(E(a,b,Ep(c,d,P))-Ep(c,d,E(a,b,P)))==0)
    # Casimir identity
    Om=sum(E(a,b,E(b,a,P)) for a in range(n) for b in range(n))
    Omp=sum(Ep(c,d,Ep(d,c,P)) for c in range(n1) for d in range(n1))
    N=sum(E(a,a,P) for a in range(n))-n*gam*P
    s0=gam+gamp-Q(n+n1,2)
    rhs=Om-2*s0*N+(n*n*n1-2*gamp*n*n1+n1*gamp**2-n*gam**2)*P
    ok2 = sp.expand(Omp-rhs)==0
    return ok,ok2

for eps,epsp in [((1,-1),(1,1,-1)),((1,1,-1),(1,-1,1,-1)),((1,-1),(1,-1)),((1,),(1,-1))]:
    print('algebra/casimir',eps,epsp,test_algebra(eps,epsp,Q(7,2),Q(-5,2)))

# matching pairs, par:relw, par:exist
def rho_parts(lam,eps):
    n=len(lam)
    A=[sum(eps[r]*eps[k]*sp.sign(lam[k]-lam[r]) for k in range(n)) for r in range(n)]
    return [lam[r]+Q(A[r],2) for r in range(n)]   # Lambda^K

def test_match(eps,epsp,trials=40):
    n,n1=len(eps),len(epsp); t=n1-n
    cnt=0
    for _ in range(trials*50):
        if cnt>=trials: break
        Bset=random.sample(range(n1),t)
        if len(set(epsp[c] for c in Bset))>1: continue
        rest=[c for c in range(n1) if c not in Bset]; random.shuffle(rest)
        cmap=dict(zip(range(n),rest))
        # xi_r sign forced: eta_{r,c(r)} = sgn xi_r
        base=Q(t-1,2)
        vals=random.sample(range(1,12),n)
        xi=[]
        for r in range(n):
            sgn=-eps[r]*epsp[cmap[r]]
            mag=Q(t+1,2)+vals[r]-1
            xi.append(sgn*mag)
        if len(set(xi))<n: continue
        gam=Q(random.randint(-9,9),2); gamp=Q(random.randint(-9,9),2)
        lam=[xi[r]+gam-Q(n1,2) for r in range(n)]
        xip=[None]*n1
        for r in range(n): xip[cmap[r]]=-xi[r]
        bs=list(range(t)); random.shuffle(bs)
        for c,b in zip(Bset,bs): xip[c]=Q(t-1,2)-b
        lamp=[xip[c]+gamp-Q(n,2) for c in range(n1)]
        if len(set(lamp))<n1: continue
        cnt+=1
        n_,n1_,z,eta,E,Ep=ops(eps,epsp,gam,gamp)
        Lam=rho_parts(lam,eps); Lamp=rho_parts(lamp,epsp)
        v=[gam-sum(1 for c in range(n1) if eta[r][c]==-1) for r in range(n)]
        vp=[gamp-sum(1 for r in range(n) if eta[r][c]==1) for c in range(n1)]
        w=[Lam[r]-v[r] for r in range(n)]; wp=[Lamp[c]-vp[c] for c in range(n1)]
        assert all(w[r]*sp.sign(xi[r])>=0 for r in range(n)), 'relw(i)'
        assert all(w[r]==-wp[cmap[r]] for r in range(n)), 'relw(ii)'
        assert all(wp[c]==0 for c in Bset), 'relw(ii)B'
        assert all(x==int(x) for x in w)
        # Phi
        Phi=sp.Integer(1)
        maxl=max([abs(x) for x in w]+[0])
        for l in range(int(maxl)):
            for e in (1,-1):
                for e1 in (1,-1):
                    L=[r for r in range(n) if abs(w[r])>l and eps[r]==e and epsp[cmap[r]]==e1]
                    if L:
                        M=sp.Matrix(len(L),len(L),lambda i,j: z[L[i]][cmap[L[j]]])
                        Phi=sp.expand(Phi*M.det())
        ok=True
        for a in range(n): ok&= sp.expand(E(a,a,Phi)-Lam[a]*Phi)==0
        for c in range(n1): ok&= sp.expand(Ep(c,c,Phi)-Lamp[c]*Phi)==0
        for a,b in itertools.permutations(range(n),2):
            if eps[a]==eps[b] and lam[a]>lam[b]: ok&= E(a,b,Phi)==0
            if eps[a]!=eps[b]:
                ok&= sp.expand(sum(sp.diff(Phi,z[a][c],z[b][c]) for c in range(n1) if eta[a][c]==-1))==0
        for c,d in itertools.permutations(range(n1),2):
            if epsp[c]==epsp[d] and lamp[c]>lamp[d]: ok&= Ep(c,d,Phi)==0
            if epsp[c]!=epsp[d]:
                ok&= sp.expand(sum(sp.diff(Phi,z[r][c],z[r][d]) for r in range(n) if eta[r][c]==1))==0
        # Casimir scalars
        kap=sum(x*x for x in lam)-sum(Q(n-1-2*i,2)**2 for i in range(n))
        kapp=sum(x*x for x in lamp)-sum(Q(n1-1-2*i,2)**2 for i in range(n1))
        s0=gam+gamp-Q(n+n1,2)
        ok&= kapp==kap-2*s0*(sum(lam)-n*gam)+n*n*n1-2*gamp*n*n1+n1*gamp**2-n*gam**2
        if not ok: print('FAIL',eps,epsp,xi,cmap,Bset); return False
    return cnt

for eps,epsp in [((1,-1),(1,1,-1)),((1,-1),(1,-1,-1)),((1,1,-1),(1,1,-1,1)),((1,-1),(1,1,1,-1)),((1,-1,-1),(1,-1,1,-1,1))]:
    print('match/exist',eps,epsp,test_match(eps,epsp,12))

Overwriting N2b/t1_fock.py


In [20]:
o = run("N2b", "t1_fock.py")
assert o.count("(True, True)") == 4 and "False" not in o and "FAIL" not in o
assert re.findall(r"match/exist .* (\d+)$", o, re.M) == ["12"]*5
record("N2b", "par:eq-osc, par:casimir, par:relw, par:exist", "oscillator relations, Casimir identity, harmonic product of minors (sympy, exact)", "4 signatures; 60 matching pairs up to U(2,1)xU(3,2)", "PASS", "small signatures only")

algebra/casimir (1, -1) (1, 1, -1) (True, True)
algebra/casimir (1, 1, -1) (1, -1, 1, -1) (True, True)
algebra/casimir (1, -1) (1, -1) (True, True)
algebra/casimir (1,) (1, -1) (True, True)
match/exist (1, -1) (1, 1, -1) 12
match/exist (1, -1) (1, -1, -1) 12
match/exist (1, 1, -1) (1, 1, -1, 1) 12
match/exist (1, -1) (1, 1, 1, -1) 12
match/exist (1, -1, -1) (1, -1, 1, -1, 1) 12
[N2b/t1_fock.py: exit code 0, 3.8 s]
RECORDED: N2b | par:eq-osc, par:casimir, par:relw, par:exist | PASS


**`t2_unique.py`: `par:unique`.** Dimension of the space of pre-joint harmonics of degree $\le\deg\Phi+3$, by linear algebra. *Refutation would be:* a dimension other than 1. (Two of the seven listed cases are rejected by the script's own validity check as not satisfying the hypotheses; five are tested.)

In [21]:
%%writefile N2b/t2_unique.py
# par:unique: dimension of the space of pre-joint harmonics (row weights fixed, column weights free) of degree <= D
import itertools, numpy as np, random
from fractions import Fraction as Q
random.seed(3)
def run(eps,epsp,xi,cmap,Bset,bs,gam,gamp,extra=3):
    n,n1=len(eps),len(epsp); t=n1-n
    eta=[[-eps[r]*epsp[c] for c in range(n1)] for r in range(n)]
    lam=[xi[r]+gam-Q(n1,2) for r in range(n)]
    xip=[None]*n1
    for r in range(n): xip[cmap[r]]=-xi[r]
    for c,b in zip(Bset,bs): xip[c]=Q(t-1,2)-b
    lamp=[xip[c]+gamp-Q(n,2) for c in range(n1)]
    sg=lambda x:(x>0)-(x<0)
    Lam=[lam[r]+Q(sum(eps[r]*eps[k]*sg(lam[k]-lam[r]) for k in range(n)),2) for r in range(n)]
    v=[gam-sum(1 for c in range(n1) if eta[r][c]==-1) for r in range(n)]
    w=[int(Lam[r]-v[r]) for r in range(n)]
    D=sum(abs(x) for x in w)+extra
    idx=[(r,c) for r in range(n) for c in range(n1)]
    # monomials: exponent tuples with row weights sum_c eta m = w_r, total degree<=D
    def rows(r,maxdeg):
        out=[]
        pos=[c for c in range(n1) if eta[r][c]==1]; neg=[c for c in range(n1) if eta[r][c]==-1]
        for dneg in range(0,maxdeg+1):
            dpos=w[r]+dneg
            if dpos<0 or dpos+dneg>maxdeg: continue
            for cp in itertools.combinations_with_replacement(pos,dpos) if pos or dpos==0 else []:
                for cn in itertools.combinations_with_replacement(neg,dneg) if neg or dneg==0 else []:
                    e=[0]*n1
                    for c in cp: e[c]+=1
                    for c in cn: e[c]+=1
                    out.append(tuple(e))
        return out
    per=[rows(r,D) for r in range(n)]
    mons=[m for m in itertools.product(*per) if sum(map(sum,m))<=D]
    pos={m:i for i,m in enumerate(mons)}
    eqs=[]  # each operator -> matrix rows; build dict-of-dict
    def apply(opterms):
        # opterms(m) -> dict mon->coef ; returns matrix
        cols={}
        tgt={}
        rowsM=[]
        for j,m in enumerate(mons):
            for mm,cf in opterms(m).items():
                if mm not in tgt: tgt[mm]=len(tgt)
                rowsM.append((tgt[mm],j,cf))
        M=np.zeros((len(tgt),len(mons)))
        for i,j,cf in rowsM: M[i,j]+=cf
        return M
    def shift(m,dec,inc):
        m=[list(x) for x in m]
        cf=1
        for (r,c) in dec:
            if m[r][c]==0: return None,0
            cf*=m[r][c]; m[r][c]-=1
        for (r,c) in inc: m[r][c]+=1
        return tuple(tuple(x) for x in m),cf
    mats=[]
    # row raising E_ab: D+_{a<-b} - D-_{b<-a}
    for a,b in itertools.permutations(range(n),2):
        if eps[a]==eps[b] and lam[a]>lam[b]:
            def op(m,a=a,b=b):
                out={}
                for c in range(n1):
                    if eta[b][c]==1:
                        mm,cf=shift(m,[(b,c)],[(a,c)])
                        if mm: out[mm]=out.get(mm,0)+cf
                    else:
                        mm,cf=shift(m,[(a,c)],[(b,c)])
                        if mm: out[mm]=out.get(mm,0)-cf
                return out
            mats.append(apply(op))
        if eps[a]!=eps[b]:
            def op(m,a=a,b=b):
                out={}
                for c in range(n1):
                    if eta[a][c]==-1:
                        mm,cf=shift(m,[(a,c),(b,c)],[])
                        if mm: out[mm]=out.get(mm,0)+cf
                return out
            mats.append(apply(op))
    for c,d in itertools.permutations(range(n1),2):
        if epsp[c]==epsp[d] and lamp[c]>lamp[d]:
            def op(m,c=c,d=d):
                out={}
                for r in range(n):
                    if eta[r][c]==1:
                        mm,cf=shift(m,[(r,c)],[(r,d)])
                        if mm: out[mm]=out.get(mm,0)-cf
                    else:
                        mm,cf=shift(m,[(r,d)],[(r,c)])
                        if mm: out[mm]=out.get(mm,0)+cf
                return out
            mats.append(apply(op))
        if epsp[c]!=epsp[d]:
            def op(m,c=c,d=d):
                out={}
                for r in range(n):
                    if eta[r][c]==1:
                        mm,cf=shift(m,[(r,c),(r,d)],[])
                        if mm: out[mm]=out.get(mm,0)+cf
                return out
            mats.append(apply(op))
    M=np.vstack([x for x in mats if x.size]) if mats else np.zeros((1,len(mons)))
    s=np.linalg.svd(M,compute_uv=False)
    rank=int((s>1e-8*max(1,s[0])).sum())
    return len(mons),len(mons)-rank,w,D
cases=[
 ((1,-1),(1,1,-1),[Q(-2),Q(-3)],{0:0,1:2},[1],[0]),
 ((1,-1),(1,1,-1),[Q(2),Q(4)],{0:2,1:0},[1],[0]),
 ((1,-1),(1,1,-1),[Q(3),Q(-2)],{0:2,1:2},[1],[0]),
 ((1,-1),(1,-1),[Q(2),Q(-1)],{0:1,1:1},[],[]),
 ((1,-1),(1,-1),[Q(2),Q(1)],{0:1,1:0},[],[]),
 ((1,1,-1),(1,1,-1,1),[Q(-1),Q(-3),Q(2)],{0:0,1:1,2:3},[2],[0]),
 ((1,-1),(1,1,1,-1),[Q(-3,1)+Q(1,2),Q(-5,1)+Q(1,2)],{0:0,1:3},[1,2],[0,1]),
]
for eps,epsp,xi,cmap,B,bs in cases:
    # validity: eta_{r,c(r)}=sgn xi_r, injective
    ok=all((-eps[r]*epsp[cmap[r]])==((xi[r]>0)-(xi[r]<0)) for r in range(len(eps))) and len(set(cmap.values()))==len(eps)
    if not ok: print('skip invalid',eps,epsp,xi,cmap); continue
    print(eps,epsp,[str(x) for x in xi],cmap,B,'-> (#monomials, dim prejoint, w, D)=',run(eps,epsp,xi,cmap,B,bs,Q(1,2),Q(3,2)))

Overwriting N2b/t2_unique.py


In [22]:
o = run("N2b", "t2_unique.py")
dims = [int(x) for x in re.findall(r"= \(\d+, (\d+),", o)]
print("dimensions:", dims)
assert len(dims) == 5 and set(dims) == {1}
record("N2b", "par:unique", "space of pre-joint harmonics is 1-dimensional", "5 valid cases, degree <= deg Phi + 3", "PASS (5/5)", "bounded degree; small signatures")

(1, -1) (1, 1, -1) ['-2', '-3'] {0: 0, 1: 2} [1] -> (#monomials, dim prejoint, w, D)= (9, 1, [-1, -4], 8)
(1, -1) (1, 1, -1) ['2', '4'] {0: 2, 1: 0} [1] -> (#monomials, dim prejoint, w, D)= (21, 1, [2, 4], 9)
skip invalid (1, -1) (1, 1, -1) [Fraction(3, 1), Fraction(-2, 1)] {0: 2, 1: 2}
skip invalid (1, -1) (1, -1) [Fraction(2, 1), Fraction(-1, 1)] {0: 1, 1: 1}
(1, -1) (1, -1) ['2', '1'] {0: 1, 1: 0} [] -> (#monomials, dim prejoint, w, D)= (3, 1, [2, 0], 5)
(1, 1, -1) (1, 1, -1, 1) ['-1', '-3', '2'] {0: 0, 1: 1, 2: 3} [2] -> (#monomials, dim prejoint, w, D)= (684, 1, [-1, -2, 2], 8)
(1, -1) (1, 1, 1, -1) ['-5/2', '-9/2'] {0: 0, 1: 3} [1, 2] -> (#monomials, dim prejoint, w, D)= (18, 1, [-1, -6], 10)
[N2b/t2_unique.py: exit code 0, 0.2 s]
dimensions: [1, 1, 1, 1, 1]
RECORDED: N2b | par:unique | PASS (5/5)


**`t3_fj.py`: formula `pco:eq-F` on $U(2,1)$.** The integral formula for the matrix coefficient, integrated over $K/T=S^2$ by Gauss–Legendre × trapezoid: independence of the auxiliary point $a$, $K$-equivariance, $F(1)=1$, the decay bound $\lvert F\rvert\cdot\lvert\det a(g)\rvert^{h/2}\le1$, the Casimir eigenvalue by finite differences (4 parameters, 2 of them neither holomorphic nor antiholomorphic), and the closed form in the scalar holomorphic case.
*Refutation would be:* dependence on $a$, failure of equivariance, $F(1)\ne1$, decay above 1, or a Casimir eigenvalue different from $\kappa$. (This is the slowest N2b script: about 2–4 minutes.)

In [23]:
%%writefile N2b/t3_fj.py
# Tests for A-coefficient: formula pco:eq-F on U(2,1) and U(1,1)
import numpy as np, itertools
np.random.seed(0)
def minor(x,S):
    S=list(S)
    return 1.0 if not S else np.linalg.det(x[np.ix_(S,S)])
def m_w(w,x):
    n=len(w); lo=min(min(w),0)-1; hi=max(max(w),0)+1
    val=1.0+0j; dx=np.linalg.det(x)
    for t in range(lo,hi+1):
        S=[a for a in range(n) if w[a]>=t]
        val*=minor(x,S)*(dx**(-1) if t<=0 else 1)
    return val
def params(lam,eps):
    n=len(lam); sg=np.sign
    rho=[0.5*sum(sg(lam[a]-lam[b]) for b in range(n) if b!=a) for a in range(n)]
    rc=[0.5*sum(sg(lam[a]-lam[b]) for b in range(n) if b!=a and eps[a]==eps[b]) for a in range(n)]
    rn=[rho[a]-rc[a] for a in range(n)]
    Lam=[int(round(lam[a]+rn[a]-rc[a])) for a in range(n)]
    mu=[int(round(lam[a]+rho[a])) for a in range(n)]
    kap=sum(l*l for l in lam)-sum(r*r for r in rho)
    return Lam,mu,kap
# quadrature on K/T for K=U(2)xU(1): S^2
def quad(nth=48,nph=64):
    xs,ws=np.polynomial.legendre.leggauss(nth)
    pts=[]
    for c,wt in zip(xs,ws):
        th=np.arccos(c)
        for j in range(nph):
            ph=2*np.pi*j/nph
            u=np.eye(3,dtype=complex)
            u[0,0]=np.cos(th/2); u[0,1]=-np.exp(-1j*ph)*np.sin(th/2)
            u[1,0]=np.exp(1j*ph)*np.sin(th/2); u[1,1]=np.cos(th/2)
            pts.append((u,wt/2/nph))
    return pts
PTS=quad()
def I(x,a,Lam,mu):
    s=0
    ainvx=np.linalg.solve(a,x)
    nmu=[-m for m in mu]
    for u,wt in PTS:
        s+=wt*m_w(Lam,a@u)*np.conj(m_w(Lam,u))/m_w(nmu,u.conj().T@ainvx@u)
    return s
def dLam(Lam):
    return 1/sum(wt*abs(m_w(Lam,u))**2 for u,wt in PTS).real
J=np.diag([1,1,-1]).astype(complex)
def rand_u21(scale=0.6):
    X=np.random.randn(3,3)+1j*np.random.randn(3,3)
    X=(X-J@X.conj().T@J)/2*scale
    from numpy.linalg import matrix_power
    # expm via series
    E=np.eye(3,dtype=complex); term=np.eye(3,dtype=complex)
    for k in range(1,40):
        term=term@X/k; E=E+term
    return E
def expm(X):
    E=np.eye(len(X),dtype=complex); term=np.eye(len(X),dtype=complex)
    for k in range(1,40):
        term=term@X/k; E=E+term
    return E
def polar(g):
    w,V=np.linalg.eigh(g.conj().T@g)
    z=V@np.diag(np.sqrt(w))@V.conj().T
    return g@np.linalg.inv(z),z
def F(g,Lam,mu,d):
    k,z=polar(g); return d*I(g,k,Lam,mu)

for lam in [(6,-5,0),(3,-4,0),(5,2,0),(2,1,9)]:
    eps=(1,1,-1)
    Lam,mu,kap=params(lam,eps); d=dLam(Lam)
    h=min(abs(lam[0]-lam[2]),abs(lam[1]-lam[2]))
    print('lambda',lam,'Lambda',Lam,'mu',mu,'kappa',kap,'d_Lambda',round(d,6),'gap',h)
    g=rand_u21(); k,z=polar(g)
    assert np.allclose(g.conj().T@J@g,J)
    F0=d*I(g,k,Lam,mu)
    # independence of a: a = k*h, h block diagonal (not unitary), with a^{-1}g accretive
    for trial in range(3):
        H=np.zeros((3,3),dtype=complex)
        B=np.random.randn(2,2)+1j*np.random.randn(2,2); H[:2,:2]=expm(0.25*B); H[2,2]=np.exp(0.2*(np.random.randn()+1j*np.random.randn()))
        a=k@H
        Z=np.linalg.solve(a,g); acc=np.linalg.eigvalsh(Z+Z.conj().T).min()
        print('   a=kH accretive margin %.3f  |I(g;a)-I(g;k)| = %.2e'%(acc,abs(d*I(g,a,Lam,mu)-F0)))
    # K-equivariance
    t=np.diag(np.exp(1j*np.random.randn(3)))
    tL=np.prod([t[i,i]**Lam[i] for i in range(3)])
    print('   F(gt)-t^Lam F(g): %.2e ; F(tg)-t^Lam F(g): %.2e ; F(1)=%.6f'%(abs(F(g@t,Lam,mu,d)-tL*F0),abs(F(t@g,Lam,mu,d)-tL*F0),abs(F(np.eye(3,dtype=complex),Lam,mu,d))))
    # decay
    worst=0
    for _ in range(6):
        gg=rand_u21(1.2); a_=(gg+J@gg@J)/2; D=abs(np.linalg.det(a_))
        val=abs(F(gg,Lam,mu,d)); worst=max(worst,val*D**(h/2))
    print('   max |F| * |det a(g)|^{h/2} = %.4f (must be <=1)'%worst)
    # Casimir by finite differences with fixed a=k (holomorphic local form)
    def Ft(x): return d*I(x,k,Lam,mu)
    def split(Zm):
        X1=(Zm-J@Zm.conj().T@J)/2; X2=(Zm+J@Zm.conj().T@J)/(2j); return X1,X2
    e=1e-3
    def d2(A,B):  # d/ds d/dt F(g e^{sA} e^{tB})
        return (Ft(g@expm(e*A)@expm(e*B))-Ft(g@expm(e*A)@expm(-e*B))-Ft(g@expm(-e*A)@expm(e*B))+Ft(g@expm(-e*A)@expm(-e*B)))/(4*e*e)
    Om=0
    for a_ in range(3):
        for b_ in range(3):
            Z1=np.zeros((3,3),dtype=complex); Z1[a_,b_]=1
            Z2=np.zeros((3,3),dtype=complex); Z2[b_,a_]=1
            A1,A2=split(Z1); B1,B2=split(Z2)
            for (A,ca) in ((A1,1),(A2,1j)):
                for (B,cb) in ((B1,1),(B2,1j)):
                    if np.abs(A).max()<1e-14 or np.abs(B).max()<1e-14: continue
                    Om+=ca*cb*d2(A,B)
    print('   R(Omega)F/F = %s  vs kappa=%s'%(np.round(Om/F0,4),kap))
# holomorphic scalar K-type: closed form
print('holomorphic scalar case: lambda=(l+1,l,l3), F(z) vs z33^{-(Lam1-Lam3)}')
for lam in [(4,3,-2),(6,5,1)]:
    Lam,mu,kap=params(lam,(1,1,-1)); d=dLam(Lam)
    for _ in range(3):
        g=rand_u21(0.8); k,z=polar(g)
        print('   Lam',Lam,'F(z)=%.8f'%F(z,Lam,mu,d).real,' z33^-(L1-L3)=%.8f'%(z[2,2].real**(-(Lam[0]-Lam[2]))), ' imag %.1e'%abs(F(z,Lam,mu,d).imag))

Overwriting N2b/t3_fj.py


In [24]:
o = run("N2b", "t3_fj.py")
ind = [float(x) for x in re.findall(r"\|I\(g;a\)-I\(g;k\)\| = ([\d.e+-]+)", o)]
eq  = [float(v) for pair in re.findall(r"F\(gt\)-t\^Lam F\(g\): ([\d.e+-]+) ; F\(tg\)-t\^Lam F\(g\): ([\d.e+-]+)", o) for v in pair]
F1  = [float(x) for x in re.findall(r"F\(1\)=([\d.]+)", o)]
dec = [float(x) for x in re.findall(r"= ([\d.]+) \(must be <=1\)", o)]
cas = re.findall(r"R\(Omega\)F/F = \(?(-?[\d.]+)[-+][\d.]+j\)?\s+vs kappa=([-\d.]+)", o)
print("max |I(g;a)-I(g;k)|:", max(ind), "| max equivariance defect:", max(eq), "| F(1):", F1, "| decay:", dec)
print("Casimir (numerical real part, kappa):", cas)
hol = re.findall(r"F\(z\)=([-\d.]+)\s+z33\^-\(L1-L3\)=([-\d.]+)", o)
assert max(ind) < 1e-6 and max(eq) < 1e-6 and all(abs(x-1) < 1e-6 for x in F1) and all(x <= 1 for x in dec)
assert len(cas) == 4 and all(abs(float(a)-float(b)) < 1e-2*max(1, abs(float(b))) for a, b in cas)
assert len(hol) == 6 and all(abs(float(a)-float(b)) < 1e-6 for a, b in hol)
record("N2b", "pco:eq-F", "matrix-coefficient formula on U(2,1): independence, equivariance, F(1)=1, decay, Casimir, closed form", "4 parameters + 2 holomorphic cases", "PASS", "U(2,1) only; quadrature")

lambda (6, -5, 0) Lambda [6, -5, 0] mu [7, -6, 0] kappa 59.0 d_Lambda 12.0 gap 5
   a=kH accretive margin 0.498  |I(g;a)-I(g;k)| = 1.53e-15
   a=kH accretive margin 0.679  |I(g;a)-I(g;k)| = 3.42e-16
   a=kH accretive margin 0.774  |I(g;a)-I(g;k)| = 1.13e-15
   F(gt)-t^Lam F(g): 8.83e-17 ; F(tg)-t^Lam F(g): 4.86e-17 ; F(1)=1.000000
   max |F| * |det a(g)|^{h/2} = 0.1642 (must be <=1)
   R(Omega)F/F = (58.9994+0.0001j)  vs kappa=59.0
lambda (3, -4, 0) Lambda [3, -4, 0] mu [4, -5, 0] kappa 23.0 d_Lambda 8.0 gap 3
   a=kH accretive margin 0.484  |I(g;a)-I(g;k)| = 8.67e-16
   a=kH accretive margin 0.602  |I(g;a)-I(g;k)| = 3.28e-16
   a=kH accretive margin 0.744  |I(g;a)-I(g;k)| = 2.52e-16
   F(gt)-t^Lam F(g): 3.23e-16 ; F(tg)-t^Lam F(g): 7.77e-17 ; F(1)=1.000000
   max |F| * |det a(g)|^{h/2} = 0.1261 (must be <=1)
   R(Omega)F/F = (22.9999+0j)  vs kappa=23.0
lambda (5, 2, 0) Lambda [5, 3, -1] mu [6, 2, -1] kappa 27.0 d_Lambda 3.0 gap 2
   a=kH accretive margin 0.353  |I(g;a)-I(g;k)| = 8.10e

**`t4_level.py`: `pl:mir`, `pl:Uprop`, the row/direction exchange of `pl:law`, and $k_y$.** At one split place with residue field $\mathbb F_N$: conjugation of the mirabolic-type group by $\mathrm{diag}(1,\dots,N)$, commutation with $U$, and the exchange "row condition after conjugation ⇔ direction condition before", for $(n,N)=(2,3),(3,5),(3,7)$ with 200 random samples each. *Refutation would be:* an `AssertionError`.

In [25]:
%%writefile N2b/t4_level.py
# pl:mir, pl:Uprop(b), pl:law (row<->dir), pl:glue (function identity), one split place, residue field F_N
import itertools, random
from fractions import Fraction as Fr
random.seed(5)
def matmul(A,B): return [[sum(A[i][k]*B[k][j] for k in range(len(B))) for j in range(len(B[0]))] for i in range(len(A))]
def det(A):
    n=len(A)
    if n==1: return A[0][0]
    return sum((-1)**j*A[0][j]*det([r[:j]+r[j+1:] for r in A[1:]]) for j in range(n))
def inv(A):
    n=len(A); d=det(A)
    C=[[(-1)**(i+j)*det([r[:j]+r[j+1:] for k,r in enumerate(A) if k!=i]) if n>1 else 1 for j in range(n)] for i in range(n)]
    return [[Fr(C[j][i],d) for j in range(n)] for i in range(n)]
def is_int(A): return all(Fr(x).denominator==1 for r in A for x in r)
def unit(x,N): return Fr(x).denominator%N!=0 and Fr(x).numerator%N!=0
def integral_inv(A,N):  # entries N-integral and det an N-unit
    return all(Fr(x).denominator%N!=0 for r in A for x in r) and unit(det(A),N)
def modN(x,N): x=Fr(x); return x.numerator*pow(x.denominator,-1,N)%N
for n,N in [(2,3),(3,5),(3,7)]:
    D=[[ (N if i==n-1 else 1) if i==j else 0 for j in range(n)] for i in range(n)]
    Di=[[ (Fr(1,N) if i==n-1 else 1) if i==j else 0 for j in range(n)] for i in range(n)]
    cnt=0
    while cnt<200:
        k=[[random.randint(-20,20) for _ in range(n)] for _ in range(n)]
        # force k in Mir: last column = e mod N
        for i in range(n-1): k[i][n-1]=N*random.randint(-3,3)
        k[n-1][n-1]=1+N*random.randint(-3,3)
        if not integral_inv(k,N): continue
        cnt+=1
        c=matmul(matmul(D,k),Di)
        assert integral_inv(c,N)
        assert all(modN(c[n-1][j],N)==(1 if j==n-1 else 0) for j in range(n)), 'last row'
        # Uprop: (c k_y) has last column c*y, with same last entry mod N
        y=[random.randint(0,N-1) for _ in range(n-1)]+[random.randint(1,N-1)]
        cy=[modN(sum(c[i][j]*y[j] for j in range(n)),N) for i in range(n)]
        assert cy[n-1]==y[n-1]
        # row(D T D^-1)=dir(T): random T n'xn with n'=n+1
        n1=n+1
        D1=[[ (N if i==n1-1 else 1) if i==j else 0 for j in range(n1)] for i in range(n1)]
        T=[[random.randint(-6,6) for _ in range(n)] for _ in range(n1)]
        if random.random()<0.6:
            for i in range(n1-1): T[i][n-1]=N*random.randint(-2,2)
            T[n1-1][n-1]=1+N*random.randint(-2,2)
        Tp=matmul(matmul(D1,T),Di)
        row=all(Fr(x).denominator%N!=0 for r in Tp for x in r) and all(modN(Tp[n1-1][j],N)==(1 if j==n-1 else 0) for j in range(n))
        dr=all(modN(T[i][n-1],N)==(1 if i==n1-1 else 0) for i in range(n1))
        assert row==dr,(T,Tp)
    print('n=%d N=%d: pl:mir conjugation, U-commutation, row/dir exchange OK (200 samples)'%(n,N))
# k_y has last row and last column y
for N in (3,5):
    for y in itertools.product(range(N),repeat=3):
        if y[2]==0: continue
        yp=y[:2]; u=y[2]; ui=pow(u,-1,N)
        k=[[ (1 if i==j else 0)+ui*yp[i]*yp[j] for j in range(2)]+[yp[i]] for i in range(2)]+[list(yp)+[u]]
        assert [k[i][2]%N for i in range(3)]==list(y) and [k[2][j]%N for j in range(3)]==list(y) and det(k)%N==u%N
print('k_y: last row = last column = y, det = u: OK')

Overwriting N2b/t4_level.py


In [26]:
o = run("N2b", "t4_level.py")
assert o.count("OK (200 samples)") == 3 and "k_y: last row = last column = y, det = u: OK" in o
record("N2b", "pl:mir, pl:Uprop, pl:law, k_y", "level-N algebra at a split place", "(n,N)=(2,3),(3,5),(3,7), 200 samples each", "PASS", "one place; small n")

n=2 N=3: pl:mir conjugation, U-commutation, row/dir exchange OK (200 samples)
n=3 N=5: pl:mir conjugation, U-commutation, row/dir exchange OK (200 samples)
n=3 N=7: pl:mir conjugation, U-commutation, row/dir exchange OK (200 samples)
k_y: last row = last column = y, det = u: OK
[N2b/t4_level.py: exit code 0, 0.1 s]
RECORDED: N2b | pl:mir, pl:Uprop, pl:law, k_y | PASS


### Node N3 — conjugates of CM theta jets and the Chowla–Selberg constant (`ps:conj`, `pf:recip`, `pf:CS`)

The exact constant $\Omega=e^{\gamma-L'/L(1,\chi)}\cdot 2\pi/rd_E$ matters at relative precision $x^{-1/2}$. The tests pin the relevant identities to 30–50 digits for imaginary quadratic fields $K=\mathbb Q(\sqrt{-d})$ with small $d$. Helper module `cm.py`: normalised CM lattices, theta functions, twisted translates; running it as a script is itself a test of (C4).

**`cs_test.py`: Chowla–Selberg average.** $(1/h)\sum\log(\mathrm{Im}\,\tau\,\lvert\eta(\tau)\rvert^4)=c_{CS}-\log(2\sqrt d)$ for $d=7,23,47,71$, with $c_{CS}=\gamma-L'/L(1,\chi)$ computed from Stieltjes constants. *Refutation would be:* a non-negligible difference.

In [27]:
%%writefile N3/cs_test.py
# Test of (C4): Chowla-Selberg average and the normalised-lattice period for K=Q(sqrt(-d))
from mpmath import *
mp.dps = 40
def kron(d):
    # chi_{-d}(n) for prime d = 3 mod 4: Legendre symbol (n/d)
    return lambda n: 0 if n % d == 0 else (1 if pow(n, (d-1)//2, d) == 1 else -1)
def LLp(d):
    chi = kron(d)
    S1 = sum(chi(a)*stieltjes(0, mpf(a)/d) for a in range(1, d))
    Sp = -sum(chi(a)*stieltjes(1, mpf(a)/d) for a in range(1, d))
    return S1/d, (Sp - log(d)*S1)/d
def forms(d):
    out = []
    a = 1
    while 3*a*a <= d:
        for b in range(-a+1, a+1):
            if (b*b + d) % (4*a) == 0:
                c = (b*b + d)//(4*a)
                if c >= a and not (a == c and b < 0):
                    out.append((a, b, c))
        a += 1
    return out
def eta(tau):
    q = exp(2j*pi*tau)
    return exp(2j*pi*tau/24)*qp(q)
for d in (7, 23, 47, 71):
    L, Lp = LLp(d)
    fs = forms(d); h = len(fs)
    assert abs(L - pi*h/sqrt(d)) < mpf(10)**(-30), (L, h)
    cCS = euler - Lp/L
    lhs = sum(log(((sqrt(d)/(2*a)) * abs(eta((b + 1j*sqrt(d))/(2*a)))**4)) for a, b, c in fs)/h
    rhs = cCS - log(2*sqrt(d))
    print(d, h, nstr(lhs, 30), nstr(rhs, 30), nstr(lhs-rhs, 5))

Overwriting N3/cs_test.py


In [28]:
o = run("N3", "cs_test.py")
diffs = [abs(float(l.split()[-1])) for l in o.strip().splitlines()]
assert len(diffs) == 4 and max(diffs) < 1e-30
record("N3", "pf:CS / (C4)", "Chowla–Selberg average equals c_CS - log(2 sqrt d)", "d = 7, 23, 47, 71", f"PASS (diff <= {max(diffs):.0e})", "class numbers 1-7 only")

7 1 -1.10452228017978948032001915445 -1.10452228017978948032001915445 4.5918e-41
23 3 -1.26155767558453996870238305409 -1.26155767558453996870238305409 4.5918e-41
47 5 -1.39984619715969183076217474488 -1.39984619715969183076217474488 -2.2959e-41
71 7 -1.46469619323452837432361737462 -1.46469619323452837432361737462 2.5255e-40
[N3/cs_test.py: exit code 0, 5.0 s]
RECORDED: N3 | pf:CS / (C4) | PASS (diff <= 3e-40)


In [29]:
%%writefile N3/cm.py
# Toolkit: normalised CM lattices, theta, twisted translates (notation of pa:ss-curve / pf:)
from mpmath import *
mp.dps = 50
def kron(d):
    return lambda n: 0 if n % d == 0 else (1 if pow(n, (d-1)//2, d) == 1 else -1)
def cCS(d):
    chi = kron(d)
    S1 = sum(chi(a)*stieltjes(0, mpf(a)/d) for a in range(1, d))
    Sp = -sum(chi(a)*stieltjes(1, mpf(a)/d) for a in range(1, d))
    L, Lp = S1/d, (Sp - log(d)*S1)/d
    return euler - Lp/L
def forms(d):
    out = []
    a = 1
    while 3*a*a <= d:
        for b in range(-a+1, a+1):
            if (b*b + d) % (4*a) == 0:
                c = (b*b + d)//(4*a)
                if c >= a and not (a == c and b < 0):
                    out.append((a, b, c))
        a += 1
    return out
class Lat:
    """Lattice Om*(Z+Z tau), tau=(b+sqrt(-d))/(2a); normalised: t admissible, wp''(t)=2."""
    def __init__(self, d, a, b, fourth=0):
        self.d, self.a, self.b = d, a, b
        self.tau = (b + 1j*sqrt(d))/(2*a)
        tau = self.tau
        self.q = exp(1j*pi*tau)
        q = self.q
        th2, th3, th4 = jtheta(2,0,q), jtheta(3,0,q), jtheta(4,0,q)
        e = {(1,0): pi**2*(th3**4+th4**4)/3, (0,1): -pi**2*(th2**4+th3**4)/3, (1,1): pi**2*(th2**4-th4**4)/3}
        self.e_unit = e
        om = (1+1j*sqrt(d))/2
        def inlat(z):
            n = z.imag/tau.imag; m = z.real - n*tau.real
            return abs(n-round(n)) < 1e-20 and abs(m-round(m)) < 1e-20
        adm = [k for k in e if not inlat(om*(k[0]+k[1]*tau)/2) and not inlat(conj(om)*(k[0]+k[1]*tau)/2)]
        assert len(adm) == 1, adm
        k = adm[0]
        oth = [e[j] for j in e if j != k]
        wpp = 2*(e[k]-oth[0])*(e[k]-oth[1])
        self.Om = (wpp/2)**mpf('0.25') * (1j)**fourth
        Om = self.Om
        self.t = Om*(k[0]+k[1]*tau)/2
        self.e1 = e[k]/Om**2
        self.a1sq = 12*self.e1
        self.A = abs(Om)**2*tau.imag
        self.thp0 = jtheta(1,0,q,1)
        E2 = 1 - 24*nsum(lambda n: n*q**(2*n)/(1-q**(2*n)), [1, inf])
        self.s2 = (pi**2/3*E2 - pi/tau.imag)/Om**2
        self.Delta = self.a1sq**2 - 64
    def vth(self, z):   # e^{-s2 z^2/2} sigma(z)
        w = z/self.Om
        return self.Om*exp(pi*w*w/(2*self.tau.imag))*jtheta(1, pi*w, self.q)/(pi*self.thp0)
    def H(self, u, w): return pi*conj(u)*w/self.A
    def theta(self, z):
        return self.vth(z+self.t)/self.vth(self.t)*exp(-self.H(self.t, z))
    def theta_u(self, u, z):
        return exp(-(pi/self.A)*(z*conj(u)+abs(u)**2/2))*self.theta(z+u)
    def jet(self, u, n):   # Taylor coefficients c_0..c_n of theta_u at 0
        return taylor(lambda z: self.theta_u(u, z), 0, n)
if __name__ == "__main__":
    for d in (7, 23, 47):
        c = cCS(d); fs = forms(d); h = len(fs)
        lats = [Lat(d, a, b) for a, b, _ in fs]
        L0 = lats[0]
        print("d", d, "h", h)
        for L in lats:
            print("  form", L.a, L.b, " a1^4 =", nstr(L.a1sq**2, 25), " Delta=a1^4-64 =", nstr(L.Delta, 20))
        # sanity: g''(0) = s2 + a1^2/12 ; check theta''(0) = -(s2+a1^2/12)
        j = L0.jet(0, 4)
        print("  theta jet", [nstr(x, 15) for x in j], " -(s2+a1^2/12)/2 =", nstr(-(L0.s2+L0.a1sq/12)/2, 15))
        # (C4): prod_i area(L_i)|Delta(L_i)|^{1/6} = (2 pi^2 e^c / sqrt d)^h   [area(L)|u|^2 to the h]
        lhs = fprod(L.A*abs(L.Delta)**(mpf(1)/6) for L in lats)**(mpf(1)/h)
        rhs = 2*pi**2*exp(c)/sqrt(d)
        print("  (C4) lhs", nstr(lhs, 35), "rhs", nstr(rhs, 35), "diff", nstr(lhs-rhs, 5))
        # Omega-side: |Omega_0|^2 |u|^2 = 4 pi^2 e^c/d

Overwriting N3/cm.py


**`cm.py` run as a script: (C4) for normalised lattices.** $\prod_i\mathrm{area}(\Lambda_i)\lvert\Delta(\Lambda_i)\rvert^{1/6}=(2\pi^2e^{c_{CS}}/\sqrt d)^h$ for $d=7,23,47$, and $\theta''(0)=-(s_2+a_1^2/12)$. *Refutation would be:* a non-negligible difference in the (C4) line.

In [30]:
o = run("N3", "cm.py")
diffs = [abs(float(x)) for x in re.findall(r"\(C4\) lhs [\d.]+ rhs [\d.]+ diff ([-\d.e]+)", o)]
print("(C4) differences:", diffs)
assert len(diffs) == 3 and max(diffs) < 1e-40
record("N3", "(C4), pf:CS", "product of area*|Delta|^(1/6) over classes = (2 pi^2 e^c/sqrt d)^h", "d = 7, 23, 47", f"PASS (diff <= {max(diffs):.0e})", "small d")

d 7 h 1
  form 1 1  a1^4 = (63.0 + 0.0j)  Delta=a1^4-64 = (-1.0 + 0.0j)
  theta jet ['1.0', '0.0', '-0.377964473009227', '0.0', '-0.0119047619047619']  -(s2+a1^2/12)/2 = (-0.377964473009227 - 9.89433867485318e-55j)
  (C4) lhs 13.081929528762447470379081439566555 rhs 13.081929528762447470379081439566555 diff 6.4146e-50
d 23 h 3
  form 1 1  a1^4 = (63.99882770250387138817591 + 0.0j)  Delta=a1^4-64 = (-0.0011722974961286118241 + 0.0j)
  form 2 -1  a1^4 = (75.00058614874806430591204 + 27.05573684642812457331699j)  Delta=a1^4-64 = (11.000586148748064306 + 27.055736846428124573j)
  form 2 1  a1^4 = (75.00058614874806430591204 - 27.05573684642812457331699j)  Delta=a1^4-64 = (11.000586148748064306 - 27.055736846428124573j)
  theta jet ['1.0', '0.0', '-0.433625662907974', '0.0', '0.0106822744328568']  -(s2+a1^2/12)/2 = (-0.433625662907974 - 1.15706979839207e-57j)
  (C4) lhs 11.180782510073900289850569601130549 rhs 11.180782510073900289850569601130549 diff 5.709e-48
d 47 h 5
  form 1 1  a1^4 = (

**`unit_test.py`: is the Chowla–Selberg number a unit?** For $d=23$ ($h=3$): the numbers $\mathsf u^{36}$ for the three classes should be conjugate over $K$, with integral symmetric functions and norm $\pm1$, and $\mathrm{area}\cdot\lvert\sigma\mathsf u\rvert^2$ should equal $2\pi^2e^{c_{CS}}/\sqrt d$ for each class. *Refutation would be:* non-integral symmetric functions, a norm other than $\pm1$, or a class-dependent value.

In [31]:
%%writefile N3/unit_test.py
# The Chowla-Selberg number u^{12h} for d=23 and its conjugates over K: is it an algebraic unit?
from cm import *
d = 23
L1, L2, L3 = Lat(d,1,1), Lat(d,2,-1), Lat(d,2,1)
c = cCS(d)
P = fprod(L.Om**12*L.Delta for L in (L1,L2,L3))
s = 1j*sqrt(d)
kap = {1: mpf(2), 2: (3+s)/2/4, 3: (3-s)/2/4}
U = {i: P/(L.Om**36*kap[i]**12) for i, L in ((1,L1),(2,L2),(3,L3))}
for i in U: print("U_%d ="%i, nstr(U[i], 30), " |U|^(1/18)*area =", nstr(abs(U[i])**(mpf(1)/18)*{1:L1,2:L2,3:L3}[i].A, 30))
print("target 2pi^2 e^c/sqrt d =", nstr(2*pi**2*exp(c)/sqrt(d), 30))
e1 = U[1]+U[2]+U[3]; e2 = U[1]*U[2]+U[1]*U[3]+U[2]*U[3]; e3 = U[1]*U[2]*U[3]
for nm, e in (("e1",e1),("e2",e2),("e3",e3)):
    # element of O_K = Z + Z(1+s)/2 ?
    y = 2*e.imag/sqrt(d); xx = e.real - y/2
    print(nm, nstr(e, 30), " coords in Z[(1+sqrt(-23))/2]:", nstr(xx, 25), nstr(y, 25))

Overwriting N3/unit_test.py


In [32]:
o = run("N3", "unit_test.py")
vals = [float(x) for x in re.findall(r"\|U\|\^\(1/18\)\*area = ([\d.]+)", o)]
tgt = float(re.search(r"target 2pi\^2 e\^c/sqrt d = ([\d.]+)", o).group(1))
e = re.findall(r"coords in Z\[\(1\+sqrt\(-23\)\)/2\]: ([-\d.e]+) ([-\d.e]+)", o)
print("per-class values:", vals, "target:", tgt, "symmetric functions:", e)
assert all(abs(v - tgt) < 1e-12 for v in vals)
assert [round(float(x)) for x, y in e] == [1222, 727653, -1] and all(abs(float(x)-round(float(x))) < 1e-15 and abs(float(y)) < 1e-15 for x, y in e)
record("N3", "pf:CS (unit)", "u^36 conjugates: symmetric functions 1222, 727653, -1; area*|u|^2 = target per class", "d = 23", "PASS", "one field")

U_1 = (-0.00000137428141942941265468898287557 + 2.21429908611541461504207950243e-62j)  |U|^(1/18)*area = 11.1807825100739002898505696011
U_2 = (611.000000687140709714706327344 - 595.257927993979722927623026233j)  |U|^(1/18)*area = 11.1807825100739002898505696011
U_3 = (611.000000687140709714706327344 + 595.257927993979722927623026233j)  |U|^(1/18)*area = 11.1807825100739002898505696011
target 2pi^2 e^c/sqrt d = 11.1807825100739002898505696011
e1 (1222.0 + 0.0j)  coords in Z[(1+sqrt(-23))/2]: 1222.0 0.0
e2 (727653.0 + 0.0j)  coords in Z[(1+sqrt(-23))/2]: 727653.0 0.0
e3 (-1.0 + 3.78580319026784237452660301664e-52j)  coords in Z[(1+sqrt(-23))/2]: -1.0 1.578789067908206948799003e-52
[N3/unit_test.py: exit code 0, 0.9 s]
per-class values: [11.1807825100739, 11.1807825100739, 11.1807825100739] target: 11.1807825100739 symmetric functions: [('1222.0', '0.0'), ('727653.0', '0.0'), ('-1.0', '1.578789067908206948799003e-52')]
RECORDED: N3 | pf:CS (unit) | PASS


**`t_values.py` and `t_den.py`: `pf:chi`(b), `pf:values`, `pf:recip`.** For $d=7,23$ and $n=3,5$: the character $\chi(\Omega a)=(-1)^{Na}$; the product over classes of the two non-zero 2-torsion theta values has modulus $2^h$; the product of values at exact-order-$n$ points has modulus 1 (units); power sums of values and jets are Galois-stable (lie in $K$) and the values are integral ($-70$, $3004$, $-61410$, $2328319012$); `t_den.py` (which imports and reruns `t_values.py`) finds the denominators of the jet power sums: only $n$ (first jet), $n$ and $d$ (second jet).
*Refutation would be:* a wrong character value, a non-unit product, a power sum outside $K$, a non-integral value sum, or denominators involving other primes.

In [33]:
%%writefile N3/t_values.py
# Tests of pf:chi(b), pf:values, pf:recip (Galois stability + integrality of torsion values and jets)
from cm import *
mp.dps = 40
def inK(S, d):
    y = 2*S.imag/sqrt(d); x = S.real - y/2
    return x, y
def pts(L, n):
    out = []
    for j in range(n):
        for k in range(n):
            if gcd(gcd(j, k), n) == 1 and (j, k) != (0, 0):
                out.append(L.Om*(j + k*L.tau)/n)
    return out
from math import gcd
def br(L, u): return L.theta_u(u, 0)
for d in (7, 23):
    lats = [Lat(d, a, b) for a, b, _ in forms(d)]
    h = len(lats)
    print("=== d =", d, "h =", h)
    # pf:chi(b): chi(Om a) = (-1)^{N a} for the principal lattice
    L = lats[0]; om = (1+1j*sqrt(d))/2
    for (m1, m2) in ((1,0),(0,1),(1,1),(2,1),(1,2),(3,1)):
        a = m1 + m2*om; lam = L.Om*a; z = mpf('0.123')+0.3j
        chi = L.theta(z+lam)/L.theta(z)/exp(L.H(lam, z+lam/2))
        Na = m1*m1 + m1*m2 + m2*m2*(1+d)//4
        print("  chi(Om*(%d+%d w)) = %s   (-1)^N = %d" % (m1, m2, nstr(chi, 12), (-1)**Na))
    # order two: |prod over classes of theta[t1] theta[t2]| = 2^h
    P = mpf(1)
    for L in lats:
        halves = [L.Om/2, L.Om*L.tau/2, L.Om*(1+L.tau)/2]
        vals = [br(L, u) for u in halves]
        nz = [v for v in vals if abs(v) > 1e-20]
        assert len(nz) == 2
        P *= abs(nz[0]*nz[1])
    print("  prod |theta[t1]theta[t2]| =", nstr(P, 25), " expected 2^h =", 2**h)
    for n in (3, 5):
        # units: product over classes and exact-order-n points has modulus 1
        P = mpf(1); k = 4*n
        S0 = 0; S1 = 0; S2 = 0
        for L in lats:
            for u in pts(L, n):
                c = L.jet(u, 2)
                P *= abs(c[0])
                S0 += c[0]**k; S1 += c[1]**k; S2 += (2*c[2])**k
        print("  n=%d: prod|theta[u]| = %s" % (n, nstr(P, 25)))
        for nm, S in (("S0", S0), ("S1", S1), ("S2", S2)):
            x, y = inK(S, d)
            print("    %s (power sum of jets^%d): coords in O_K: %s , %s" % (nm, k, nstr(x, 30), nstr(y, 30)))

Overwriting N3/t_values.py


In [34]:
%%writefile N3/t_den.py
from t_values import *   # reruns prints; fine
print("---- denominators")
mp.dps = 60
from fractions import Fraction
for d in (7, 23):
    lats = [Lat(d, a, b) for a, b, _ in forms(d)]
    for n in (3, 5):
        k = 4*n
        S1 = 0; S2 = 0
        for L in lats:
            for u in pts(L, n):
                c = L.jet(u, 2)
                S1 += c[1]**k; S2 += (2*c[2])**k
        for nm, S in (("S1", S1), ("S2", S2)):
            x, y = inK(S, d)
            # find e with x*d^e near integer, also allow n
            found = None
            for e in range(0, 4*k):
                for e2 in range(0, 3):
                    v = x*mpf(d)**e*mpf(n)**(e2*k)
                    if abs(v - nint(v)) < mpf(10)**(-25) and abs(y) < mpf(10)**(-30):
                        found = (e, e2*k, nint(v)); break
                if found: break
            print(d, n, nm, "x*d^e*n^e2 integer:", found)

Overwriting N3/t_den.py


In [35]:
o = run("N3", "t_den.py")
chis = re.findall(r"= \(([-\d.]+) [-+] [\d.e-]+j\)\s+\(-1\)\^N = (-?1)", o)
assert len(chis) == 12 and all(round(float(a)) == int(b) for a, b in chis)
assert re.findall(r"prod \|theta\[t1\]theta\[t2\]\| = ([\d.]+)", o) == ["2.0", "8.0"]
assert re.findall(r"prod\|theta\[u\]\| = ([\d.]+)", o) == ["1.0"]*4
S0 = [float(x) for x in re.findall(r"S0 \(power sum of jets\^\d+\): coords in O_K: ([-\d.]+) ,", o)]
assert [round(x) for x in S0] == [-70, 3004, -61410, 2328319012]
imag = [abs(float(y)) for y in re.findall(r"coords in O_K: [-\d.e]+ , ([-\d.e]+)", o)]
assert max(imag) < 1e-25            # Galois stability: power sums lie in Q (rational part of O_K coordinates)
den = re.findall(r"(\d+) (\d) (S[12]) x\*d\^e\*n\^e2 integer: \((\d+), (\d+),", o)
assert len(den) == 8 and all((s == "S1" and e == "0") or s == "S2" for d, n, s, e, e2 in den)
record("N3", "pf:chi(b), pf:values, pf:recip", "torsion characters, units, Galois stability and integrality of values/jets, denominators", "d = 7, 23; n = 3, 5", "PASS", "two fields, n <= 5")

=== d = 7 h = 1
  chi(Om*(1+0 w)) = (-1.0 + 9.08531502086e-42j)   (-1)^N = -1
  chi(Om*(0+1 w)) = (1.0 - 7.00372104638e-41j)   (-1)^N = 1
  chi(Om*(1+1 w)) = (1.0 - 1.71869115613e-40j)   (-1)^N = 1
  chi(Om*(2+1 w)) = (1.0 - 2.90060627695e-40j)   (-1)^N = 1
  chi(Om*(1+2 w)) = (-1.0 + 6.70012398001e-40j)   (-1)^N = -1
  chi(Om*(3+1 w)) = (1.0 - 2.53654370736e-40j)   (-1)^N = 1
  prod |theta[t1]theta[t2]| = 2.0  expected 2^h = 2
  n=3: prod|theta[u]| = 1.0
    S0 (power sum of jets^12): coords in O_K: -70.0 , 8.50338061362827115729575935065e-39
    S1 (power sum of jets^12): coords in O_K: 0.354823959762231367169638774577 , -3.57391231228595426827739740552e-43
    S2 (power sum of jets^12): coords in O_K: 0.0000231188916648051114624034619563 , -5.01517469234801919058910530266e-46
  n=5: prod|theta[u]| = 1.0
    S0 (power sum of jets^20): coords in O_K: 3004.0 , -1.03363476758360674088015807004e-36
    S1 (power sum of jets^20): coords in O_K: 0.792767638011904 , -3.438429603096820397257

**`t_frame.py`: `pf:frame`.** For $d=7$, $q_1=8$ ($E=K(\zeta_8)$, $f=4$): the frame matrix $\sum v v^*$ over the family of `pf:frame` should be the scalar $h\cdot I$ with $h=67$ given by the formula. *Refutation would be:* a non-scalar matrix or a scalar different from the formula.

In [36]:
%%writefile N3/t_frame.py
# pf:frame for d=7 (omega*conj(omega)=2), q_1=8: E=K(zeta_8), f=4, P=X^4+1.
import numpy as np, cmath, math
d=7; q1=8; f=4; J=3
om=(1+1j*math.sqrt(d))/2; omb=om.conjugate()
units=[1,3,5,7]
# an element of E = K(zeta): represented as function of the embedding tau_a: zeta -> exp(2 pi i a/8), K fixed.
def emb(coefs):  # coefs: list of 4 complex numbers (in K) c_i for zeta^i ; returns vector over tau
    return np.array([sum(c*cmath.exp(2j*math.pi*a*i/8) for i,c in enumerate(coefs)) for a in units])
def embbar(coefs): # embedding of the complex conjugate element: conj of emb (E is CM)
    return emb(coefs).conjugate()
z=lambda k: np.array([cmath.exp(2j*math.pi*a*k/8) for a in units])
# P(X)/(X-zeta) = X^3 + zeta X^2 + zeta^2 X + zeta^3 ; b_i = zeta^{3-i}; P'(zeta)=4 zeta^3
e=[omb**J * z(3-i)/(4*z(3)) for i in range(4)]
# Galois conjugates rho_c: zeta->zeta^c : on embedding vectors, permutes: (rho_c w)(tau_a)=w(tau_{ac})
def gal(w,c): return np.array([w[units.index((a*c)%8)] for a in units])
fam=[]
for k in range(q1):
    if k<f: continue           # members zeta^{-k}, k<f, are replaced
    fam.append(z(-k))
for i in range(4):
    for k in range(q1):
        for c in units:
            if k==0 and c==1: continue
            fam.append(z(k)*gal(e[i],c))
for i in range(4): fam.append(z(-i)+e[i])
M=sum(np.outer(v,v.conjugate()) for v in fam)
nu=[sum(abs(x)**2 for x in ei) for ei in e]
h=q1*(1+sum(nu))+2*(om**J).real
print("nu(e_i) =",nu," h formula =",h)
print(np.round(M,10))

Overwriting N3/t_frame.py


In [37]:
o = run("N3", "t_frame.py")
h = float(re.search(r"h formula = ([\d.]+)", o).group(1))
diag = [float(x) for x in re.findall(r"\[?\[?\s*(67\.)[-+]", o)]
assert abs(h - 67) < 1e-9 and o.count("67.") >= 4
record("N3", "pf:frame", "frame matrix is h*I with h from the formula", "d = 7, q1 = 8", "PASS (67*I)", "one case")

nu(e_i) = [np.float64(2.000000000000001), np.float64(2.000000000000001), np.float64(2.000000000000001), np.float64(2.000000000000001)]  h formula = 67.00000000000003
[[67.-0.j  0.+0.j -0.-0.j  0.-0.j]
 [ 0.-0.j 67.+0.j -0.+0.j -0.+0.j]
 [-0.+0.j -0.-0.j 67.-0.j  0.-0.j]
 [ 0.+0.j -0.-0.j  0.+0.j 67.+0.j]]
[N3/t_frame.py: exit code 0, 0.1 s]
RECORDED: N3 | pf:frame | PASS (67*I)


### Node N4 — the second moment of the Poincaré map (`au:moment`, Theorem B), sub-nodes N4a, N4b, N4c

`au:moment` itself (near-isometry and second moment uniformly in rank $r\approx x^5$) has **no small instance** and was not tested numerically. The sub-node tests check finite-group lemmas, measure normalisations and explicit formulas used in its proof.

#### N4a
**`tests.py`:** (1) masses of unimodular hermitian lattices for $E=\mathbb Q(\zeta_{12})$ against Gan–Hanke–Yu ($1/12$, $1/288$) and the mass-ratio lemma for $l=1,2$; (2) exponent identities of (C0) on 2000 random $(k,\rho)$; (3) the block determinant identity of `pbe:link`(a) (ratio $\pm1$); (4) Ostrowski–Taussky ($\det H\le 2^m\lvert\det\Theta\rvert$ when $\Theta+\Theta^*-H\ge0$) on 2000 random cases; (5) Hua's one-row integral $\int(1+\lvert u\rvert^2)^{-(k+1)}=\pi^k/k!$; (6) Gaussian binomial bounds of `pbm:crude`. The reference masses were inlined from Gan–Hanke–Yu in the original script; no data file is needed.
*Refutation would be:* any mismatch, an assertion error, or OT violations $>0$.

In [38]:
%%writefile N4a/tests.py
import mpmath as mp, random, itertools, cmath
import numpy as np
mp.mp.dps=30
# 1. mass ratio, E=Q(zeta_12), E+=Q(sqrt3), f=2, rd_E=sqrt(12).  mass_l = 2*2^{-lf} prod_{j<=l}|L(1-j,chi^j)| (GHY 4.4/4.5)
chi12=lambda n: {1:1,11:1,5:-1,7:-1}.get(n%12,0)
chi4=lambda n: {1:1,3:-1}.get(n%4,0)
chi3=lambda n: {1:1,2:-1}.get(n%3,0)
def L(chi,q,s): # via Hurwitz zeta
    return sum(chi(a)*mp.zeta(s,mp.mpf(a)/q) for a in range(1,q+1))*mp.mpf(q)**(-s)
L0 = L(chi4,4,0)*L(chi3,3,0)                 # L(0,chi_{E/E+})
zm1 = mp.zeta(-1)*L(chi12,12,-1)             # zeta_{E+}(-1)
m1 = 2*mp.mpf(2)**-2*abs(L0); m2 = 2*mp.mpf(2)**-4*abs(L0)*abs(zm1)
print("mass_1",m1,"(1/12=",mp.mpf(1)/12,")  mass_2",m2,"(1/288=",mp.mpf(1)/288,")")
rd=mp.sqrt(12); f=2
ratio=(rd**mp.mpf(1.5)*1/(2*mp.pi)**2)**f*mp.zeta(2)*L(chi12,12,2)
print("lemma ratio l=1:",ratio," mass_2/mass_1:",m2/m1)
# l=2 -> 3 : ratio lemma vs 2^{-f}|L(-2,chi)|
Lm2=L(chi4,4,-2)*L(chi3,3,-2)
L3=L(chi4,4,3)*L(chi3,3,3)
print("l=2:",(rd**mp.mpf(2.5)*2/(2*mp.pi)**3)**f*L3, mp.mpf(2)**-f*abs(Lm2))
# 2. (C0) exponents identity, symbolic check over random k,rho
for _ in range(2000):
    k=random.randint(1,60); rho=random.randint(1,30); n=rho+1
    S_l=sum(range(k,k+2*rho+1)); S_h=S_l+(2*rho+1)/2
    assert k+n+2*rho*(rho+k)==1+S_l
    assert abs((0.5-k-n-2*rho*(rho+k))-(rho-S_h))<1e-9
    m=2*rho+2; assert rho*(2*k+2*rho+1)==(m-2)*k+(m-2)*(m-1)//2
    assert sum(2*(2*(rho-a)+k+1)+1 for a in range(1,rho+1))==rho*(2*k+2*rho+1)
print("exponent identities ok")
# 3. block determinant identity of pbe:link(a)
rng=np.random.default_rng(1)
for n_ in (2,3,5):
    Z=rng.normal(size=(n_,n_))+1j*rng.normal(size=(n_,n_)); N=7.0
    D=np.zeros((n_,n_)); D[-1,-1]=-N
    G=np.block([[D,Z.conj().T],[Z,D]])
    g=abs(np.linalg.det(Z))**2-N**2*abs(np.linalg.det(Z[:-1,:-1]))**2
    print(n_, np.linalg.det(G).real/g)
# 4. Ostrowski-Taussky: H, Theta+Theta^*-H psd => det H <= 2^m |det Theta|
bad=0
for _ in range(2000):
    m=random.randint(1,5)
    A=rng.normal(size=(m,m))+1j*rng.normal(size=(m,m)); H=A@A.conj().T
    B=rng.normal(size=(m,m))+1j*rng.normal(size=(m,m)); P=B@B.conj().T
    K=rng.normal(size=(m,m))+1j*rng.normal(size=(m,m)); K=K-K.conj().T
    Th=(H+P)/2+K
    if np.linalg.det(H).real>2**m*abs(np.linalg.det(Th))*(1+1e-9): bad+=1
print("OT violations",bad)
# 5. Hua one-row integral: int_{C^k}(1+|u|^2)^{-(k+1)} du = pi^k/k!
for k in (1,2,3,7):
    val=mp.pi**k/mp.factorial(k-1)*mp.quad(lambda t:t**(k-1)*(1+t)**(-(k+1)),[0,mp.inf])
    print(k,val,mp.pi**k/mp.factorial(k))
# 6. pbm:crude exponent & Gaussian binomial bound
def gb(a,d,q):
    num=1;den=1
    for i in range(d): num*=(q**(a-i)-1); den*=(q**(i+1)-1)
    return num//den
ok=all(q**(d*(a-d))<=gb(a,d,q)<=4*q**(d*(a-d)) for q in (2,3,5,7) for a in range(0,9) for d in range(0,a+1))
print("gaussian binomial bounds",ok)

Overwriting N4a/tests.py


In [39]:
o = run("N4a", "tests.py")
m = re.search(r"mass_1 ([\d.]+) \(1/12= ([\d.]+) \)  mass_2 ([\d.]+) \(1/288= ([\d.]+) \)", o)
assert abs(float(m.group(1))-1/12) < 1e-25 and abs(float(m.group(3))-1/288) < 1e-25
r1 = re.search(r"lemma ratio l=1: ([\d.]+)  mass_2/mass_1: ([\d.]+)", o); assert abs(float(r1.group(1))-float(r1.group(2))) < 1e-25
r2 = re.search(r"l=2: ([\d.]+) ([\d.]+)", o); assert abs(float(r2.group(1))-float(r2.group(2))) < 1e-25
assert "exponent identities ok" in o and "OT violations 0" in o and "gaussian binomial bounds True" in o
blk = [float(x) for x in re.findall(r"^[235] ([-\d.]+)$", o, re.M)]; assert len(blk) == 3 and all(abs(abs(b)-1) < 1e-9 for b in blk)
hua = re.findall(r"^\d ([\d.]+) ([\d.]+)$", o, re.M); assert len(hua) == 4 and all(abs(float(a)-float(b)) < 1e-20 for a, b in hua)
record("N4a", "pbm masses, (C0), pbe:link(a), Ostrowski–Taussky, Hua, pbm:crude", "mass formula vs GHY, exponent identities, block determinant, inequalities", "E=Q(zeta_12); random samples", "PASS", "one field for masses; identities on random samples")

mass_1 0.0833333333333333333333333333333 (1/12= 0.0833333333333333333333333333333 )  mass_2 0.00347222222222222222222222222222 (1/288= 0.00347222222222222222222222222222 )
lemma ratio l=1: 0.0416666666666666666666666666667  mass_2/mass_1: 0.0416666666666666666666666666667
l=2: 0.0277777777777777777777777777778 0.0277777777777777777777777777778
exponent identities ok
2 1.0000000000000002
3 -0.9999999999999994
5 -0.9999999999999971
OT violations 0
1 3.14159265358979323846264338328 3.14159265358979323846264338328
2 4.93480220054467930941724549994 4.93480220054467930941724549994
3 5.16771278004997002924605251118 5.16771278004997002924605251118
7 0.599264529320792076887739383546 0.599264529320792076887739383546
gaussian binomial bounds True
[N4a/tests.py: exit code 0, 0.2 s]
RECORDED: N4a | pbm masses, (C0), pbe:link(a), Ostrowski–Taussky, Hua, pbm:crude | PASS


**`e4.py`: finite-group lemmas of (E4)** (`pbm:pos`(c), `pbm:pair`, `pbm:first`) by brute force over $\mathrm{GL}_r(\mathbb F_q)$ for chain data of type $(1,1)$ with $(q,m)=(3,1)$ and $(2,2)$. The script prints the averaged function $\Phi$, the average of $f$ over $\Omega$ (which `pbm:pair` identifies with $\Phi$ of the reduced datum $D'$, here trivial, i.e. 1) and the sum over $\{h: he_1=e_1\}$ (`pbm:first`). Hand values from the proof for $\mathrm{GL}_3(\mathbb F_3)$: $\Phi=q^5/((q^2-1)(q^3-1))=243/208$, $\Omega$-average $=1$, sum $=q^{2(r-1)}\cdot 9=729$. *Refutation would be:* any of these three values differing. For $\mathrm{GL}_4(\mathbb F_2)$ no hand value was recorded; the output is shown, and only the $\Omega$-average $=1$ and positivity are checked. About 30 s.

In [40]:
%%writefile N4a/e4.py
# brute-force checks of pbm:pos(c), pbm:pair, pbm:first for chain data of type (1,1), m=1 and (1,1), m=2 over F_q
import itertools, cmath, sys
import numpy as np
def run(q,m):
    r=m+2
    # basis: e1 ; X' (m vectors, z = first of them) ; e1*    zeta = coordinate of the LAST vector of X' (zeta(z) may be 0 if m>=2)
    def inv(M):
        M=[row[:] for row in M]; n=len(M); I=[[int(i==j) for j in range(n)] for i in range(n)]
        for c in range(n):
            p=next((i for i in range(c,n) if M[i][c]%q),None)
            if p is None: return None
            M[c],M[p]=M[p],M[c]; I[c],I[p]=I[p],I[c]
            a=pow(M[c][c],-1,q); M[c]=[x*a%q for x in M[c]]; I[c]=[x*a%q for x in I[c]]
            for i in range(n):
                if i!=c and M[i][c]%q:
                    t=M[i][c]; M[i]=[(x-t*y)%q for x,y in zip(M[i],M[c])]; I[i]=[(x-t*y)%q for x,y in zip(I[i],I[c])]
        return I
    def mul(A,B): return tuple(tuple(sum(A[i][k]*B[k][j] for k in range(r))%q for j in range(r)) for i in range(r))
    G=[]
    for ent in itertools.product(range(q),repeat=r*r):
        M=[list(ent[i*r:(i+1)*r]) for i in range(r)]
        Mi=inv(M)
        if Mi is not None: G.append((tuple(map(tuple,M)),tuple(map(tuple,Mi))))
    # U: identity + first row (cols 1..r-1) + last column (rows 0..r-2)
    U=[]
    for top in itertools.product(range(q),repeat=r-1):
        for col in itertools.product(range(q),repeat=r-2):
            M=[[int(i==j) for j in range(r)] for i in range(r)]
            for j in range(1,r): M[0][j]=top[j-1]
            for i in range(1,r-1): M[i][r-1]=col[i-1]
            U.append(tuple(map(tuple,M)))
    Uset=set(U)
    w=cmath.exp(2j*cmath.pi/q)
    zl=r-2  # index of the vector of X' on which zeta is the coordinate
    psi=lambda u: w**((u[0][1]+u[zl][r-1])%q)   # eps_1(u z) with z = 2nd basis vector ; zeta(u e1*)
    def fD(h,hi):
        s=0
        for u in U:
            v=mul(mul(h,u),hi)
            if v in Uset: s+=psi(v)*psi(u).conjugate()
        return s
    tot=0; om=0; nom=0; fix=0
    for h,hi in G:
        v=fD(h,hi); tot+=v
        if h[r-1][0]%q and hi[r-1][0]%q: om+=v; nom+=1
        if all(h[i][0]==int(i==0) for i in range(r)): fix+=v
    Phi=(tot/len(G)).real
    print("q",q,"m",m,"#G",len(G),"Phi",Phi,"  sum_Omega f/#Omega",(om/nom).real,"  sum_{he1=e1} f",fix.real)
    return Phi
run(3,1)
run(2,2)

Overwriting N4a/e4.py


In [41]:
o = run("N4a", "e4.py")
rows = re.findall(r"q (\d) m (\d) #G (\d+) Phi ([-\d.e]+)\s+sum_Omega f/#Omega ([-\d.e]+)\s+sum_\{he1=e1\} f ([-\d.e]+)", o)
for r in rows: print("q,m =", r[0], r[1], " Phi =", r[3], " Omega-average =", r[4], " sum over he1=e1 =", r[5])
assert len(rows) == 2
(q1, m1, _, P1, A1, S1), (q2, m2, _, P2, A2, S2) = rows
assert abs(float(P1) - 243/208) < 1e-12 and abs(float(A1) - 1) < 1e-12 and abs(float(S1) - 729) < 1e-9
assert abs(float(A2) - 1) < 1e-12 and float(P2) > 0 and float(S2) > 0
record("N4a", "pbm:pos(c), pbm:pair, pbm:first", "finite-group lemmas of (E4) by brute force vs hand values", "GL_3(F_3) (m=1); GL_4(F_2) (m=2, partial check)", "PASS", "two tiny groups; hand values only for GL_3(F_3)")

q 3 m 1 #G 11232 Phi 1.1682692307692308   sum_Omega f/#Omega 1.0   sum_{he1=e1} f 729.0
q 2 m 2 #G 20160 Phi 1.2698412698412698   sum_Omega f/#Omega 1.0   sum_{he1=e1} f 3072.0
[N4a/e4.py: exit code 0, 12.5 s]
q,m = 3 1  Phi = 1.1682692307692308  Omega-average = 1.0  sum over he1=e1 = 729.0
q,m = 2 2  Phi = 1.2698412698412698  Omega-average = 1.0  sum over he1=e1 = 3072.0
RECORDED: N4a | pbm:pos(c), pbm:pair, pbm:first | PASS


#### N4b
**`t1_section.py`: `pbo:section` and the formula $M(Z,W)$ of `pbo:sizes`(c).** $\gamma_Z\in G$, $\tilde Z(\gamma_Z)=Z$, $\gamma_ZF=F^\vee(Z)Z$, the formula for $M(Z,Z^*)$, $Y(Z,Z^*)=Y(Z)$, and $\lvert\det\mathrm{Gram}(F\mid\gamma_ZF)\rvert=g(Z)$, for $(\rho,k)\in\{(0,1),(0,3),(1,1),(1,3),(2,2),(3,4)\}$, about 190 random $Z$ each. *Refutation would be:* an assertion error (residual above tolerance).

In [42]:
%%writefile N4b/t1_section.py
# Test of Lemma pbo:section and of the formula M(Z,W) in pbo:sizes(c)
import numpy as np
rng=np.random.default_rng(1)
def run(rho,k):
    n=rho+1; r=k+2*rho+1; k0=0
    # basis order: e_1..e_rho, x_1..x_k, z, e*_1..e*_rho
    J1=np.zeros((r,r))
    for a in range(rho):
        J1[a,rho+k+1+a]=1; J1[rho+k+1+a,a]=1
    for i in range(k): J1[rho+i,rho+i]=1
    J1[rho+k,rho+k]=-1
    F=np.zeros((r,n)); Fd=np.zeros((r,n))
    for a in range(rho):
        F[a,a]=1; Fd[rho+k+1+a,a]=1
    F[rho+k,rho]=1; Fd[rho+k0,rho]=1
    def Delta(x):
        d=np.eye(n,dtype=complex); d[-1,-1]=x; return d
    assert np.allclose(F.T@J1@F,Delta(0)-np.eye(n)) and np.allclose(F.T@J1@Fd,Delta(0))
    def Ydef(Z):
        zeta=np.linalg.inv(Z)[-1,:]
        Y=np.zeros((n,n),dtype=complex); Y[-1,-1]=-1
        for a in range(rho):
            for b in range(n):
                c=0.5 if b<rho else 1.0
                Y[a,b]=-c*np.conj(zeta[a])*zeta[b]
        return Y
    def nu(Z):
        xi=np.linalg.inv(Z)[-1,-1]; return np.sqrt(1-abs(xi)**2)
    def Fv(Z): return F@Ydef(Z)+Fd@Delta(nu(Z))
    ok=0
    for trial in range(200):
        Z=rng.normal(size=(n,n))+1j*rng.normal(size=(n,n))
        Z*=3
        xi=np.linalg.inv(Z)[-1,-1]
        if abs(xi)>=1: continue
        Zs=Z.conj().T
        # gamma maps cols of F Z^-1 -> Fv(Z), cols of Fv(Z*) -> F (Z*)^-1, identity on complement
        src=np.hstack([F@np.linalg.inv(Z),Fv(Zs)]); dst=np.hstack([Fv(Z),F@np.linalg.inv(Zs)])
        comp=[i for i in range(rho,rho+k) if i!=rho+k0]
        E=np.eye(r)[:,comp]
        S=np.hstack([src,E]); D=np.hstack([dst,E])
        g=D@np.linalg.inv(S)
        e1=np.abs(g.conj().T@J1@g-J1).max()
        e2=np.abs(F.T@J1@g@F-Z).max()
        e3=np.abs(g@F-Fv(Z)@Z).max()
        # M(Z,W) formula
        W=Zs
        def Y2(Z,W): return Delta(0)-np.eye(n)-Delta(0)@np.linalg.inv(W)@(np.eye(n)-Delta(0.5))@np.linalg.inv(Z)@Delta(2)
        e4=np.abs(Y2(Z,Zs)-Ydef(Z)).max()
        sig=np.sqrt(1-np.linalg.inv(Z)[-1,-1]*np.linalg.inv(W)[-1,-1])
        FvZW=F@Y2(Z,W)+Fd@Delta(sig)
        M=np.eye(r)+(FvZW@Z-F)@F.T+((F@np.linalg.inv(W)-FvZW@Z@Y2(W,Z))@np.linalg.inv(Delta(sig))-Fd)@Fd.T
        e5=np.abs(M-g).max()
        # g(Z) = |det Z|^2 nu^2 ; Gram det of (F | gF)
        gg=abs(np.linalg.det(Z))**2*(1-abs(xi)**2)
        Gm=np.block([[Delta(0)-np.eye(n),Z],[Zs,Delta(0)-np.eye(n)]])
        e6=abs(abs(np.linalg.det(Gm))-gg)/gg
        assert max(e1,e2,e3,e4,e5)<1e-7*max(1,np.abs(g).max()**2) and e6<1e-8,(e1,e2,e3,e4,e5,e6)
        ok+=1
    print("rho,k=",rho,k,"cases ok:",ok)
for rho,k in [(0,1),(0,3),(1,1),(1,3),(2,2),(3,4)]: run(rho,k)

Overwriting N4b/t1_section.py


In [43]:
o = run("N4b", "t1_section.py")
cnt = [int(x) for x in re.findall(r"cases ok: (\d+)", o)]
assert len(cnt) == 6 and min(cnt) > 150
record("N4b", "pbo:section, pbo:sizes(c)", "section gamma_Z, formula M(Z,W), Gram determinant", "6 (rho,k) up to (3,4), ~190 random Z each", f"PASS ({sum(cnt)} cases)", "small rho,k")

rho,k= 0 1 cases ok: 182
rho,k= 0 3 cases ok: 192
rho,k= 1 1 cases ok: 194
rho,k= 1 3 cases ok: 195
rho,k= 2 2 cases ok: 194
rho,k= 3 4 cases ok: 197
[N4b/t1_section.py: exit code 0, 0.4 s]
RECORDED: N4b | pbo:section, pbo:sizes(c) | PASS (1154 cases)


**`t2_siegel.py`: Siegel's formula with every constant (`pff:siegelformula`).** For $E=\mathbb Q(\zeta_{12})$ (unramified over $\mathbb Q(\sqrt3)$), $\alpha=1$, $\Phi=1_{\mathcal O^k}$ the formula is equivalent to $\mathrm{Mass}(U_{k-1})=\mathrm{Mass}(U_k)\lvert\mathrm{disc}E\rvert^{1/4}(\pi^k/(k-1)!)^f\mathrm{vol}^k/L(k,\chi^k)$, with Gan–Hanke–Yu masses. *Refutation would be:* a ratio different from 1.

In [44]:
%%writefile N4b/t2_siegel.py
# Test of the constant in Theorem pff:siegelformula (Siegel's formula for x^*x), in a case that specialises:
# E = Q(zeta_12), E^+ = Q(sqrt 3) (E/E^+ unramified at all finite places), f = 2, |disc E| = 144, alpha = 1, Phi = 1_{O^k}.
# Full level: sum_c r_c(1)/#Aut = Mass(U_k) * |disc E|^{1/4} * prod_tau pi^k/(k-1)! * vol^k * prod_v beta_v(1),
# beta_v(1) = 1-(-q)^{-k} (inert), 1-q^{-k} (split): prod = 1/L_{E^+}(k, chi^k).   vol = (2/rd_E)^f = 2^f/sqrt|disc E|.
# Left side = Mass(U_{k-1}) (pairs (L,v), v^*v=1  <->  L = Ov + L').
# Masses from Gan-Hanke-Yu: Mass(U_m) = 2^{-mf} * 2 * prod_{r=1}^m |L_{E^+}(1-r, chi^r)|.
from mpmath import mp, mpf, pi, zeta, sqrt, factorial, dirichlet, bernpoly
mp.dps = 30
chi4=[0,1,0,-1]; chi3=[0,1,-1]; chi12=[0,1,0,0,0,-1,0,-1,0,0,0,1]
def Lneg(chi,n):   # L(1-n, chi) = -B_{n,chi}/n
    fN=len(chi); B=fN**(n-1)*sum(chi[a%fN]*bernpoly(n,mpf(a)/fN) for a in range(1,fN+1)) if fN>1 else None
    return -B/n
def L_Eplus_chi(s):  # L_{E^+}(s,chi) = L(s,chi_-4) L(s,chi_-3)
    return dirichlet(s,chi4)*dirichlet(s,chi3)
def zeta_Eplus(s): return zeta(s)*dirichlet(s,chi12)
def Lfac(r):   # |L_{E^+}(1-r, chi^r)|
    if r%2==1: return abs(Lneg(chi4,r)*Lneg(chi3,r))
    return abs(zeta(1-r)*Lneg(chi12,r))
f=2; disc=144
def mass(m):
    v=mpf(2)**(-m*f)*2
    for r in range(1,m+1): v*=Lfac(r)
    return v
print("Mass U(1),U(2),U(3),U(4):",[mass(m) for m in (1,2,3,4)], " (1/12, 1/288 expected for m=1,2)")
vol=mpf(2)**f/sqrt(disc)
for k in (3,4,5):
    dens = 1/L_Eplus_chi(k) if k%2==1 else 1/zeta_Eplus(k)
    rhs = mass(k)*mpf(disc)**0.25*(pi**k/factorial(k-1))**f*vol**k*dens
    print("k=%d: lhs Mass(U(k-1)) = %s   rhs = %s   ratio = %s"%(k,mass(k-1),rhs,rhs/mass(k-1)))

Overwriting N4b/t2_siegel.py


In [45]:
o = run("N4b", "t2_siegel.py")
ratios = [float(x) for x in re.findall(r"ratio = ([\d.]+)", o)]
assert len(ratios) == 3 and all(abs(r-1) < 1e-25 for r in ratios)
record("N4b", "pff:siegelformula", "constant in Siegel's formula vs Gan–Hanke–Yu masses", "E=Q(zeta_12), k=3,4,5", "PASS (ratio 1 to 25+ digits)", "one field, full level")

Mass U(1),U(2),U(3),U(4): [mpf('0.0833333333333333333333333333333292'), mpf('0.00347222222222222222222222222222192'), mpf('0.0000964506172839506172839506172839383'), mpf('0.0000092431841563786008230452674897109')]  (1/12, 1/288 expected for m=1,2)
k=3: lhs Mass(U(k-1)) = 0.00347222222222222222222222222222   rhs = 0.00347222222222222222222222222222   ratio = 1.0
k=4: lhs Mass(U(k-1)) = 0.0000964506172839506172839506172839   rhs = 0.0000964506172839506172839506172839   ratio = 1.0
k=5: lhs Mass(U(k-1)) = 0.00000924318415637860082304526748971   rhs = 0.0000092431841563786008230452674897   ratio = 0.999999999999999999999999999999
[N4b/t2_siegel.py: exit code 0, 0.1 s]
RECORDED: N4b | pff:siegelformula | PASS (ratio 1 to 25+ digits)


**`t3_degree_orbital.py`:** (1) formal degree $\prod(\lambda_i-\lambda_j)/(j-i)$ for the compact-dual Haar measure on $U(k,1)$, scalar holomorphic series, $k=1,2,3,5$; (2) $c_G=k/\pi$ at $\rho=0$ (equality in `pbo:meanvalue`); (3) the orbital identity (`pbr:eq-unstable`) on $U(1,1)$ for three parameters; (4) Monte Carlo of the Beta$(1,k)$ law of $\lvert g_{nn}\rvert^2$ on $U(k+1)$. *Refutation would be:* an assertion error in (1)–(3), or moments in (4) off by more than sampling error.

In [46]:
%%writefile N4b/t3_degree_orbital.py
# (1) Formal degree in the normalisation by the compact dual (pbr:degree / pds:haar), scalar holomorphic discrete
#     series of U(k,1): |c(g)|^2 = |g_zz|^{-2K}; for the Haar measure compatible with mass 1 on U(k+1) the law of
#     s=|g_zz|^2 is k (s-1)^{k-1} ds on [1,oo) (dual of Beta(1,k) on U(k+1)).  Parameter lambda = (k/2-1,...,-k/2; -K+k/2).
#     Claim: int |c|^2 = 1/prod_{i<j}(lambda_i-lambda_j)/(j-i).
# (2) The same density against c_G of pbo:eq-cG at rho=0 (r=k+1): c_G = k/pi, i.e. exact equality in pbo:meanvalue.
# (3) pbr:eq-unstable for U(1,1): c(g)=det(g)^{mu1} g_22^{-(h+1)}, (g^-1 t g)_22 = t2 s - t1 (s-1), law of s: ds on [1,oo):
#     d*int c(g^-1 t g) dg = - t^kappa/(t1-t2), kappa=(mu1, mu1-h), d=h.
# (4) Monte-Carlo check that |g_nn|^2 ~ Beta(1,k) on U(k+1).
from mpmath import mp, mpf, quad, inf, factorial, pi, exp, mpc, gamma
import numpy as np
mp.dps=25
for k in (1,2,3,5):
    for K in (k+2,k+5,3*k+7):
        lam=[mpf(k)/2-i for i in range(1,k+1)]+[-K+mpf(k)/2]
        d=mpf(1)
        for i in range(k+1):
            for j in range(i+1,k+1): d*=(lam[i]-lam[j])/(j-i)
        I=quad(lambda s:k*(s-1)**(k-1)*s**(-K),[1,2,inf])
        assert abs(I*d-1)<mpf(10)**(-15),(k,K,I*d)
print("(1) formal degree U(k,1), scalar holomorphic: int|c|^2 * prod(lam_i-lam_j)/(j-i) = 1  OK")
for k in (1,2,3,4,7):
    volU=lambda j: mp.fprod(2*pi**i/factorial(i-1) for i in range(1,j+1))
    cG=volU(k)/volU(k+1)*2*pi**k/factorial(k-1)
    assert abs(cG-k/pi)<mpf(10)**(-20)
print("(2) c_G = k/pi at rho=0: pushforward of dg to Z is (k/pi)(|Z|^2-1)^{k-1} dZ = c_G g(Z)^{k-1} dZ  OK")
for (h,mu1,th1,th2) in [(3,2,0.3,1.7),(5,-1,2.2,-0.4),(8,4,1.0,2.9)]:
    t1=exp(mpc(0,th1)); t2=exp(mpc(0,th2))
    I=quad(lambda s:(t2*s-t1*(s-1))**(-(h+1)),[1,2,10,inf])
    lhs=h*(t1*t2)**mu1*I
    rhs=-(t1**mu1*t2**(mu1-h))/(t1-t2)
    assert abs(lhs-rhs)<mpf(10)**(-12),(lhs,rhs)
print("(3) U(1,1): d int c(g^-1 t g) dg = (-1)^{pq} t^kappa/(t1-t2)  OK")
rng=np.random.default_rng(0)
for k in (1,2,4):
    n=k+1; N=200000
    z=rng.normal(size=(N,n))+1j*rng.normal(size=(N,n)); z/=np.linalg.norm(z,axis=1)[:,None]
    s=np.abs(z[:,0])**2
    # Beta(1,k): E s = 1/(k+1), E s^2 = 2/((k+1)(k+2))
    print("(4) k=%d  E s=%.4f (%.4f)  E s^2=%.4f (%.4f)"%(k,s.mean(),1/(k+1),(s**2).mean(),2/((k+1)*(k+2))))

Overwriting N4b/t3_degree_orbital.py


In [47]:
o = run("N4b", "t3_degree_orbital.py")
assert o.count("OK") == 3
mc = re.findall(r"E s=([\d.]+) \(([\d.]+)\)  E s\^2=([\d.]+) \(([\d.]+)\)", o)
assert len(mc) == 3 and all(abs(float(a)-float(b)) < 3e-3 and abs(float(c)-float(d)) < 3e-3 for a, b, c, d in mc)
record("N4b", "pbr:degree, pbo:meanvalue, pbr:eq-unstable", "formal degree, c_G = k/pi, orbital identity on U(1,1), Beta(1,k) law", "U(k,1), k<=5; U(1,1)", "PASS", "rank <= 6; scalar holomorphic series only")

(1) formal degree U(k,1), scalar holomorphic: int|c|^2 * prod(lam_i-lam_j)/(j-i) = 1  OK
(2) c_G = k/pi at rho=0: pushforward of dg to Z is (k/pi)(|Z|^2-1)^{k-1} dZ = c_G g(Z)^{k-1} dZ  OK
(3) U(1,1): d int c(g^-1 t g) dg = (-1)^{pq} t^kappa/(t1-t2)  OK
(4) k=1  E s=0.4999 (0.5000)  E s^2=0.3331 (0.3333)
(4) k=2  E s=0.3332 (0.3333)  E s^2=0.1667 (0.1667)
(4) k=4  E s=0.1998 (0.2000)  E s^2=0.0667 (0.0667)
[N4b/t3_degree_orbital.py: exit code 0, 0.2 s]
RECORDED: N4b | pbr:degree, pbo:meanvalue, pbr:eq-unstable | PASS


**`t4_haar_ratio.py`: `prp:fibre` / `pfr:haar`** in the smallest analogue (one hyperbolic plane, $G=U(1,1)$): $\eta_0*\lambda_0*\eta_0=4\pi^3\,dg$, which confirms the factor 4 per hyperbolic pair, the factor 1/2 of the Heisenberg fibre and $C=\mathrm{vol}U(r)/2^r$. *Refutation would be:* a ratio clearly different from 1 (quadrature error is about $10^{-4}$).

In [48]:
%%writefile N4b/t4_haar_ratio.py
# Test of the constants of Lemma prp:fibre / pfr:haar (factor 4 per hyperbolic pair, Lebesgue Heisenberg measure, cell density)
# in the smallest analogue: V = one hyperbolic plane (e,e*), G = U(1,1), M = eta_0 * lambda_0 * eta_0,
# g = n(i t1) w_c^{-1} n(i t2), dM = dt1 dc dt2.  The proof of prp:fibre gives M <= 4 * volU(2)/2^2 dg = 4 pi^3 dg
# (dg compatible with mass 1 on U(2)); the parametrisation is injective on the big cell, so equality is expected.
# For dg the law of s=|g_--|^2 is ds on [1,oo).  Here s = (u A + 1/u + 2 - 2 t1 t2)/4, u=|c|^2, A=(1+t1^2)(1+t2^2).
import numpy as np
def vol(S,n=6000):
    # the region m>0, m^2>=4A is unbounded along t1=t2 -> use t = tan(phi) substitution on both variables
    ph=(np.arange(n)+0.5)/n*np.pi-np.pi/2
    t=np.tan(ph); w=(np.pi/n)/np.cos(ph)**2
    T1,T2=np.meshgrid(t,t,indexing='ij'); W=np.outer(w,w)
    A=(1+T1*T1)*(1+T2*T2); m=4*S-2+2*T1*T2
    d=m*m-4*A
    f=np.where((m>0)&(d>0),np.pi*np.sqrt(np.maximum(d,0))/A,0.0)
    return float((f*W).sum())
for S in (1.5,2.0,4.0):
    v=vol(S); print("S=%.1f  M{s<=S} = %.4f   4 pi^3 (S-1) = %.4f   ratio %.5f"%(S,v,4*np.pi**3*(S-1),v/(4*np.pi**3*(S-1))))

Overwriting N4b/t4_haar_ratio.py


In [49]:
o = run("N4b", "t4_haar_ratio.py")
ratios = [float(x) for x in re.findall(r"ratio ([\d.]+)", o)]
assert len(ratios) == 3 and all(abs(r-1) < 5e-4 for r in ratios)
record("N4b", "prp:fibre, pfr:haar", "Haar-measure constant 4 pi^3 on U(1,1)", "S = 1.5, 2, 4", f"PASS (ratios {min(ratios):.5f}-{max(ratios):.5f})", "one hyperbolic plane; quadrature")

S=1.5  M{s<=S} = 62.0179   4 pi^3 (S-1) = 62.0126   ratio 1.00009
S=2.0  M{s<=S} = 124.0316   4 pi^3 (S-1) = 124.0251   ratio 1.00005
S=4.0  M{s<=S} = 372.0848   4 pi^3 (S-1) = 372.0753   ratio 1.00003
[N4b/t4_haar_ratio.py: exit code 0, 4.3 s]
RECORDED: N4b | prp:fibre, pfr:haar | PASS (ratios 1.00003-1.00009)


#### N4c
**`tests.py`:** in split/finite models — (A) `pb:singular` (a counting inequality for singular pairings over $\mathrm{GL}_r(\mathbb F_p)$, 5 cases, 3 positive-type weights); (B) `pbm:chain` (the character on the conjugate place is minus the chain datum, 3 cases); (C) `pbs:stab` (exactly one element of $U$ conjugates into the stabiliser, exhaustive over $U$, 3 cases); (D) the descent step of `pbe:scalar` (2 cases, 200 trials). *Refutation would be:* any `False`.

In [50]:
%%writefile N4c/tests.py
# Tests for node N4c: pb:singular, pbm:chain, pbs:stab, pbe:scalar (descent), pb:central, in split/finite models.
import itertools, random
import numpy as np
random.seed(1)

def inv_mod(A, p):
    n = A.shape[0]; M = np.concatenate([A % p, np.eye(n, dtype=np.int64)], axis=1)
    for c in range(n):
        piv = next((r for r in range(c, n) if M[r, c] % p), None)
        if piv is None: return None
        M[[c, piv]] = M[[piv, c]]
        M[c] = (M[c] * pow(int(M[c, c]), -1, p)) % p
        for r in range(n):
            if r != c and M[r, c]: M[r] = (M[r] - M[r, c] * M[c]) % p
    return M[:, n:]

def det_mod(A, p):
    A = A.copy() % p; n = A.shape[0]; d = 1
    for c in range(n):
        piv = next((r for r in range(c, n) if A[r, c] % p), None)
        if piv is None: return 0
        if piv != c: A[[c, piv]] = A[[piv, c]]; d = -d
        d = d * int(A[c, c]) % p
        iv = pow(int(A[c, c]), -1, p)
        for r in range(c + 1, n):
            if A[r, c]: A[r] = (A[r] - A[r, c] * iv * A[c]) % p
    return d % p

def GL(r, p):
    out = []
    for t in itertools.product(range(p), repeat=r * r):
        A = np.array(t, dtype=np.int64).reshape(r, r)
        if det_mod(A, p): out.append(A)
    return out

def key(A): return A.tobytes()

# ---------------- Test A: Lemma pb:singular ----------------
def test_singular(r, p, l, trials=20):
    G = GL(r, p)
    idx = {key(g): i for i, g in enumerate(G)}
    ok = True
    # positive type beta >= 0: (1) permutation character on F^r (number of fixed vectors);
    # (2) square of it; (3) indicator of a subgroup (upper unitriangular)
    def fixed(g): return p ** (r - np.linalg.matrix_rank(((g - np.eye(r, dtype=np.int64)) % p).astype(float))) if p == 2 else None
    betas = []
    fx = []
    vecs = [np.array(v, dtype=np.int64) for v in itertools.product(range(p), repeat=r)]
    for g in G:
        fx.append(sum(1 for v in vecs if np.array_equal(g @ v % p, v)))
    betas.append(np.array(fx, float))
    betas.append(np.array(fx, float) ** 2)
    betas.append(np.array([1.0 if all(g[i, j] == (1 if i == j else 0) for i in range(r) for j in range(i + 1)) else 0.0 for g in G]))
    worst = 0
    for beta in betas:
        for _ in range(trials):
            A0 = np.array([[random.randrange(p) for _ in range(l)] for _ in range(r)], dtype=np.int64)
            A1 = np.array([[random.randrange(p) for _ in range(l)] for _ in range(r)], dtype=np.int64)
            Y = 0
            for v in itertools.product(range(p), repeat=l):
                if not any(v): continue
                v = np.array(v, dtype=np.int64)
                z = A0 @ v % p; a = A1 @ v % p
                s1 = sum(b for g, b in zip(G, beta) if np.array_equal(g @ z % p, z))
                # tpi(s)^{-1} a = a  <=>  tpi(s) a = a
                s2 = sum(b for g, b in zip(G, beta) if np.array_equal(g.T @ a % p, a))
                Y = max(Y, s1, s2)
            lhs = p * sum(b for g, b in zip(G, beta) if det_mod(A1.T @ g @ A0 % p, p) == 0)
            rhs = 2 * beta.sum() + p ** (l + r / 2) * Y
            worst = max(worst, lhs / rhs)
            ok &= lhs <= rhs + 1e-9
    return ok, worst

print("A pb:singular r=2 p=3 l=1:", test_singular(2, 3, 1))
print("A pb:singular r=2 p=3 l=2:", test_singular(2, 3, 2))
print("A pb:singular r=3 p=2 l=1:", test_singular(3, 2, 1))
print("A pb:singular r=3 p=2 l=2:", test_singular(3, 2, 2))
print("A pb:singular r=2 p=5 l=1:", test_singular(2, 5, 1, trials=8))

# ---------------- split model of the chosen place ----------------
# basis order: e_1..e_rho ; x_1..x_k, z ; e*_rho..e*_1 ; symmetric form J.
def model(rho, k, p, N):
    r = 2 * rho + k + 1
    J = np.zeros((r, r), dtype=np.int64)
    for a in range(rho):
        J[a, r - 1 - a] = J[r - 1 - a, a] = 1
    for i in range(k): J[rho + i, rho + i] = 1
    J[rho + k, rho + k] = (-N) % p
    return r, J
def rand_U(rho, k, p):
    # unipotent radical for the flag A_1<...<A<A^perp<...: strictly block upper triangular, blocks 1..1,k+1,1..1
    r = 2 * rho + k + 1
    blocks = [[a] for a in range(rho)] + [list(range(rho, rho + k + 1))] + [[rho + k + 1 + a] for a in range(rho)]
    u = np.eye(r, dtype=np.int64)
    for bi in range(len(blocks)):
        for bj in range(bi + 1, len(blocks)):
            for i in blocks[bi]:
                for j in blocks[bj]:
                    u[i, j] = random.randrange(p)
    return u
def all_U(rho, k, p):
    r = 2 * rho + k + 1
    blocks = [[a] for a in range(rho)] + [list(range(rho, rho + k + 1))] + [[rho + k + 1 + a] for a in range(rho)]
    pos = [(i, j) for bi in range(len(blocks)) for bj in range(bi + 1, len(blocks)) for i in blocks[bi] for j in blocks[bj]]
    for t in itertools.product(range(p), repeat=len(pos)):
        u = np.eye(r, dtype=np.int64)
        for (i, j), v in zip(pos, t): u[i, j] = v
        yield u
def contra(u, J, p):  # action at the conjugate place: J^{-1} tu^{-1} J
    return inv_mod(J, p) @ inv_mod(u, p).T @ J % p

# ---------------- Test B: Lemma pbm:chain ----------------
def test_chain(rho, k, p, N, trials=300):
    r, J = model(rho, k, p, N)
    zi = rho + k
    estar = lambda a: r - a          # index of e*_a (a=1..rho)
    ok = True
    for _ in range(trials):
        u = rand_U(rho, k, p); ub = contra(u, J, p)
        # sigma at w and at wbar: entries just above the diagonal of the block on the frame
        sig = lambda g: (sum(g[a - 1, a] for a in range(1, rho)) + g[rho - 1, zi]) % p
        # chain datum: eps*_{a+1}(u e*_a) and zeta(u e*_rho), zeta(y)= tz J y
        zeta_row = J[zi]
        sec = (sum(u[estar(a + 1), estar(a)] for a in range(1, rho)) + int(zeta_row @ u[:, estar(rho)])) % p
        ok &= (sig(ub) == (-sec) % p)
        # ub is again in the same unipotent group
        ok &= all(ub[i, j] == (1 if i == j else 0) for i in range(r) for j in range(r)
                  if (i > j and not (rho <= i <= rho + k and rho <= j <= rho + k)) or (rho <= i <= rho + k and rho <= j <= rho + k) or (i==j))
    return ok
print("B pbm:chain rho=1,k=2,p=5:", test_chain(1, 2, 5, 2))
print("B pbm:chain rho=2,k=3,p=7:", test_chain(2, 3, 7, 3))
print("B pbm:chain rho=3,k=2,p=11:", test_chain(3, 2, 11, 5))

# ---------------- Test C: Lemma pbs:stab (split model: only det Z != 0 at both places is used) ----------------
def test_stab(rho, k, p, N, ngamma=6):
    r, J = model(rho, k, p, N)
    frame = list(range(rho)) + [rho + k]
    Us = list(all_U(rho, k, p))
    def fixes_frame(u):
        ub = contra(u, J, p)
        return all(np.array_equal(u[:, f], np.eye(r, dtype=np.int64)[:, f]) and np.array_equal(ub[:, f], np.eye(r, dtype=np.int64)[:, f]) for f in frame)
    U0 = set(key(u) for u in Us if fixes_frame(u))
    done = 0; ok = True; bad_when_singular = 0
    while done < ngamma:
        g = np.array([[random.randrange(p) for _ in range(r)] for _ in range(r)], dtype=np.int64)
        if not det_mod(g, p): continue
        gi = inv_mod(g, p); gb = contra(g, J, p)
        Zw = np.array([[int(J[fa] @ g[:, fb]) % p for fb in frame] for fa in frame], dtype=np.int64)
        Zb = np.array([[int(J[fa] @ gb[:, fb]) % p for fb in frame] for fa in frame], dtype=np.int64)
        if not (det_mod(Zw, p) and det_mod(Zb, p)): continue
        done += 1
        cnt = sum(1 for w in Us if key(g @ w @ gi % p) in U0)
        ok &= (cnt == 1)
    return ok, len(Us), len(U0)
print("C pbs:stab rho=1,k=1,p=3:", test_stab(1, 1, 3, 1))
print("C pbs:stab rho=1,k=2,p=3:", test_stab(1, 2, 3, 1))
print("C pbs:stab rho=2,k=1,p=2:", test_stab(2, 1, 2, 1, ngamma=3))

# ---------------- Test D: descent step of pbe:scalar (identity for sigma at the place w) ----------------
def test_descent(rho, k, p, N, trials=200):
    r, J = model(rho, k, p, N)
    zi = rho + k; ok = True; n_used = 0
    sig = lambda g: (sum(g[a - 1, a] for a in range(1, rho)) + g[rho - 1, zi]) % p
    I = np.eye(r, dtype=np.int64)
    for _ in range(trials):
        j = random.randrange(1, rho)            # 1 <= j < rho
        zeta = random.randrange(1, p)
        # a in GL(A): arbitrary invertible on A_j, preserves A_i and acts by zeta on A_{i+1}/A_i for j<=i<rho
        a = np.zeros((rho, rho), dtype=np.int64)
        while True:
            blk = np.array([[random.randrange(p) for _ in range(j)] for _ in range(j)], dtype=np.int64)
            if det_mod(blk, p): break
        a[:j, :j] = blk
        for c in range(j, rho):
            a[c, c] = zeta
            for rr in range(c): a[rr, c] = random.randrange(p)
        # gamma = l*n, l = diag(a, m, a^vee) in the Levi of P_A, n in N_A (random element of U trivial on A)
        while True:
            m = np.array([[random.randrange(p) for _ in range(k + 1)] for _ in range(k + 1)], dtype=np.int64)
            if det_mod(m, p): break
        l = np.zeros((r, r), dtype=np.int64); l[:rho, :rho] = a; l[rho:rho + k + 1, rho:rho + k + 1] = m
        P = np.zeros((rho, rho), dtype=np.int64)
        for c in range(rho): P[c, rho - 1 - c] = 1          # order e*_rho..e*_1
        l[rho + k + 1:, rho + k + 1:] = P @ inv_mod(a, p).T @ P % p
        n = I.copy()
        n[:rho, rho:] = np.array([[random.randrange(p) for _ in range(r - rho)] for _ in range(rho)], dtype=np.int64)
        n[rho:rho + k + 1, rho + k + 1:] = np.array([[random.randrange(p) for _ in range(rho)] for _ in range(k + 1)], dtype=np.int64)
        g = l @ n % p; gi = inv_mod(g, p)
        v = np.zeros(rho, dtype=np.int64); v[:j] = [random.randrange(p) for _ in range(j)]
        phi = np.zeros((rho, rho), dtype=np.int64); phi[:, j] = v      # phi(e_{j+1}) = v
        uv = I.copy(); uv[:rho, :rho] = (np.eye(rho, dtype=np.int64) + phi) % p
        uv[rho + k + 1:, rho + k + 1:] = P @ inv_mod((np.eye(rho, dtype=np.int64) + phi) % p, p).T @ P % p
        c = gi @ uv @ g % p
        # conjugate lies in the unipotent radical (strictly block upper triangular) ?
        inU = all(c[i, jj] == (1 if i == jj else 0) for i in range(r) for jj in range(r)
                  if (i > jj and not (rho <= i <= rho + k and rho <= jj <= rho + k)) or (rho <= i <= rho + k and rho <= jj <= rho + k) or i == jj)
        ainv = inv_mod(a, p)
        lhs = sig(c); rhs = zeta * int((ainv @ v)[j - 1]) % p
        ok &= inU and (sig(uv) == v[j - 1] % p) and (lhs == rhs)
        n_used += 1
    return ok, n_used
print("D pbe:scalar descent rho=3,k=1,p=5:", test_descent(3, 1, 5, 1))
print("D pbe:scalar descent rho=4,k=2,p=7:", test_descent(4, 2, 7, 3))

Overwriting N4c/tests.py


In [51]:
o = run("N4c", "tests.py")
assert "False" not in o and o.count("True") == 13
record("N4c", "pb:singular, pbm:chain, pbs:stab, pbe:scalar", "finite-field models: counting inequality, chain character, unique stabiliser element, descent", "5+3+3+2 cases over F_2..F_11", "PASS", "split model; tiny groups")

A pb:singular r=2 p=3 l=1: (np.True_, np.float64(0.2736842105263158))
A pb:singular r=2 p=3 l=2: (np.True_, np.float64(0.10344827586206896))
A pb:singular r=3 p=2 l=1: (np.True_, np.float64(0.29649024869685103))
A pb:singular r=3 p=2 l=2: (np.True_, np.float64(0.2152581701795029))
A pb:singular r=2 p=5 l=1: (np.True_, np.float64(0.24390243902439024))
B pbm:chain rho=1,k=2,p=5: True
B pbm:chain rho=2,k=3,p=7: True
B pbm:chain rho=3,k=2,p=11: True
C pbs:stab rho=1,k=1,p=3: (True, 243, 27)
C pbs:stab rho=1,k=2,p=3: (True, 2187, 243)
C pbs:stab rho=2,k=1,p=2: (True, 16384, 256)
D pbe:scalar descent rho=3,k=1,p=5: (np.True_, 200)
D pbe:scalar descent rho=4,k=2,p=7: (np.True_, 200)
[N4c/tests.py: exit code 0, 2.4 s]
RECORDED: N4c | pb:singular, pbm:chain, pbs:stab, pbe:scalar | PASS


**`opencell_c.py`: `pbo:sizes`(c).** $\gamma_Z=M(Z,Z^*)$ is an isometry with frame matrix $Z$, $\lVert M\rVert\le15P^6$, and the Cauchy bound for the derivative; 300 random $Z$ for each $(\rho,k)\in\{(1,1),(2,2),(3,2)\}$. *Refutation would be:* `False`, or a printed ratio $>1$.

In [52]:
%%writefile N4c/opencell_c.py
# Check of pbo:sizes(c) as described in B-covolume (iii): formula M(Z,W), isometry, bound 15P^6, Cauchy bound for the derivative.
import numpy as np
rng = np.random.default_rng(3)
def setup(rho, k):
    n = rho + 1; r = 2 * rho + k + 1
    # basis: e_1..e_rho, x_1..x_k, z, e*_1..e*_rho
    J = np.zeros((r, r)); 
    for a in range(rho): J[a, rho + k + 1 + a] = J[rho + k + 1 + a, a] = 1
    for i in range(k): J[rho + i, rho + i] = 1
    J[rho + k, rho + k] = -1
    F = np.zeros((r, n)); Fd = np.zeros((r, n))
    for a in range(rho): F[a, a] = 1; Fd[rho + k + 1 + a, a] = 1
    F[rho + k, rho] = 1; Fd[rho, rho] = 1      # z ; x_{k0}=x_1
    return n, r, J, F, Fd
def Delta(n, x):
    D = np.eye(n, dtype=complex); D[-1, -1] = x; return D
def Mmat(Z, W, n, r, J, F, Fd):
    Zi = np.linalg.inv(Z); Wi = np.linalg.inv(W)
    xiZ = Zi[-1, -1]; xiW = Wi[-1, -1]
    sig = np.sqrt(1 - xiZ * xiW)
    def Y(Z1i, W1i): return Delta(n, 0) - np.eye(n) - Delta(n, 0) @ W1i @ (np.eye(n) - Delta(n, .5)) @ Z1i @ Delta(n, 2)
    Fv = F @ Y(Zi, Wi) + Fd @ Delta(n, sig)
    return np.eye(r) + (Fv @ Z - F) @ F.conj().T + ((F @ Wi - Fv @ Z @ Y(Wi, Zi)) @ np.linalg.inv(Delta(n, sig)) - Fd) @ Fd.conj().T
op = lambda A: np.linalg.norm(A, 2)
for (rho, k) in [(1, 1), (2, 2), (3, 2)]:
    n, r, J, F, Fd = setup(rho, k)
    worst = 0; cnt = 0; okiso = True; okZ = True; worstD = 0
    while cnt < 300:
        Z = (rng.normal(size=(n, n)) + 1j * rng.normal(size=(n, n))) * rng.uniform(.3, 3)
        Zi = np.linalg.inv(Z); xi = Zi[-1, -1]
        if abs(xi) >= 1: continue
        nu = np.sqrt(1 - abs(xi) ** 2)
        P = max(op(Z), op(Zi), 1 / nu, 8.0); T = max(op(Z), op(Zi), 1 / nu, 4.0)
        M = Mmat(Z, Z.conj().T, n, r, J, F, Fd)
        okiso &= np.allclose(M.conj().T @ J @ M, J, atol=1e-7 * max(1, op(M) ** 2))
        okZ &= np.allclose(F.conj().T @ J @ M @ F, Z, atol=1e-8 * max(1, op(M)))
        worst = max(worst, op(M) / (15 * P ** 6))
        # real derivative in a random direction H, ||H||=1
        H = rng.normal(size=(n, n)) + 1j * rng.normal(size=(n, n)); H /= op(H)
        e = 1e-6 * T ** -6
        D = (Mmat(Z + e * H, (Z + e * H).conj().T, n, r, J, F, Fd) - Mmat(Z - e * H, (Z - e * H).conj().T, n, r, J, F, Fd)) / (2 * e)
        worstD = max(worstD, op(D) / (2 * (2 * T) ** 12 * T ** 6))
        cnt += 1
    print((rho, k), "isometry:", okiso, " Z(gamma_Z)=Z:", okZ, " max ||M||/(15P^6):", worst, " max ||dM||/(2(2T)^12 T^6):", worstD)

Overwriting N4c/opencell_c.py


In [53]:
o = run("N4c", "opencell_c.py")
rows = re.findall(r"isometry: (\w+)  Z\(gamma_Z\)=Z: (\w+)  max \|\|M\|\|/\(15P\^6\): ([\d.e-]+)  max \|\|dM\|\|/\(2\(2T\)\^12 T\^6\): ([\d.e-]+)", o)
assert len(rows) == 3 and all(a == b == "True" and float(c) <= 1 and float(d) <= 1 for a, b, c, d in rows)
record("N4c", "pbo:sizes(c)", "isometry, frame matrix, norm bound 15P^6, derivative bound", "(rho,k) = (1,1),(2,2),(3,2); 300 Z each", "PASS", "small rho,k")

(1, 1) isometry: True  Z(gamma_Z)=Z: True  max ||M||/(15P^6): 3.587293846777943e-06  max ||dM||/(2(2T)^12 T^6): 1.5591061467071842e-14
(2, 2) isometry: True  Z(gamma_Z)=Z: True  max ||M||/(15P^6): 3.2157630193559825e-06  max ||dM||/(2(2T)^12 T^6): 1.1362919636346761e-14
(3, 2) isometry: True  Z(gamma_Z)=Z: True  max ||M||/(15P^6): 3.314111331183877e-06  max ||dM||/(2(2T)^12 T^6): 7.634317700656192e-15
[N4c/opencell_c.py: exit code 0, 0.3 s]
RECORDED: N4c | pbo:sizes(c) | PASS


**`measure.py`: measure interface.** The Jacobian between the parameters of $\eta_U$ (product of Heisenberg Lebesgue measures, `prp:heis`) and the (Cv3) row coordinates is 1, also after a left translation (`pbv:haar`), for $\rho\le3$. *Refutation would be:* a Jacobian different from 1.

In [54]:
%%writefile N4c/measure.py
# Measure interface: eta_U (product of Heisenberg Lebesgue measures, prp:heis) versus Lebesgue measure in the
# (Cv3) row coordinates (rho_a, t_a) of pbv:eq-coord, and versus du^0 dx of pbo:unfold. Jacobians must be 1.
import numpy as np
rng = np.random.default_rng(5)
def setup(rho, k):
    r = 2 * rho + k + 1
    idx = {}
    for a in range(rho): idx[('e', a)] = a
    for i in range(k): idx[('x', i)] = rho + i
    idx['z'] = rho + k
    for a in range(rho): idx[('s', a)] = rho + k + 1 + a
    J = np.zeros((r, r))
    for a in range(rho): J[idx[('e', a)], idx[('s', a)]] = J[idx[('s', a)], idx[('e', a)]] = 1
    for i in range(k): J[idx[('x', i)], idx[('x', i)]] = 1
    J[idx['z'], idx['z']] = -1
    return r, idx, J
def hform(J, v, w): return v @ J @ w.conj()          # linear in the first variable (Cv1)
def Vbasis(rho, k, idx, a):                            # basis vectors of floor > a
    return [idx[('e', c)] for c in range(a + 1, rho)] + [idx[('s', c)] for c in range(a + 1, rho)] + [idx[('x', i)] for i in range(k)] + [idx['z']]
def heis(rho, k, idx, J, a, v, t):                     # n(v, it) of floor a ; v a vector of V_{a+1} (full coordinates)
    r = J.shape[0]; n = np.eye(r, dtype=complex)
    ea, sa = idx[('e', a)], idx[('s', a)]
    col = np.zeros(r, dtype=complex); col[sa] = 1; col += v; col[ea] += 1j * t - 0.5 * hform(J, v, v)
    n[:, sa] = col
    for w in Vbasis(rho, k, idx, a):
        ew = np.zeros(r, dtype=complex); ew[w] = 1
        n[ea, w] += -hform(J, ew, v)
    return n
def params_to_u(rho, k, idx, J, par):
    r = J.shape[0]; u = np.eye(r, dtype=complex); pos = 0
    for a in range(rho):
        B = Vbasis(rho, k, idx, a); d = len(B)
        v = np.zeros(r, dtype=complex); v[B] = par[pos:pos + 2 * d:2] + 1j * par[pos + 1:pos + 2 * d:2]; pos += 2 * d
        t = par[pos]; pos += 1
        u = u @ heis(rho, k, idx, J, a, v, t)
    return u
def rowcoords(rho, k, idx, u):
    out = []
    for a in range(rho):
        for w in Vbasis(rho, k, idx, a):
            out += [u[idx[('e', a)], w].real, u[idx[('e', a)], w].imag]
        out.append(u[idx[('e', a)], idx[('s', a)]].imag)
    return np.array(out)
def colcoords(rho, k, idx, u):
    out = []
    for a in range(rho):
        for w in Vbasis(rho, k, idx, a):
            out += [u[w, idx[('s', a)]].real, u[w, idx[('s', a)]].imag]
        out.append(u[idx[('e', a)], idx[('s', a)]].imag)
    return np.array(out)
def jac(f, x, e=1e-5):
    f0 = f(x); Jm = np.zeros((len(f0), len(x)))
    for i in range(len(x)):
        d = np.zeros(len(x)); d[i] = e
        Jm[:, i] = (f(x + d) - f(x - d)) / (2 * e)
    return Jm
for (rho, k) in [(1, 1), (2, 1), (2, 2), (3, 1)]:
    r, idx, J = setup(rho, k)
    dim = sum(2 * len(Vbasis(rho, k, idx, a)) + 1 for a in range(rho))
    assert dim == rho * (2 * k + 2 * rho + 1)
    res = []
    for _ in range(4):
        par = rng.normal(size=dim)
        u = params_to_u(rho, k, idx, J, par)
        assert np.allclose(u.T @ J @ u.conj(), J)                       # isometry
        assert np.allclose(colcoords(rho, k, idx, u), par)              # prp:heis(c): free column entries = (v_a,t_a)
        d1 = abs(np.linalg.det(jac(lambda p: rowcoords(rho, k, idx, params_to_u(rho, k, idx, J, p)), par)))
        # left translation by a fixed x in row coordinates
        x = params_to_u(rho, k, idx, J, rng.normal(size=dim))
        d2 = abs(np.linalg.det(jac(lambda p: rowcoords(rho, k, idx, x @ params_to_u(rho, k, idx, J, p)), par)))
        res.append((round(d1, 6), round(d2, 6)))
    print((rho, k), "dim", dim, "|det d(row coords)/d(eta_U params)|, same after left translation:", res)

Overwriting N4c/measure.py


In [55]:
o = run("N4c", "measure.py")
jac = [float(x) for x in re.findall(r"np\.float64\(([\d.]+)\)", o)] or [float(x) for x in re.findall(r"\(([\d.]+), ([\d.]+)\)", o) for x in x]
assert len(jac) == 32 and all(abs(j-1) < 1e-5 for j in jac)
record("N4c", "pbv:haar, prp:heis, (Cv3)", "Jacobian 1 between eta_U parameters and row coordinates, left invariance", "rho <= 3", "PASS", "numerical Jacobians; rho <= 3")
record("N4c", "pbg:index", "-", "-", "NOT TESTED", "no non-vacuous small case found; proof re-derived by hand only")
record("N4", "au:moment (Theorem B itself)", "-", "-", "NOT TESTED", "rank r ~ x^5; no small instance")

(1, 1) dim 5 |det d(row coords)/d(eta_U params)|, same after left translation: [(np.float64(1.0), np.float64(1.0)), (np.float64(1.0), np.float64(1.0)), (np.float64(1.0), np.float64(1.0)), (np.float64(1.0), np.float64(1.0))]
(2, 1) dim 14 |det d(row coords)/d(eta_U params)|, same after left translation: [(np.float64(1.0), np.float64(1.0)), (np.float64(1.0), np.float64(1.0)), (np.float64(1.0), np.float64(1.0)), (np.float64(1.0), np.float64(1.0))]
(2, 2) dim 18 |det d(row coords)/d(eta_U params)|, same after left translation: [(np.float64(1.0), np.float64(1.0)), (np.float64(1.0), np.float64(1.0)), (np.float64(1.0), np.float64(1.0)), (np.float64(1.0), np.float64(1.0))]
(3, 1) dim 27 |det d(row coords)/d(eta_U params)|, same after left translation: [(np.float64(1.0), np.float64(1.0)), (np.float64(1.0), np.float64(1.0)), (np.float64(1.0), np.float64(1.0)), (np.float64(1.0), np.float64(1.0))]
[N4c/measure.py: exit code 0, 0.1 s]
RECORDED: N4c | pbv:haar, prp:heis, (Cv3) | PASS
RECORDED: N4c |

### Node N5 — exchange identities (`au:factor`, `au:inj`)

The identities have constants $\lvert c_{i,n}\rvert^2=C\cdot L_*(\dots)$ and involve ranks far beyond computation; the tests check their combinatorial skeleton and a finite-field model of the Fourier step.

**`tests.py`:** (9) the sign identity of the squares $q(i,n+1)+q(i+1,n+1)=q(i,n+2)+q(i+1,n)$ for $m\le6$, $k\le4$; (11) the block rule of `ptr:match` (a),(b) for all ascending and descending pairs, $m\le5$, $k\in\{3,5\}$. *Refutation would be:* `sign identity: False` or a positive number of block-rule failures.

In [56]:
%%writefile N5/tests.py
import itertools, random, cmath, math
from fractions import Fraction as Fr
# ---- (9) sign identity of the squares, and q<=r
def E(i,n,a): return i+a+min(max(a-i,0),n)
def signs(i,n,r,k): return [(-1)**E(i,n,max(p-k,0)) for p in range(1,r+1)]
ok=True
for m in range(2,7):
  for k in range(2,5):
    r=k-1+m
    for i in range(0,m+1):
      for n in range(0,m-i-1):
        if i+n+2>m: continue
        q=lambda i_,n_: sum(1 for s in signs(i_,n_,r,k) if s<0)
        if q(i,n+1)+q(i+1,n+1)!=q(i,n+2)+q(i+1,n): ok=False; print("sign fail",m,k,i,n)
print("sign identity:",ok)
# ---- (11) block rule ptr:match (a),(b)
def param(i,n,m,k,kap,D,t):
    # returns list of (number, sign, tag)
    r=k-1+m; out=[]
    sh=Fr(1,2) if n%2==1 else 0
    stem=[Fr(k,2)-1-j for j in range(k-1)]+[-kap-ip*D for ip in range(1,m+1)]
    sg=signs(i,n,r,k)
    for p in range(r): out.append((stem[p]-sh,sg[p],('s',p)))
    for l in range(1,n+1):
        kl=kap+(i+l+Fr(1,2))*D
        for b in range(t): out.append((-kl+Fr(t-1,2)-b-sh,1,('b',i+l,b)))
    return out
def kappa(j,kap,D): return kap+(j+Fr(1,2))*D
def check_rule(rows,cols,tau0,tau0p,t,newblock):
    # rows: list (lam,eps,tag) ; cols likewise ; pairing by tag ; new block tags = newblock
    cd={c[2]:c for c in cols}
    for (lam,eps,tag) in rows:
        if tag not in cd: return "row tag missing"
        lp,ep,_=cd[tag]
        if lp!=-lam+tau0+tau0p: return ("shift",tag,lp,-lam+tau0+tau0p)
        if abs(lp-tau0p)<=Fr(t,2): return ("dist",tag)
        if ep!=eps*(1 if lp-tau0p>0 else -1): return ("sign",tag,ep,eps,lp-tau0p)
    B=[c for c in cols if c[2] in newblock]
    if sorted(c[0] for c in B)!=sorted(tau0p+Fr(t-1,2)-b for b in range(t)): return "string"
    if len(set(c[1] for c in B))!=1: return "string sign"
    if len(B)+len(rows)!=len(cols): return "count"
    return None
bad=0
for m in range(2,6):
  for k in [3,5]:
    r=k-1+m
    if r%2==0: continue
    t=5*r; D=2*r**8; kap=Fr(10**6*1+1,1)*1+D  # generic large kappa (integer or half-integer irrelevant here)
    for i in range(0,m):
      for n in range(0,m-i):
        # ascending pair (G_{i,n},G_{i,n+1})
        rows=[(-x,s,tg) for (x,s,tg) in param(i,n,m,k,kap,D,t)]
        cols=param(i,n+1,m,k,kap,D,t)
        kk=kappa(i+n+1,kap,D)
        tau0=kk+(Fr(1,2) if n%2==1 else 0); tau0p=-kk-(Fr(1,2) if n%2==0 else 0)
        res=check_rule(rows,cols,tau0,tau0p,t,{('b',i+n+1,b) for b in range(t)})
        if res: bad+=1; print("asc fail",m,k,i,n,res)
        # descending pair (G_{i+1,n},G_{i,n+1})
        rows=param(i+1,n,m,k,kap,D,t)
        cols=[(-x,s,tg) for (x,s,tg) in param(i,n+1,m,k,kap,D,t)]
        kk=kappa(i+1,kap,D)
        tau0=-kk-(Fr(1,2) if n%2==1 else 0); tau0p=kk+(Fr(1,2) if n%2==0 else 0)
        res=check_rule(rows,cols,tau0,tau0p,t,{('b',i+1,b) for b in range(t)})
        if res: bad+=1; print("desc fail",m,k,i,n,res)
print("block rule failures:",bad)

Overwriting N5/tests.py


In [57]:
o = run("N5", "tests.py")
assert "sign identity: True" in o and "block rule failures: 0" in o
record("N5", "ptr:match (a),(b); sign identity of squares", "combinatorics of the triangle", "m <= 6, k <= 4 (signs); m <= 5, k in {3,5} (blocks)", "PASS", "small m,k")

sign identity: True
block rule failures: 0
[N5/tests.py: exit code 0, 0.1 s]
RECORDED: N5 | ptr:match (a),(b); sign identity of squares | PASS


**`test_fourier.py`: `pc:pointlaw` / `pc:next` and `ptv:fourier` over $\mathbb F_3$.** With $\chi$ the quadratic character and random $L'$ satisfying the square law, the identities of `pc:next` ($\lvert N_0\rvert=\lvert T\rvert=\lvert T'\rvert=1$) and `ptv:fourier` ($\lvert N\rvert=\lvert T\rvert=1$, $\gamma=G\cdot\#M_{N\times T}$) should hold to rounding error, and $\lvert\gamma\rvert^2=27$. *Refutation would be:* errors above rounding, or the equivariance check `False`.

In [58]:
%%writefile N5/test_fourier.py
# Numerical test of Lemma pc:pointlaw / pc:next and Proposition ptv:fourier over F_3.
import itertools, cmath, random, numpy as np
q=3
def chi(a):
    a%=q
    return 0 if a==0 else (1 if a==1 else -1)
def psi(a): return cmath.exp(2j*cmath.pi*(a%q)/q)
def det(Mx):
    M=[list(r) for r in Mx]; n=len(M); d=1
    for i in range(n):
        p=None
        for r in range(i,n):
            if M[r][i]%q: p=r;break
        if p is None: return 0
        if p!=i: M[i],M[p]=M[p],M[i]; d=-d
        d=d*M[i][i]%q; inv=pow(M[i][i],q-2,q)
        for r in range(i+1,n):
            f=M[r][i]*inv%q
            M[r]=[(a-f*b)%q for a,b in zip(M[r],M[i])]
    return d%q
def inv(Mx):
    n=len(Mx); M=[list(r)+[1 if i==j else 0 for j in range(n)] for i,r in enumerate(Mx)]
    for i in range(n):
        p=[r for r in range(i,n) if M[r][i]%q][0]
        M[i],M[p]=M[p],M[i]; iv=pow(M[i][i],q-2,q); M[i]=[a*iv%q for a in M[i]]
        for r in range(n):
            if r!=i and M[r][i]%q:
                f=M[r][i]; M[r]=[(a-f*b)%q for a,b in zip(M[r],M[i])]
    return [r[n:] for r in M]
def mats(r,c):
    for t in itertools.product(range(q),repeat=r*c):
        yield tuple(tuple(t[i*c:(i+1)*c]) for i in range(r))
def mul(A,B): return tuple(tuple(sum(a*b for a,b in zip(r,col))%q for col in zip(*B)) for r in A)
def tr(A): return sum(A[i][i] for i in range(len(A)))%q
random.seed(1)
def run(U,Bidx,Aidx,checks):
    nB=len(Bidx); nA=len(Aidx)
    GLB=[h for h in mats(nB,nB) if det(h)]; GLA=[h for h in mats(nA,nA) if det(h)]
    # L' on M_{U x B}: equivariant L'(y h)=chi(det h) L'(y)
    Lp={}
    for y in mats(U,nB):
        if y in Lp: continue
        orb={}
        for h in GLB: orb.setdefault(mul(y,h),[]).append(h)
        if len(orb)<len(GLB):   # rank deficient
            for z in orb: Lp[z]=0
        else:
            r=complex(random.gauss(0,1),random.gauss(0,1))
            for z,hs in orb.items(): Lp[z]=chi(det(hs[0]))*r
    lam=0.7+0.2j
    # L on M_{A x U}: L(x_k)=lam*chi(det k)^{-1} L'(y_k)
    L={}
    others=[i for i in range(U) if i not in Aidx]
    for x in mats(nA,U):
        # complete to invertible M=k^{-1} with rows A = x
        found=None
        for extra in mats(len(others),U):
            M=[None]*U
            for a,i in enumerate(Aidx): M[i]=x[a]
            for a,i in enumerate(others): M[i]=extra[a]
            if det(M): found=M;break
        if found is None: L[x]=0; continue
        k=inv(found); yk=tuple(tuple(k[i][j] for j in Bidx) for i in range(U))
        L[x]=lam*chi(det(k))*Lp[yk]   # chi^{-1}=chi
    c=len(GLB)/(lam*len(GLA))
    return Lp,L,c,GLB,GLA
# ---- ptv:fourier with |N|=1,|T|=1: U order (N, M0, M1, T'), B=M=(1,2), A=(0,3)
Lp,L,c,GLB,GLA=run(4,[1,2],[0,3],None)
# check equivariance of L under GL_M on all x (hypothesis of ptv:fourier)
ok=all(abs(L[mul(k,x)]-chi(det(k))*L[x])<1e-9 for x in list(mats(2,4))[::37] for k in GLA)
print("L(kx)=chi(det k)L(x):",ok)
iota=((1,),(0,)); iotap=((1,0),(0,1),(0,0))
G=sum(chi(d)*psi(d) for d in range(1,q))
gamma=G*q**(1*1)    # #M_{N x T}=3
maxerr=0
for trial in range(3):
    p={v:complex(random.gauss(0,1),random.gauss(0,1)) for v in mats(2,1)}
    pbar={v:p[v].conjugate() for v in p}
    def F(f,r,cdim,rho):  # f on M_{r x c}, rho in M_{c x r}
        return sum(f[v]*psi(tr(mul(rho,v))) for v in f)
    # p^+ on 3x2 matrices [[eta,s],[0,1]], eta 2x1, s 2x1
    pplus={}
    for v in mats(3,2): pplus[v]=0
    for eta in mats(2,1):
        for s in mats(2,1):
            v=((eta[0][0],s[0][0]),(eta[1][0],s[1][0]),(0,1))
            neg=tuple(((-eta[i][0])%q,) for i in range(2))
            pplus[v]=p[neg]/q**2
    ppbar={v:pplus[v].conjugate() for v in pplus}
    lhs=0
    for y in mats(1,2):
        col=(y[0],)+iotap
        lhs+=F(pbar,2,1,y)*Lp[col]
    rhs=0
    for x in mats(2,3):
        m=tuple((iota[i][0],)+x[i] for i in range(2))
        rhs+=F(ppbar,3,2,x)*L[m]
    maxerr=max(maxerr,abs(gamma*lhs-c*rhs))
    print("ptv:fourier trial",trial,"|gamma*lhs|=",abs(gamma*lhs),"err=",abs(gamma*lhs-c*rhs))
print("|gamma|^2 =",abs(gamma)**2,"expected",q**(1*(1+2*1)))
# ---- pc:next with |N0|=1,|T|=1,|T'|=1: N1=N0+T (2), U2=N0+T' (2), A=N1+T' (3), N=N1+U2 (4), B=N0 in U2
# order of N: (N1_0,N1_1, U2_N0, U2_T'); B=(2), A=(0,1,3) ; A as rows order (N1_0,N1_1,T')
Lp,L,c,GLB,GLA=run(4,[2],[0,1,3],None)
err=0
for trial in range(3):
    p={v:complex(random.gauss(0,1),random.gauss(0,1)) for v in mats(2,1)}
    # p^+ on M_{A x U2}=3x2, rows (N1,T'), cols (N0,T')
    lhs=sum(p[y]*Lp[(y[0],y[1],(1,),(0,))] for y in mats(2,1))*len(GLB)
    rhs=0
    for eta in mats(2,1):
        for s in mats(2,1):
            x=((eta[0][0],s[0][0]),(eta[1][0],s[1][0]),(0,1))
            neg=tuple(((-eta[i][0])%q,) for i in range(2))
            # (iota' x): columns N1 (2), U2 (2): iota' = [[1,0],[0,1],[0,0]]
            m=((1,0)+x[0],(0,1)+x[1],(0,0)+x[2])
            rhs+=p[neg]/q**2*L[m]
    rhs*=c*len(GLA)
    err=max(err,abs(lhs-rhs)); print("pc:next trial",trial,abs(lhs),abs(lhs-rhs))

Overwriting N5/test_fourier.py


In [59]:
o = run("N5", "test_fourier.py")
assert "L(kx)=chi(det k)L(x): True" in o
errs = [float(x) for x in re.findall(r"ptv:fourier trial \d .*err= ([\d.e-]+)", o)]
nxt  = [float(l.split()[-1]) for l in o.splitlines() if l.startswith("pc:next trial")]
g2 = re.search(r"\|gamma\|\^2 = ([\d.]+) expected (\d+)", o)
print("ptv:fourier errors:", errs, " pc:next errors:", nxt)
assert len(errs) == 3 and max(errs) < 1e-9 and len(nxt) == 3 and max(nxt) < 1e-9
assert abs(float(g2.group(1)) - 27) < 1e-9 and g2.group(2) == "27"
record("N5", "pc:pointlaw, pc:next, ptv:fourier", "Fourier identities in a finite-field model", "F_3, |N|,|T|,|T'| = 1", "PASS", "q = 3, smallest block sizes")
record("N5", "au:factor, au:inj (constants with L-values)", "-", "-", "NOT TESTED numerically", "checked by hand only (Rallis exponent, c_Q)")

L(kx)=chi(det k)L(x): True
ptv:fourier trial 0 |gamma*lhs|= 140.69146066095257 err= 5.753312400962914e-14
ptv:fourier trial 1 |gamma*lhs|= 60.11266796462879 err= 4.263256414560601e-14
ptv:fourier trial 2 |gamma*lhs|= 94.59578756544731 err= 0.0
|gamma|^2 = 27.0 expected 27
pc:next trial 0 9.588864700517895 2.02730493912521e-14
pc:next trial 1 11.046512448699643 1.432144669219779e-14
pc:next trial 2 10.846893532625804 2.8435583831733384e-15
[N5/test_fourier.py: exit code 0, 45.3 s]
ptv:fourier errors: [5.753312400962914e-14, 4.263256414560601e-14, 0.0]  pc:next errors: [2.02730493912521e-14, 1.432144669219779e-14, 2.8435583831733384e-15]
RECORDED: N5 | pc:pointlaw, pc:next, ptv:fourier | PASS
RECORDED: N5 | au:factor, au:inj (constants with L-values) | NOT TESTED numerically


### Node N6 — the determinant of the comparison operator (`au:onelift`), and N6b (grid files)

**N6: no numerical test was run.** The statements of the node have no small instance (ranks $\ge x^5$, moduli $p_0p_1>b^{20}$) and are exact identities of characters and weights or estimates; only consistency checks by hand were made.

**N6b** tested auxiliary statements of the grid files of Theorem C.

**`t1_stat.py`: `pgl:gram`.** Exact rational check of the Gram relations for 60 random integer configurations ($m=2,4,6$), and numerical check that $w^*J_Zw=J_Z$ and $\iota^\dagger\mu_Z\iota=\mu_V$. *Refutation would be:* an assertion error (exact identities) or large residuals.

In [60]:
%%writefile N6b/t1_stat.py
# pgl:gram, pgl:stat, pgm:residue: exact rational check of Gram relations, numeric check of w in U(J_Z)
from fractions import Fraction as Fr
import random, math
import numpy as np
random.seed(1)
def prod(it):
    p=Fr(1)
    for a in it: p*=a
    return p
def trial(m,kx):
    # mu_0>...>mu_m integers, nu_1<...<nu_m integers, all nu > all mu
    mus=sorted(random.sample(range(-60,-20),m+1),reverse=True)
    nus=sorted(random.sample(range(20,60),m))
    p=lambda t:prod(t-mu for mu in mus)
    dp=lambda i:prod(mus[i]-mus[l] for l in range(m+1) if l!=i)
    q=lambda t:prod(t-nu for nu in nus)
    qj=lambda j,t:prod(t-nus[l] for l in range(m) if l!=j)
    dq=lambda j:qj(j,nus[j])
    # (i) signs
    for i in range(m+1):
        assert (dp(i)>0)==(i%2==0) and q(mus[i])>0
    B=lambda F1,F2: sum(F1(mus[i])*F2(mus[i])/(dp(i)*q(mus[i])) for i in range(m+1))
    assert B(q,q)==1
    for j in range(m):
        assert B(lambda t:qj(j,t),q)==0
        for l in range(m):
            val=B(lambda t:qj(j,t),lambda t:qj(l,t))
            exp=(Fr((-1)**j)*abs(dq(j))/p(nus[j])) if j==l else 0   # j 0-based: (-1)^{j-1} 1-based
            assert val==exp,(val,exp)
        assert p(nus[j])>0
    # numeric w
    n1=kx+m+1; n=kx+m
    JZ=np.diag([1.0]*kx+[(-1.0)**i for i in range(m+1)])
    J=np.diag([1.0]*kx+[(-1.0)**j for j in range(m)])   # V far j (1-based) sign (-1)^{j-1}
    br=lambda F: np.array([float(F(mus[i]))/math.sqrt(abs(float(dp(i)*q(mus[i])))) for i in range(m+1)])
    w=np.zeros((n1,n1))
    w[:kx,:kx]=np.eye(kx)
    for j in range(m):
        s=math.sqrt(float(p(nus[j])/abs(dq(j))))
        w[kx:,kx+j]=s*br(lambda t:qj(j,t))
    w[kx:,n]=br(q)
    e1=np.abs(w.T@JZ@w-JZ).max()
    iota=w[:,:n]; v=w[:,n]
    idag=J@iota.T@JZ
    muZ=np.diag([0.0]*kx+[float(x) for x in mus]); muV=np.diag([0.0]*kx+[float(x) for x in nus])
    e2=np.abs(idag@muZ@iota-muV).max(); e3=np.abs(idag@iota-np.eye(n)).max(); e4=np.abs(idag@v).max()
    Bmax=max(abs(x) for x in mus+nus)/1.0
    assert np.abs(w).max()<= (2*60)**m
    return max(e1,e3,e4), e2/ max(nus)
worst=(0,0)
for m in (2,4,6):
    for _ in range(20):
        a,b=trial(m,3); worst=(max(worst[0],a),max(worst[1],b))
print("pgl:gram exact identities OK; max |w*J_Z w - J_Z| etc:",worst)

Overwriting N6b/t1_stat.py


In [61]:
o = run("N6b", "t1_stat.py")
m = re.search(r"OK; max \|w\*J_Z w - J_Z\| etc: \(np\.float64\(([\d.e-]+)\), np\.float64\(([\d.e-]+)\)\)", o)
print("residuals (absolute; relative):", m.groups())
assert "pgl:gram exact identities OK" in o and float(m.group(2)) < 0.05
record("N6b", "pgl:gram (+ pgl:stat)", "exact Gram relations; w in U(J_Z)", "60 random configurations, m = 2,4,6", "PASS (exact part)", "floating-point residuals of the second check are ~1e-2 absolute on entries up to ~1e6 (scale of the integers used)")

pgl:gram exact identities OK; max |w*J_Z w - J_Z| etc: (np.float64(0.0078125), np.float64(0.006380012293077626))
[N6b/t1_stat.py: exit code 0, 0.3 s]
residuals (absolute; relative): ('0.0078125', '0.006380012293077626')
RECORDED: N6b | pgl:gram (+ pgl:stat) | PASS (exact part)


**`t2_minor.py`: `pbp:minorconj`, the row identity of `pgl:rows`, and `pgk:param`.** $\Delta_S(k^*\hat gk)=\exp(\mathrm{tr}(PZ)+\lvert[P,Z]\rvert^2/4+r)$ with $\lvert r\rvert\le10^3\delta(\lvert[P,Z]\rvert^2+\lvert[P,k]\rvert^2)$ on 400 random cases; the row identity; the composition law of `pgk:param`. *Refutation would be:* $\max\lvert r\rvert/\text{bound}>1$, or residuals above rounding.

In [62]:
%%writefile N6b/t2_minor.py
# pbp:minorconj (used by pgm:gauss): Delta_S(k^* ghat k) = exp(tr(PZ) + |[P,Z]|^2/4 + r), |r| <= 1e3*delta*(|[P,Z]|^2+|[P,k]|^2)
# and pgl:rows row identity, in U(J0); pgk:param algebra.
import numpy as np
class la:
    @staticmethod
    def expm(A):
        A=np.array(A,dtype=complex); n=np.linalg.norm(A); s=max(0,int(np.ceil(np.log2(max(n,1e-16))))+6)
        B=A/2**s; E=np.eye(len(A),dtype=complex); T=np.eye(len(A),dtype=complex)
        for j in range(1,25):
            T=T@B/j; E=E+T
        for _ in range(s): E=E@E
        return E
rng=np.random.default_rng(0)
def rand_u(J,scale):
    N=len(J); A=rng.normal(size=(N,N))+1j*rng.normal(size=(N,N))
    Jm=np.diag(J)
    Z=(A-Jm@A.conj().T@Jm)/2          # J Z^* J = -Z
    return Z*scale/np.linalg.norm(Z)
def rand_K(J,scale):
    N=len(J); A=rng.normal(size=(N,N))+1j*rng.normal(size=(N,N))
    A=(A-A.conj().T)/2
    Jv=np.array(J); mask=(Jv[:,None]==Jv[None,:])
    A=A*mask
    return la.expm(A*scale)
worst=0
for trial in range(400):
    N=rng.integers(3,7); J=list(rng.choice([1,-1],size=N))
    delta=10**rng.uniform(-4,np.log10(0.05))
    Z=rand_u(J,delta)
    # make Z mostly diagonal sometimes (small crossing part)
    if trial%2: 
        D=np.diag(np.diag(Z)); Z=D+ (Z-D)*10**rng.uniform(-3,0); 
    o=rng.permutation(N).astype(float)
    order=np.argsort(o); sz=rng.integers(1,N); S=order[:sz]      # o_i<o_j for i in S, j not in S
    P=np.zeros((N,N)); P[S,S]=1
    Jv=np.array(J); same=(Jv[:,None]==Jv[None,:])
    um=np.where(same&(o[:,None]<o[None,:]),Z,0); up=np.where(same&(o[:,None]>o[None,:]),Z,0)
    ghat=la.expm(-um)@la.expm(Z)@la.expm(-up)
    k=rand_K(J,10**rng.uniform(-3,0.5))
    val=np.linalg.det((k.conj().T@ghat@k)[np.ix_(S,S)])
    x2=np.linalg.norm(P@Z-Z@P)**2; k2=np.linalg.norm(P@k-k@P)**2
    pred=np.trace(P@Z)+x2/4
    r=np.log(val)-pred
    r=complex(r.real,(r.imag+np.pi)%(2*np.pi)-np.pi)
    d=np.linalg.norm(Z)
    bound=1e3*d*(x2+k2)
    worst=max(worst,abs(r)/bound if bound>0 else 0)
    assert abs(np.linalg.det(ghat)-np.exp(np.trace(Z)))<1e-9
print("minorconj: max |r|/bound =",worst)
# row identity (pgl:rows): 2Re x_ii = 2 sum_j |y_ij|^2 - sum_j |x_ij|^2
w2=0
for _ in range(100):
    N=5; J=[1,1,-1,1,-1]; g=la.expm(rand_u(J,rng.uniform(0.1,3))); Jm=np.diag(J)
    x=g-np.eye(N); y=(g-Jm@g@Jm)/2
    lhs=2*np.real(np.diag(x)); rhs=2*(abs(y)**2).sum(1)-(abs(x)**2).sum(1)
    w2=max(w2,abs(lhs-rhs).max()/ (1+abs(g).max()**2))
print("row identity residual",w2)
# pgk:param: E(w)=exp(2P_w) in U(S), composition law
w3=0
for _ in range(100):
    n=6;i=2
    S=np.diag([1,1,1,-1,-1,1.]).astype(complex)
    # isotropic a: columns e1+e4, e2+e5
    a=np.zeros((n,i),complex); a[0,0]=1;a[3,0]=1;a[1,1]=1;a[4,1]=1
    def randw():
        w=rng.normal(size=(n,i))+1j*rng.normal(size=(n,i))
        # project to a^* S w = 0 : choose b with a^*S b = 1
        b=np.zeros((n,i),complex); b[0,0]=.5;b[3,0]=-.5;b[1,1]=.5;b[4,1]=-.5
        return w-b@(a.conj().T@S@w)
    Pw=lambda w:(w@a.conj().T-a@w.conj().T)@S
    E=lambda w:np.eye(n)+2*Pw(w)+2*Pw(w)@Pw(w)
    w=randw(); wp=randw()
    assert abs(a.conj().T@S@w).max()<1e-12
    e=E(w)
    w3=max(w3,abs(e.conj().T@S@e-S).max(),abs(e-la.expm(2*Pw(w))).max(),abs(e@a-a).max(),
           abs(E(w)@E(wp)-E(w+wp-a@(w.conj().T@S@wp))).max(), abs(Pw(w)@Pw(w)+a@(w.conj().T@S@w)@a.conj().T@S).max())
print("pgk:param residual",w3)

Overwriting N6b/t2_minor.py


In [63]:
o = run("N6b", "t2_minor.py")
r = float(re.search(r"max \|r\|/bound = ([\d.e-]+)", o).group(1))
row = float(re.search(r"row identity residual ([\d.e-]+)", o).group(1))
par = float(re.search(r"pgk:param residual ([\d.e-]+)", o).group(1))
assert r < 1 and row < 1e-9 and par < 1e-8
record("N6b", "pbp:minorconj, pgl:rows, pgk:param", "minor expansion with error bound; row identity; composition law", "400 + 100 + 100 random cases", f"PASS (max |r|/bound = {r:.3f})", "small matrices")

minorconj: max |r|/bound = 0.025553712784581673
row identity residual 6.505708282565663e-14
pgk:param residual 1.362326223246534e-11
[N6b/t2_minor.py: exit code 0, 0.2 s]
RECORDED: N6b | pbp:minorconj, pgl:rows, pgk:param | PASS (max |r|/bound = 0.026)


**`t3_match.py`: `pgy:match`, `pgy:gap`, `pgy:T1`, `pgy:series`.** The block rule (ascending and descending pairs), the gap bound, the sign count and rank conditions of the squares, exactly, for $(m,k)\in\{(2,2),(2,4),(4,2),(4,4)\}$; and the local inequalities of `pgy:series`. *Refutation would be:* an assertion error.

In [64]:
%%writefile N6b/t3_match.py
# pgy:match, pgy:gap, pgy:T1 (sign count), pgy:chars (weights) for the triangle of the second grid, small parameters.
from fractions import Fraction as Fr
def run(m,k):
    r=k+m-1; t=5*r; Delta=2*r**8; kap=Fr(7*Delta+3)   # any kappa(tau)
    kp=lambda l: kap+(Fr(2*l+1,2))*Delta
    a=lambda rho: max(rho+1-k,0)
    E=lambda i,n,aa: i+aa+min(max(aa-i,0),n)
    def node(i,n):
        # coordinates: ('s',rho) rho=0..r ; ('b',l,beta) l=i+1..i+n, beta=0..t-2
        lam={};sg={}
        for rho in range(r+1):
            sv = Fr(k,2)-1-rho if rho<=k-2 else -kap-(rho+2-k)*Delta
            lam[('s',rho)]=sv+Fr(1,2); sg[('s',rho)]=(-1)**E(i,n,a(rho))
        for l in range(i+1,i+n+1):
            for be in range(t-1):
                lam[('b',l,be)]=-kp(l)+Fr(t-2,2)-be; sg[('b',l,be)]=1
        return lam,sg
    def rule(lam,sg,lamp,sgp,tau0,tau0p,newblock,d):
        cnt=0
        for c in lam:
            assert lamp[c]==-lam[c]+tau0+tau0p
            dist=lamp[c]-tau0p
            assert abs(dist)>Fr(d,2)
            assert sgp[c]==sg[c]*(1 if dist>0 else -1),(c,)
            cnt+=1
        B=[c for c in lamp if c not in lam]
        assert len(B)==d and all(c[0]=='b' and c[1]==newblock for c in B)
        assert sorted(lamp[c] for c in B)==sorted(tau0p+Fr(d-1,2)-be for be in range(d))
        assert len(set(sgp[c] for c in B))==1
    mingap=None
    for i in range(m+1):
        for n in range(m+1-i):
            lam,sg=node(i,n); N=len(lam); assert N==r+1+n*(t-1) and N%2==0
            vals=list(lam.values()); assert len(set(vals))==N
            for c1 in lam:
                for c2 in lam:
                    if sg[c1]!=sg[c2]:
                        g=abs(lam[c1]-lam[c2]); mingap=g if mingap is None else min(mingap,g)
            if i+n<m:
                # ascending (G_{i,n},G_{i,n+1}): rows -lam^{(i,n)}, cols lam^{(i,n+1)}
                lam2,sg2=node(i,n+1)
                N2=len(lam2)
                wa=-kp(i+n+1)+Fr(N2,2); wb=-kp(i+n+1)+Fr(N,2)
                tau0=Fr(N2,2)-wa; tau0p=wb-Fr(N,2)
                rule({c:-v for c,v in lam.items()},sg,lam2,sg2,tau0,tau0p,i+n+1,t-1)
                # descending (G_{i+1,n},G_{i,n+1}): rows lam^{(i+1,n)}, cols -lam^{(i,n+1)}, c_i=0
                lam1,sg1=node(i+1,n)
                wa=kp(i+1)+Fr(N2,2); wb=kp(i+1)+Fr(len(lam1),2)
                tau0=Fr(N2,2)-wa; tau0p=wb-Fr(len(lam1),2)
                rule(lam1,sg1,{c:-v for c,v in lam2.items()},sg2,tau0,tau0p,i+1,t-1)
            if i+n+2<=m:
                neg=lambda nd: sum(1 for v in nd[1].values() if v<0)
                q=neg(node(i,n+1))+neg(node(i+1,n+1)); qA=neg(node(i,n+2)); qAp=neg(node(i+1,n))
                assert q==qA+qAp
                Ntot=2*(r+1+(n+1)*(t-1)); assert 2*q+2<=Ntot<=2*(n+1)*Delta+2*q
    assert mingap>=Fr(Delta-t,2)-1
    return mingap,Fr(Delta-t,2)-1
for (m,k) in [(2,2),(2,4),(4,2),(4,4)]:
    print((m,k),"block rule (asc, desc), gap, sign count of squares: OK; min gap / bound =",float(run(m,k)[0]/run(m,k)[1]))
# pgy:series numerics: 250 f (4/5)^n < 1/3 with n>x^5, f<=x^12, x>=128^100 (logs)
import math
lx=100*math.log(128)
# log(250 f (4/5)^n) <= log 250 + 12 lx - x^5 log(5/4), and x^5 = exp(5 lx) > 10^1000
assert 5*lx/math.log(10) > 1000 and math.log(250)+12*lx < 1e4
print('250 f (4/5)^n < 1/3: exponent <= 1e4 - 10^1000*log(5/4) < 0')
for q in (5,7,11,13):
    for n in (3,4,5,10):
        s=10*q**(-2*n+2); ns=2*4**n*q**(-n+1)
        assert s<=2*0.8**n*q**-2+1e-18 and ns<=250*0.8**n*q**-2+1e-18
print("pgy:series local inequalities OK")

Overwriting N6b/t3_match.py


In [65]:
o = run("N6b", "t3_match.py")
assert o.count(": OK; min gap / bound") == 4 and "pgy:series local inequalities OK" in o
record("N6b", "pgy:match, pgy:gap, pgy:T1, pgy:series", "block rule, gap, sign count of squares; local inequalities", "(m,k) in {2,4}^2", "PASS", "small m,k")
record("N6", "au:onelift", "-", "-", "NOT TESTED (no numerical test)", "no small instance; hand consistency checks only")

(2, 2) block rule (asc, desc), gap, sign count of squares: OK; min gap / bound = 2.0025944296070204
(2, 4) block rule (asc, desc), gap, sign count of squares: OK; min gap / bound = 2.0000691223888696
(4, 2) block rule (asc, desc), gap, sign count of squares: OK; min gap / bound = 2.0000691223888696
(4, 4) block rule (asc, desc), gap, sign count of squares: OK; min gap / bound = 2.0000064182820427
250 f (4/5)^n < 1/3: exponent <= 1e4 - 10^1000*log(5/4) < 0
pgy:series local inequalities OK
[N6b/t3_match.py: exit code 0, 0.1 s]
RECORDED: N6b | pgy:match, pgy:gap, pgy:T1, pgy:series | PASS
RECORDED: N6 | au:onelift | NOT TESTED (no numerical test)


### Node N7 — congruences, Selmer groups, local loss (`pg:I`–`pg:III`, `pg:loss`, `pg:odd`), with N7b and N7c

**N7 and N7b: hand tests only** (no scripts). They include two rank-one computations showing that the hypothesis NM of `pki:generic` cannot be dropped (for $Y=\mathbb Z_\ell(i)$, $L=\mathbb Z_\ell$ the cup product is non-zero exactly where NM excludes it). N7b recommended an independent **rank-two** test, which N7c carried out by computer for one family.

**N7c, rank-two test of `pki:generic`(i).** $\ell=41$, $V$ crystalline irreducible with Hodge–Tate weights $(0,k-1)$, $k-1\in\{60,\dots,63\}$, trace of Frobenius $a=\ell X$, $X$ a unit; $Y$ a lattice in $V$, $L=\mathbb Z_\ell$. All hypotheses of `pki:generic` hold, and it predicts: for every map $t:Y\to\mu_\ell$, $t_*H^1_f(Y)\subseteq$ units. Method: Wach modules modulo $\ell$, all crystalline extensions computed by linear algebra over $\mathbb F_\ell$, their classes in $H^1(\mathbb F_\ell(1))$ read off in a normal form; the unit line has normal form $1/\pi+1/2$, i.e. ratio slot1/slot0 $=2$.
*Refutation would be:* classes spanning more than the unit line (normal-form rank 2, or a ratio other than 2, or slots other than $\{0,1\}$). A control (rank-one module $\varphi=q^\ell$) checks that the procedure does detect *très ramifié* classes when they exist.

Helper modules: `wach.py` (power series, Frobenius, Wach module construction), `ext.py` (extension computation).

In [66]:
%%writefile N7c/wach.py
# Wach module of a lattice in V_{k,a}, v(a)=1, p+2<=k-1<2p-2 ; mod p computations.
# Conventions (covariant Wach module of T=Y^vee, HT weights (0,-(k-1))):
#   phi(e1)=q^{k-1} e2, phi(e2)=-e1+X z e2 ; P=[[0,-1],[q^{k-1},X z]] (columns=images)
import numpy as np, sys
from fractions import Fraction
from math import comb

def zpoly(p,k,nmax=7):
    """z = p*(lambda_-/lambda_+)^{k-1} mod pi^{k-1}, exact rational coefficients (truncated product)"""
    n=k-1
    def mul(a,b):
        c=[Fraction(0)]*n
        for i,x in enumerate(a):
            if x==0: continue
            for j,y in enumerate(b):
                if i+j>=n: break
                c[i+j]+=x*y
        return c
    def inv(a):
        b=[Fraction(0)]*n; b[0]=1/a[0]
        for i in range(1,n):
            s=sum(a[j]*b[i-j] for j in range(1,i+1))
            b[i]=-s*b[0]
        return b
    def phin_q_over_p(m):
        # ((1+pi)^{p^{m+1}}-1)/(p((1+pi)^{p^m}-1))
        A=[Fraction(comb(p**(m+1),j+1)) for j in range(n)]   # divided by pi
        B=[Fraction(comb(p**m,j+1))*p for j in range(n)]
        return mul(A,inv(B))
    num=[Fraction(0)]*n; num[0]=Fraction(1); den=list(num)
    for m in range(nmax+1):
        f=phin_q_over_p(m)
        if m%2==0: num=mul(num,f)
        else: den=mul(den,f)
    r=mul(num,inv(den))
    # power k-1
    res=[Fraction(0)]*n; res[0]=Fraction(1)
    e=k-1; base=r
    while e:
        if e&1: res=mul(res,base)
        base=mul(base,base); e>>=1
    return [p*x for x in res]

class R:
    """truncated power series over Z/m, precision N"""
    def __init__(s,m,N): s.m=m; s.N=N
    def z(s): return np.zeros(s.N,dtype=np.int64)
    def one(s): a=s.z(); a[0]=1; return a
    def mono(s,i,c=1):
        a=s.z()
        if i<s.N: a[i]=c%s.m
        return a
    def mul(s,a,b):
        na=np.nonzero(a)[0]; nb=np.nonzero(b)[0]
        if len(na)==0 or len(nb)==0: return s.z()
        if len(na)<=40 or len(nb)<=40:
            if len(na)>len(nb): a,b,na,nb=b,a,nb,na
            c=s.z()
            for i in na:
                L=s.N-i
                c[i:]=(c[i:]+a[i]*b[:L])%s.m
            return c
        # chunk to avoid overflow
        c=np.convolve(a.astype(object),b.astype(object))[:s.N] if s.m*s.m*s.N>2**62 else np.convolve(a,b)[:s.N]
        return (c%s.m).astype(np.int64)
    def inv(s,a):
        # a[0] unit
        a0=pow(int(a[0]),-1,s.m)
        b=s.z(); b[0]=a0
        n=1
        while n<s.N:
            n=min(2*n,s.N)
            t=s.mul(a,b); t=(-t)%s.m; t[0]=(t[0]+2)%s.m
            b=s.mul(b,t); b[n:]=0
        return b
    def pw(s,a,e):
        r=s.one(); b=a.copy()
        while e:
            if e&1: r=s.mul(r,b)
            b=s.mul(b,b); e>>=1
        return r
    def shift(s,a,i):
        c=s.z()
        if i>=0:
            if i<s.N: c[i:]=a[:s.N-i]
        else:
            c[:s.N+i]=a[-i:]
        return c
    def comp(s,f,g):
        """f(g), g(0)=0. Horner."""
        d=np.nonzero(f)[0]
        if len(d)==0: return s.z()
        r=s.z(); 
        for i in range(d[-1],-1,-1):
            r=s.mul(r,g); r[0]=(r[0]+f[i])%s.m
        return r

def setup(p,k,g0,N,c=1,Xunit=None,verbose=True):
    m=p**c; S=R(m,N); M=(p-1)*(k-1)
    zf=zpoly(p,k)
    zint=[]
    for x in zf:
        assert x.denominator%p!=0, "z not integral"
        zint.append(int(x.numerator*pow(x.denominator,-1,p**(c+3)))%(p**(c+3)))
    z=S.z(); z[:k-1]=[t%m for t in zint]
    # phi(pi), gamma(pi)
    phipi=S.z(); 
    for j in range(1,min(p,N-1)+1): phipi[j]=comb(p,j)%m
    gpi=S.z()
    for j in range(1,min(g0,N-1)+1): gpi[j]=comb(g0,j)%m
    if c==1:
        phi=lambda f: _frob(S,f,p)
    else:
        phi=lambda f: S.comp(f,phipi)
    gam=lambda f: S.comp(f,gpi)
    q=S.shift(phipi,-1)
    h=S.shift(gpi,-1)
    u=S.mul(h,S.inv(phi(h)))            # q/gamma(q)
    # g_- = prod_{n even} phi^n(u)^{k-1}, g_+ = phi(g_-)
    uk=S.pw(u,k-1)
    gm=S.one(); t=uk
    for it in range(60):
        gm_new=S.mul(gm,t)
        t=phi(phi(t))
        if np.array_equal(gm_new,gm) and it>2: break
        gm=gm_new
    gp=phi(gm)
    return dict(p=p,k=k,g0=g0,N=N,c=c,m=m,S=S,M=M,z=z,phi=phi,gam=gam,q=q,h=h,u=u,uk=uk,gm=gm,gp=gp,gpi=gpi,phipi=phipi)

def _frob(S,f,p):
    r=S.z(); n=(S.N+p-1)//p
    r[::p][:n]=f[:len(r[::p])]
    return r

def build_G(d,X):
    S=d['S']; p=d['p']; k=d['k']; m=d['m']; phi=d['phi']; gam=d['gam']
    z=d['z']; gm=d['gm']; gp=d['gp']
    qk=S.pw(d['q'],k-1)
    gz=gam(z)
    diff=(S.mul(z,gp)-S.mul(gm,gz))%m
    assert not diff[:k-1].any(), ("z-compat fails",diff[:k-1])
    cc=S.shift(diff,-(k-1))     # precision N-(k-1)
    Xz=(X*z)%m
    P=[[S.z(),(-S.one())%m],[qk,Xz]]
    gP=[[S.z(),(-S.one())%m],[gam(qk),gam(Xz)]]
    adj=[[gP[1][1],(-gP[0][1])%m],[(-gP[1][0])%m,gP[0][0]]]
    def mm(A,B):
        return [[(S.mul(A[i][0],B[0][j])+S.mul(A[i][1],B[1][j]))%m for j in range(2)] for i in range(2)]
    def Lop(H):
        pH=[[phi(H[i][j]) for j in range(2)] for i in range(2)]
        T=mm(mm(P,pH),adj)
        return [[S.mul(d['uk'],T[i][j]) for j in range(2)] for i in range(2)]
    H0=[[S.z(),S.z()],[(-X*cc)%m,S.z()]]
    H=[[x.copy() for x in r] for r in H0]; term=H0
    for it in range(400):
        term=Lop(term)
        if not any(term[i][j].any() for i in range(2) for j in range(2)): break
        H=[[(H[i][j]+term[i][j])%m for j in range(2)] for i in range(2)]
    else:
        print("WARNING: L-series did not terminate")
    d['Liter']=it
    G=[[S.shift(H[i][j],k-1) for j in range(2)] for i in range(2)]
    G[0][0]=(G[0][0]+gp)%m; G[1][1]=(G[1][1]+gm)%m
    # check
    pG=[[phi(G[i][j]) for j in range(2)] for i in range(2)]
    lhs=mm(P,pG); rhs=mm(G,gP)
    bad=[int(np.nonzero((lhs[i][j]-rhs[i][j])%m)[0][0]) if ((lhs[i][j]-rhs[i][j])%m).any() else -1 for i in range(2) for j in range(2)]
    d.update(P=P,gP=gP,G=G,X=X,Xz=Xz,qk=qk,check=bad)
    return d

Overwriting N7c/wach.py


In [67]:
%%writefile N7c/ext.py
import sys; sys.path.insert(0,'.')
from wach import *
from wach import _frob
import time

def kernel_modp(A,p):
    """right kernel of A (rows x cols) over F_p; returns list of basis vectors"""
    A=A.copy()%p; rows,cols=A.shape
    piv=[]; r=0
    for c in range(cols):
        if r>=rows: break
        nz=np.nonzero(A[r:,c])[0]
        if len(nz)==0: continue
        i=r+nz[0]
        if i!=r: A[[r,i]]=A[[i,r]]
        A[r]=(A[r]*pow(int(A[r,c]),-1,p))%p
        col=A[:,c].copy(); col[r]=0
        idx=np.nonzero(col)[0]
        if len(idx): A[idx]=(A[idx]-np.outer(col[idx],A[r]))%p
        piv.append(c); r+=1
    free=[c for c in range(cols) if c not in set(piv)]
    ker=[]
    for f in free:
        v=np.zeros(cols,dtype=np.int64); v[f]=1
        for i,c in enumerate(piv): v[c]=(-A[i,f])%p
        ker.append(v)
    return ker

def rank_modp(vecs,p):
    if len(vecs)==0: return 0
    A=np.array(vecs,dtype=np.int64)%p
    r=0; rows,cols=A.shape
    for c in range(cols):
        if r>=rows: break
        nz=np.nonzero(A[r:,c])[0]
        if len(nz)==0: continue
        i=r+nz[0]; A[[r,i]]=A[[i,r]]
        A[r]=(A[r]*pow(int(A[r,c]),-1,p))%p
        col=A[:,c].copy(); col[r]=0
        idx=np.nonzero(col)[0]
        if len(idx): A[idx]=(A[idx]-np.outer(col[idx],A[r]))%p
        r+=1
    return r

def NF(a_low,p,top):
    """a = pi^{-top} * (series a_low[0..top]) ; normal form in D/(phi-1): slots n' (p!|n') and 0"""
    nf={}
    for idx in range(top+1):
        cval=int(a_low[idx])%p
        if cval==0: continue
        n=top-idx
        while n>0 and n%p==0: n//=p
        nf[n]=(nf.get(n,0)+cval)%p
    return {n:v for n,v in nf.items() if v}

def nfvec(nf,p,top):
    v=np.zeros(top+1,dtype=np.int64)
    for n,c in nf.items(): v[n]=c
    return v

def rank_one(p,s,g0,verbose=True):
    """mod p Wach extensions of (phi=pi^{s(p-1)}, trivial unramified part) by trivial; returns NF span"""
    D=s*p+1; J=s+5; Mu=s*(p-1); N=Mu+J+1+ (0)
    N=max(N,D)
    S=R(p,N)
    gpi=S.z()
    for j in range(1,min(g0,N-1)+1): gpi[j]=comb(g0,j)%p
    h=S.shift(gpi,-1); hn=(h*pow(g0,-1,p))%p
    g=S.pw(hn,s); phig=_frob(S,g,p)
    gpow=[S.one()]
    for i in range(1,max(D,Mu+1)+1): gpow.append(S.mul(gpow[-1],gpi))
    cols=[]
    for i in range(D):
        cols.append((S.shift(phig,i)-gpow[i])%p)      # x phi(g) - gamma(x)
    for j in range(1,J+1):
        # -( z gamma(mu) - phi(z) ) = phi(z) - gamma(mu) z   moved: x phi(g)+phi(z) - z gamma(mu) - gamma(x)=0
        col=(S.mono(p*j)-S.shift(gpow[Mu],j))%p
        cols.append(col)
    A=np.array(cols).T
    ker=kernel_modp(A,p)
    cob=[]
    for j in range(0,s+1):
        y=(S.mono(p*j)-S.mono(Mu+j))%p; cob.append(y[:D].copy())
    xs=[v[:D] for v in ker]
    nfs=[NF(x,p,p*s) for x in xs]
    nfc=[NF(x,p,p*s) for x in cob]
    return dict(kerdim=len(ker),xrank=rank_modp(xs,p),cobrank=rank_modp(cob,p),
                nfs=nfs,nfc=nfc,nfrank=rank_modp([nfvec(n,p,p*s) for n in nfs],p))

def rank_two(p,k,g0,verbose=True,X=None,extraJ=5):
    r=k-1-p; M=(p-1)*(k-1); D=M+r+1; J=r+extraJ; N=M+J+1
    t0=time.time()
    d=setup(p,k,g0,N,c=1)
    S=d['S']; z=d['z']; phi=d['phi']; gam=d['gam']
    zc=int(z[p-1]); assert zc%p!=0 and not z[:p-1].any()
    if X is None: X=pow(zc,-1,p)      # w_0 = 1  => unramified part trivial
    build_G(d,X)
    assert d['check']==[-1]*4, d['check']
    G=d['G']; w=d['Xz']
    # beta: phi(beta)(beta+w) = -pi^M
    beta=(-w)%p
    for it in range(50):
        fb=S.shift(phi(beta),-p*(p-1))          # unit (w_0 != 0)
        t=S.shift(S.inv(fb),M-p*(p-1))
        nb=(-w-t)%p
        if np.array_equal(nb,beta): break
        beta=nb
    chk=(S.mul(phi(beta),(beta+w)%p)+S.mono(M))%p
    okb = not chk[:N-p*(p-1)].any()
    mu=(-phi(beta))%p
    u=S.shift(mu,-p*(p-1)); u[N-p*(p-1):]=0
    # rho/phi(rho)=u  => phi(rho)=prod_{n>=1} phi^n(u)   (need only mod pi^{p^2+1})
    top=p*p
    T=R(p,top+1)
    ut=u[:top+1].copy()
    phirho=T.one(); t=ut
    for n in range(1,4):
        t=_frob(T,t,p); phirho=T.mul(phirho,t)
    # matrix
    gpi=d['gpi']
    gpow=[S.one()]
    for i in range(1,max(D,M)+1): gpow.append(S.mul(gpow[-1],gpi))
    pG11=phi(G[0][0]); ppG12=phi(phi(G[0][1])); pgw=phi(gam(w))
    cols=[]
    for i in range(D):
        col=(S.shift(pG11,i)-S.shift(ppG12,p*i)-gpow[i])%p
        cols.append(col)
    for j in range(1,J+1):
        col=(S.mono(p*p*j)-S.shift(pgw,p*j)+S.shift(gpow[M],j))%p
        cols.append((-col)%p)
    A=np.array(cols).T
    t1=time.time()
    ker=kernel_modp(A,p)
    t2=time.time()
    pw_=_frob(S,w,p)
    cob=[]
    for j in range(0,r+1):
        y=(S.mono(p*p*j)-S.shift(pw_,p*j)+S.mono(M+j))%p
        cob.append(((-y)%p)[:D].copy())
    xs=[v[:D] for v in ker]
    def nf_of(x):
        xx=np.zeros(top+1,dtype=np.int64); L=min(top+1,len(x)); xx[:L]=x[:L]
        return NF(T.mul(phirho,xx),p,top)
    nfs=[nf_of(x) for x in xs]; nfc=[nf_of(x) for x in cob]
    # are coboundaries in kernel-projection?
    cob_in=rank_modp(xs+cob,p)==rank_modp(xs,p)
    res=dict(p=p,k=k,r=r,X=X,u0=int(u[0]),beta_ok=okb,kerdim=len(ker),xrank=rank_modp(xs,p),cobrank=rank_modp(cob,p),
             cob_in_ker=cob_in,nfs=nfs,nfc=nfc,nfrank=rank_modp([nfvec(n,p,top) for n in nfs],p),times=(t1-t0,t2-t1))
    res['nfbasis']=[n for n in nfs if n]
    return res

if __name__=="__main__":
    p=int(sys.argv[1]); 
    g0=next(g for g in range(2,p) if all(pow(g,(p-1)//f,p)!=1 for f in range(2,p) if (p-1)%f==0) and pow(g,p-1,p*p)!=1)
    print("p",p,"g0",g0)
    for s in (1,2,p):
        R1=rank_one(p,s,g0)
        print("rank one s=%d: kerdim %d xrank %d cobrank %d; coboundary NFs nonzero: %s; NF rank %d"%(s,R1['kerdim'],R1['xrank'],R1['cobrank'],[n for n in R1['nfc'] if n],R1['nfrank']))
        seen=[]
        for n in R1['nfs']:
            if n and n not in seen: seen.append(n)
        print("   NFs:",seen[:6])
    for k in [int(a) for a in sys.argv[2:]]:
        R2=rank_two(p,k,g0)
        print({kk:vv for kk,vv in R2.items() if kk not in('nfs','nfc','nfbasis')})
        print("   coboundary NFs nonzero:",[n for n in R2['nfc'] if n])
        seen=[]
        for n in R2['nfs']:
            if n and n not in seen: seen.append(n)
        print("   NFs of kernel classes:",seen[:8])

Overwriting N7c/ext.py


**Construction checks (`t1.py`–`t4.py`).** Internal diagnostics of the Wach-module construction for $p=7$, $k=11$: `t1` checks that $P\varphi(G)-G\gamma(P)$ vanishes to the working precision (printed "first bad orders" $=-1$) and $G\equiv1\bmod\pi$; `t3` checks that the mod-$p$ and mod-$p^4$ constructions agree; `t2` and `t4` print the orders of the correction terms (diagnostic output, no pass/fail criterion).

In [68]:
%%writefile N7c/t1.py
import sys; sys.path.insert(0,'.')
from wach import *
p,k,g0=7,11,3   # k-1=10=p+3
for c in (1,3):
    N=140
    d=setup(p,k,g0,N,c=c)
    print("z mod p^c:",d['z'][:k-1])
    for X in (1,2,3):
        build_G(d,X)
        print("c",c,"X",X,"Liter",d['Liter'],"first bad orders (PphiG-GgammaP):",d['check'], "G=1 mod pi:",[int(d['G'][i][j][0]) for i in range(2) for j in range(2)])
        if c==3: G3=[[x%p for x in r] for r in d['G']]; 

Overwriting N7c/t1.py


In [69]:
%%writefile N7c/t2.py
import sys; sys.path.insert(0,'.')
from wach import *
p,k,g0=7,11,3
for c in (1,4):
    d=setup(p,k,g0,200,c=c); S=d['S']; m=d['m']
    z=d['z']; diff=(S.mul(z,d['gp'])-S.mul(d['gm'],d['gam'](z)))%m
    print(c,"diff nonzero idx",np.nonzero(diff)[0][:10], diff[np.nonzero(diff)[0][:6]])
    build_G(d,1)
    G=d['G']
    for i in range(2):
        for j in range(2):
            nz=np.nonzero((G[i][j]-(d['gp'] if (i,j)==(0,0) else d['gm'] if (i,j)==(1,1) else 0))%m)[0]
            print("  G-G0",i,j,nz[:8], (G[i][j]%p)[nz[:5]] if len(nz) else '')

Overwriting N7c/t2.py


In [70]:
%%writefile N7c/t3.py
import sys; sys.path.insert(0,'.')
from wach import *
p,k,g0=7,11,3
ds={}
for c in (1,4):
    d=setup(p,k,g0,120,c=c); ds[c]=d
a,b=ds[1],ds[4]
for key in ('z','q','h','u','uk','gm','gp'):
    print(key, np.array_equal(a[key], b[key]%p), np.nonzero((a[key]-b[key])%p)[0][:5])
S=b['S']
f=b['h']
print("phi mod p agree:", np.array_equal(a['phi'](a['h']), b['phi'](b['h'])%p))
print("gam agree:", np.array_equal(a['gam'](a['z']), b['gam'](b['z'])%p))

Overwriting N7c/t3.py


In [71]:
%%writefile N7c/t4.py
import sys; sys.path.insert(0,'.')
from wach import *
p,k,g0=7,11,3
def terms(c,X=1,N=120):
    d=setup(p,k,g0,N,c=c); S=d['S']; m=d['m']; phi=d['phi']; gam=d['gam']; z=d['z']
    qk=S.pw(d['q'],k-1)
    diff=(S.mul(z,d['gp'])-S.mul(d['gm'],gam(z)))%m
    cc=S.shift(diff,-(k-1)); Xz=(X*z)%m
    P=[[S.z(),(-S.one())%m],[qk,Xz]]
    gP=[[S.z(),(-S.one())%m],[gam(qk),gam(Xz)]]
    adj=[[gP[1][1],(-gP[0][1])%m],[(-gP[1][0])%m,gP[0][0]]]
    def mm(A,B): return [[(S.mul(A[i][0],B[0][j])+S.mul(A[i][1],B[1][j]))%m for j in range(2)] for i in range(2)]
    def Lop(H):
        pH=[[phi(H[i][j]) for j in range(2)] for i in range(2)]
        T=mm(mm(P,pH),adj)
        return [[S.mul(d['uk'],T[i][j]) for j in range(2)] for i in range(2)]
    H0=[[S.z(),S.z()],[(-X*cc)%m,S.z()]]
    out=[H0]; t=H0
    for it in range(8):
        t=Lop(t); out.append(t)
    return out
A=terms(1); B=terms(4)
for j in range(8):
    print(j,[ (np.array_equal(A[j][a][b][:90],B[j][a][b][:90]%p), int(np.nonzero(B[j][a][b]%p)[0][0]) if (B[j][a][b]%p).any() else -1, int(np.nonzero(B[j][a][b])[0][0]) if (B[j][a][b]).any() else -1) for a in range(2) for b in range(2)])

Overwriting N7c/t4.py


In [72]:
o1 = run("N7c", "t1.py"); o2 = run("N7c", "t2.py"); o3 = run("N7c", "t3.py"); o4 = run("N7c", "t4.py")
assert o1.count("first bad orders (PphiG-GgammaP): [-1, -1, -1, -1] G=1 mod pi: [1, 0, 0, 1]") == 6
assert "False" not in o3 and o3.count("True") == 9
assert "False" not in o4
record("N7c", "Wach-module construction", "P phi(G) = G gamma(P) to precision; mod p vs mod p^c agreement", "p=7, k=11", "PASS (t1, t3, t4); t2 diagnostic", "construction sanity only")

z mod p^c: [0 0 0 0 0 0 3 4 1 1]
c 1 X 1 Liter 2 first bad orders (PphiG-GgammaP): [-1, -1, -1, -1] G=1 mod pi: [1, 0, 0, 1]
c 1 X 2 Liter 2 first bad orders (PphiG-GgammaP): [-1, -1, -1, -1] G=1 mod pi: [1, 0, 0, 1]
c 1 X 3 Liter 2 first bad orders (PphiG-GgammaP): [-1, -1, -1, -1] G=1 mod pi: [1, 0, 0, 1]
z mod p^c: [  7 112 294 189 301  49 248 102 253 162]
c 3 X 1 Liter 3 first bad orders (PphiG-GgammaP): [-1, -1, -1, -1] G=1 mod pi: [1, 0, 0, 1]
c 3 X 2 Liter 3 first bad orders (PphiG-GgammaP): [-1, -1, -1, -1] G=1 mod pi: [1, 0, 0, 1]
c 3 X 3 Liter 3 first bad orders (PphiG-GgammaP): [-1, -1, -1, -1] G=1 mod pi: [1, 0, 0, 1]
[N7c/t1.py: exit code 0, 0.2 s]


1 diff nonzero idx [10 11 12 13 14 16 17 19 20 21] [3 3 6 6 1 6]
  G-G0 0 0 [16 17 18 19 20 21 22 23] [2 5 3 3 5]
  G-G0 0 1 [10 11 12 13 14 15 16 17] [3 2 5 2 4]
  G-G0 1 0 [10 11 12 13 14 16 17 19] [4 4 1 1 6]
  G-G0 1 1 [16 17 18 19 20 21 22 23] [5 3 2 4 1]
4 diff nonzero idx [10 11 12 13 14 15 16 17 18 19] [1634 2159  293  412 2269 1603]
  G-G0 0 0 [10 11 12 13 14 15 16 17] [0 0 0 0 4]
  G-G0 0 1 [10 11 12 13 14 15 16 17] [3 2 5 2 4]
  G-G0 1 0 [10 11 12 13 14 15 16 17] [4 4 1 1 6]
  G-G0 1 1 [10 11 12 13 14 15 16 17] [1 2 4 2 0]
[N7c/t2.py: exit code 0, 0.1 s]


z True []
q True []
h True []
u True []
uk True []
gm True []
gp True []
phi mod p agree: True
gam agree: True
[N7c/t3.py: exit code 0, 0.1 s]
0 [(True, -1, -1), (True, -1, -1), (True, 0, 0), (True, -1, -1)]
1 [(True, 6, 0), (True, 0, 0), (True, 12, 0), (True, 6, 0)]
2 [(True, 90, 0), (True, 84, 0), (True, 96, 6), (True, 90, 0)]
3 [(True, -1, 48), (True, -1, 42), (True, -1, 54), (True, -1, 48)]
4 [(True, -1, -1), (True, -1, -1), (True, -1, -1), (True, -1, -1)]
5 [(True, -1, -1), (True, -1, -1), (True, -1, -1), (True, -1, -1)]
6 [(True, -1, -1), (True, -1, -1), (True, -1, -1), (True, -1, -1)]
7 [(True, -1, -1), (True, -1, -1), (True, -1, -1), (True, -1, -1)]
[N7c/t4.py: exit code 0, 0.1 s]
RECORDED: N7c | Wach-module construction | PASS (t1, t3, t4); t2 diagnostic


**`ext.py 7 …`, `ext.py 11 …`, `ext.py 13 …`: rank one and rank two modulo ℓ for small ℓ.** For rank one with $\varphi=\pi^{s(p-1)}$: $s=1,2$ give normal-form rank 1, and the control $s=p$ (the *très ramifié* case) gives rank 2. For rank two, the kernel classes must span exactly the unit line (NF rank 1, ratio 2). The original command line did not record the weights used for $\ell=11,13$; here $k=p+3,p+4,p+5$ are used (the knob test below covers the edges $r=2,3,p-3,p-2$ for all three primes).

In [73]:
outs = [run("N7c", "ext.py", p, p+3, p+4, p+5) for p in (7, 11, 13)]
for o in outs:
    r1 = re.findall(r"rank one s=(\d+): .*NF rank (\d)", o)
    assert [int(n) for s, n in r1] == [1, 1, 2]          # s=1,2: unit line only; s=p: control detects tres ramifie classes
    assert re.findall(r"'nfrank': (\d)", o) == ["1"]*3
    assert re.findall(r"'cob_in_ker': (\w+)", o) == ["True"]*3
    for line in re.findall(r"NFs of kernel classes: (.*)", o):
        nfs = eval(line)
        assert all(set(n) <= {0, 1} for n in nfs)
        assert all((n.get(1, 0) * pow(n[0], -1, int(re.search(r"^p (\d+)", o).group(1)))) % int(re.search(r"^p (\d+)", o).group(1)) == 2 for n in nfs)
record("N7c", "pki:generic(i), rank two, small l", "crystalline extension classes span only the unit line; control detects tres ramifie", "l = 7, 11, 13; k = l+3..l+5", "PASS", "one family (V_{k,a}, v(a)=1); outside the paper's l range")

p 7 g0 3
rank one s=1: kerdim 3 xrank 2 cobrank 1; coboundary NFs nonzero: []; NF rank 1
   NFs: [{1: 2, 0: 1}]
rank one s=2: kerdim 4 xrank 3 cobrank 2; coboundary NFs nonzero: []; NF rank 1
   NFs: [{2: 5, 1: 5, 0: 1}]
rank one s=7: kerdim 10 xrank 9 cobrank 7; coboundary NFs nonzero: []; NF rank 2
   NFs: [{1: 2, 0: 1}, {1: 6, 6: 5, 3: 1, 2: 1, 13: 3, 12: 2, 11: 1, 10: 1, 9: 2, 8: 3, 5: 2, 4: 3}, {1: 5, 6: 3, 5: 4, 4: 6, 3: 2, 2: 2, 13: 6, 12: 4, 11: 2, 10: 2, 9: 4, 8: 6}, {1: 2, 6: 4, 5: 3, 3: 5, 2: 5, 13: 1, 12: 3, 11: 5, 10: 5, 9: 3, 8: 1, 4: 1}, {1: 1, 6: 2, 5: 5, 4: 4, 3: 6, 2: 6, 13: 4, 12: 5, 11: 6, 10: 6, 9: 5, 8: 4}]
{'p': 7, 'k': 10, 'r': 2, 'X': 4, 'u0': 1, 'beta_ok': True, 'kerdim': 5, 'xrank': 4, 'cobrank': 2, 'cob_in_ker': True, 'nfrank': 1, 'times': (0.003253936767578125, 0.0007617473602294922)}
   coboundary NFs nonzero: []
   NFs of kernel classes: [{1: 1, 0: 4}, {1: 6, 0: 3}]
{'p': 7, 'k': 11, 'r': 3, 'X': 5, 'u0': 1, 'beta_ok': True, 'kerdim': 6, 'xrank': 5, 'cobr

p 11 g0 2
rank one s=1: kerdim 3 xrank 2 cobrank 1; coboundary NFs nonzero: []; NF rank 1
   NFs: [{1: 2, 0: 1}]
rank one s=2: kerdim 4 xrank 3 cobrank 2; coboundary NFs nonzero: []; NF rank 1
   NFs: [{2: 1, 1: 1, 0: 1}]
rank one s=11: kerdim 14 xrank 13 cobrank 11; coboundary NFs nonzero: []; NF rank 2
   NFs: [{1: 2, 0: 1}, {1: 3, 10: 4, 9: 2, 8: 1, 7: 3, 6: 10, 5: 6, 4: 9, 3: 3, 2: 4, 21: 8, 20: 7, 19: 10, 18: 9, 17: 6, 16: 6, 15: 9, 14: 10, 13: 7, 12: 8}, {1: 8, 10: 7, 9: 9, 8: 10, 7: 8, 6: 1, 5: 5, 4: 2, 3: 8, 2: 7, 21: 3, 20: 4, 19: 1, 18: 2, 17: 5, 16: 5, 15: 2, 14: 1, 13: 4, 12: 3}, {1: 9, 10: 1, 9: 6, 3: 9, 2: 1, 21: 2, 20: 10, 19: 8, 18: 5, 17: 7, 16: 7, 15: 5, 14: 8, 13: 10, 12: 2, 8: 3, 7: 9, 6: 8, 5: 7, 4: 5}, {10: 9, 9: 10, 8: 5, 7: 4, 6: 6, 5: 8, 3: 4, 2: 9, 21: 7, 20: 2, 19: 6, 18: 1, 17: 8, 16: 8, 15: 1, 14: 6, 13: 2, 12: 7, 4: 1, 1: 4}, {1: 7, 10: 2, 9: 1, 8: 6, 7: 7, 6: 5, 5: 3, 3: 7, 2: 2, 21: 4, 20: 9, 19: 5, 18: 10, 17: 3, 16: 3, 15: 10, 14: 5, 13: 9, 12: 4, 4: 1

p 13 g0 2
rank one s=1: kerdim 3 xrank 2 cobrank 1; coboundary NFs nonzero: []; NF rank 1
   NFs: [{1: 2, 0: 1}]
rank one s=2: kerdim 4 xrank 3 cobrank 2; coboundary NFs nonzero: []; NF rank 1
   NFs: [{2: 12, 1: 12, 0: 1}]
rank one s=13: kerdim 16 xrank 15 cobrank 13; coboundary NFs nonzero: []; NF rank 2
   NFs: [{1: 2, 0: 1}, {12: 9, 11: 6, 10: 12, 9: 12, 8: 2, 7: 1, 6: 8, 4: 6, 3: 3, 2: 11, 25: 5, 24: 4, 23: 6, 22: 2, 21: 1, 20: 10, 19: 10, 18: 1, 17: 2, 16: 6, 15: 4, 14: 5}, {12: 12, 11: 8, 10: 3, 9: 3, 8: 7, 7: 10, 6: 2, 4: 8, 3: 4, 2: 6, 25: 11, 24: 1, 23: 8, 22: 7, 21: 10, 20: 9, 19: 9, 18: 10, 17: 7, 16: 8, 15: 1, 14: 11}, {12: 7, 11: 9, 9: 5, 8: 3, 7: 8, 6: 12, 4: 9, 3: 11, 2: 10, 25: 1, 24: 6, 23: 9, 22: 3, 21: 8, 20: 2, 19: 2, 18: 8, 17: 3, 16: 9, 15: 6, 14: 1, 10: 5}, {12: 2, 11: 10, 10: 7, 9: 7, 8: 12, 7: 6, 6: 9, 4: 10, 3: 5, 2: 1, 25: 4, 24: 11, 23: 10, 22: 12, 21: 6, 20: 8, 19: 8, 18: 6, 17: 12, 16: 10, 15: 11, 14: 4}, {12: 5, 11: 12, 10: 11, 8: 4, 7: 2, 6: 3, 4: 12, 3

**`knob.py`: edges of the range.** $\ell=7,11,13$ and $r=k-1-\ell\in\{2,3,\ell-3,\ell-2\}$. *Refutation would be:* NF rank $\ne1$ or ratio $\ne2$.

In [74]:
%%writefile N7c/knob.py
import sys; sys.path.insert(0,'.')
from ext import *
for p in (7,11,13):
    g0=next(g for g in range(2,p) if all(pow(g,(p-1)//f,p)!=1 for f in range(2,p) if (p-1)%f==0) and pow(g,p-1,p*p)!=1)
    for r in (2,3,p-3,p-2):
        k=p+r+1
        try:
            R2=rank_two(p,k,g0)
            seen=[]
            for n in R2['nfs']:
                if n and n not in seen: seen.append(n)
            print("p",p,"r",r,"kerdim",R2['kerdim'],"xrank",R2['xrank'],"cobrank",R2['cobrank'],"cob_in",R2['cob_in_ker'],"cobNF",[n for n in R2['nfc'] if n],"NFrank",R2['nfrank'],"ratios",set((n.get(1,0)*pow(n.get(0,1),-1,p))%p if n.get(0,0) else 'inf' for n in seen),"slots",sorted(set(kk for n in seen for kk in n)))
        except Exception as e:
            print("p",p,"r",r,"assert",str(e)[:80])

Overwriting N7c/knob.py


In [75]:
o = run("N7c", "knob.py")
rows = re.findall(r"NFrank (\d) ratios \{(\d+)\} slots \[0, 1\]", o)
assert len(rows) == 12 and all(a == "1" and b == "2" for a, b in rows)
record("N7c", "pki:generic(i), edges", "same, at the edges r = 2, 3, l-3, l-2", "l = 7, 11, 13", "PASS (12/12)", "one family")

p 7 r 2 kerdim 5 xrank 4 cobrank 2 cob_in True cobNF [] NFrank 1 ratios {2} slots [0, 1]
p 7 r 3 kerdim 6 xrank 5 cobrank 3 cob_in True cobNF [] NFrank 1 ratios {2} slots [0, 1]
p 7 r 4 kerdim 7 xrank 6 cobrank 4 cob_in True cobNF [] NFrank 1 ratios {2} slots [0, 1]
p 7 r 5 kerdim 8 xrank 7 cobrank 5 cob_in True cobNF [] NFrank 1 ratios {2} slots [0, 1]
p 11 r 2 kerdim 5 xrank 4 cobrank 2 cob_in True cobNF [] NFrank 1 ratios {2} slots [0, 1]
p 11 r 3 kerdim 6 xrank 5 cobrank 3 cob_in True cobNF [] NFrank 1 ratios {2} slots [0, 1]
p 11 r 8 kerdim 11 xrank 10 cobrank 8 cob_in True cobNF [] NFrank 1 ratios {2} slots [0, 1]
p 11 r 9 kerdim 12 xrank 11 cobrank 9 cob_in True cobNF [] NFrank 1 ratios {2} slots [0, 1]
p 13 r 2 kerdim 5 xrank 4 cobrank 2 cob_in True cobNF [] NFrank 1 ratios {2} slots [0, 1]
p 13 r 3 kerdim 6 xrank 5 cobrank 3 cob_in True cobNF [] NFrank 1 ratios {2} slots [0, 1]
p 13 r 10 kerdim 13 xrank 12 cobrank 10 cob_in True cobNF [] NFrank 1 ratios {2} slots [0, 1]
p 13 r

**`run41.py`: the case in the paper's range, $\ell=41$.** $k=61,62,63,64$ (linear algebra over $\mathbb F_{41}$ of size about $2420\times2445$, roughly 20–40 s per weight). Default here: $k=61,64$; set `N7C_KS = [61, 62, 63, 64]` for the full set of the report. *Refutation would be:* NF rank $\ne1$, ratio $\ne2$, or slots other than $\{0,1\}$.

In [76]:
N7C_KS = [61, 64]     # the report used [61, 62, 63, 64] (all four were re-run at full size while preparing this notebook: same outcome)

In [77]:
%%writefile N7c/run41.py
import sys; sys.path.insert(0,'.')
from ext import *
p=41; g0=next(g for g in range(2,p) if all(pow(g,(p-1)//f,p)!=1 for f in range(2,p) if (p-1)%f==0) and pow(g,p-1,p*p)!=1)
R1=rank_one(p,1,g0); print("units line NF:",[n for n in R1['nfs'] if n][:1], "rank",R1['nfrank'])
for k in [int(a) for a in sys.argv[1:]]:
    t=time.time()
    R2=rank_two(p,k,g0)
    print({kk:vv for kk,vv in R2.items() if kk not in('nfs','nfc','nfbasis')}, "total %.1fs"%(time.time()-t))
    print("   coboundary NFs nonzero:",[n for n in R2['nfc'] if n])
    seen=[]
    for n in R2['nfs']:
        if n and n not in seen: seen.append(n)
    print("   NFs of kernel classes:",seen[:8], " ratios slot1/slot0:",set((n.get(1,0)*pow(n.get(0,1),-1,p))%p for n in seen), "other slots:",set(kk for n in seen for kk in n)-{0,1})
    sys.stdout.flush()

Overwriting N7c/run41.py


In [78]:
o = run("N7c", "run41.py", *N7C_KS)
assert "units line NF: [{1: 2, 0: 1}] rank 1" in o
assert re.findall(r"'nfrank': (\d)", o) == ["1"]*len(N7C_KS)
assert re.findall(r"ratios slot1/slot0: \{(\d+)\}", o) == ["2"]*len(N7C_KS)
assert re.findall(r"other slots: (.*)", o) == ["set()"]*len(N7C_KS)
if len(N7C_KS) < 4: record("N7c", "pki:generic(i), l = 41, remaining weights", "same", "k in {62, 63}", "recorded only (NF rank 1, ratio 2)", "set N7C_KS to all four to re-run")
record("N7c", "pki:generic(i), rank two, l = 41", "extension classes of a rank-two crystalline lattice lie in the unit line", f"l = 41, k in {N7C_KS}", "PASS", "rank two, one family (irreducible, v(a_l)=1); mod l")

units line NF: [{1: 2, 0: 1}] rank 1
{'p': 41, 'k': 61, 'r': 19, 'X': 13, 'u0': 1, 'beta_ok': True, 'kerdim': 22, 'xrank': 21, 'cobrank': 19, 'cob_in_ker': True, 'nfrank': 1, 'times': (0.4236030578613281, 18.068442821502686)} total 18.5s
   coboundary NFs nonzero: []
   NFs of kernel classes: [{1: 26, 0: 13}, {1: 7, 0: 24}, {1: 1, 0: 21}, {1: 25, 0: 33}, {1: 14, 0: 7}, {1: 39, 0: 40}, {1: 11, 0: 26}, {1: 28, 0: 14}]  ratios slot1/slot0: {2} other slots: set()
{'p': 41, 'k': 64, 'r': 22, 'X': 28, 'u0': 1, 'beta_ok': True, 'kerdim': 25, 'xrank': 24, 'cobrank': 22, 'cob_in_ker': True, 'nfrank': 1, 'times': (0.4902801513671875, 19.200372219085693)} total 19.7s
   coboundary NFs nonzero: []
   NFs of kernel classes: [{1: 15, 0: 28}, {1: 30, 0: 15}, {1: 24, 0: 12}, {1: 14, 0: 7}, {1: 13, 0: 27}, {1: 33, 0: 37}, {1: 6, 0: 3}, {1: 27, 0: 34}]  ratios slot1/slot0: {2} other slots: set()
[N7c/run41.py 61 64: exit code 0, 38.3 s]
RECORDED: N7c | pki:generic(i), l = 41, remaining weights | recorde

**`integral.py`: integral generator.** For $\ell=7$, $k=10,11,12$ ($\ell=11$, $k=16$ in the report): liftable classes have dimension 2, 1, 1 modulo $\ell,\ell^2,\ell^3$, and the surviving class has non-zero unit image, so the statement also holds for the lattices $Y'\supseteq Y$. *Refutation would be:* a surviving class outside the unit line (ratio $\ne2$) or dimensions not dropping to 1. Default: $\ell=7$, $k=10,11,12$, three values of $X_1$ each (the report used all $X$ modulo $\ell^2$; set `N7C_INT_FULL = True` for all seven, and for $\ell=11$, $k=16$).

In [79]:
N7C_INT_FULL = False
N7C_INT_X1 = [] if N7C_INT_FULL else [0, 1, 2]     # [] = all X1 mod p

In [80]:
%%writefile N7c/integral.py
# Integral generator of H^1_f(Y) (Wach extensions of N by the trivial module), reduced mod p, by p-adic lifting
import sys; sys.path.insert(0,'.')
from ext import *
from wach import _frob

def modp_kernel(A,p): return kernel_modp(A,p)

def lift_chain(Aint,p,cmax):
    """Aint: integer matrix mod p^cmax. Returns list K_m (m=1..cmax) of generator sets (mod p vectors) of the
    reductions mod p of solutions of A v=0 mod p^m."""
    n=Aint.shape[1]
    Abar=Aint%p
    K1=modp_kernel(Abar,p)
    gens=[v.astype(object) for v in K1]          # integer vectors with A g = 0 mod p^m
    out=[[v%p for v in K1]]
    Aobj=Aint.astype(object)
    for m in range(1,cmax):
        pm=p**m
        t=len(gens)
        B=np.zeros((Aint.shape[0],t),dtype=np.int64)
        for i,g in enumerate(gens):
            w=Aobj.dot(g)
            assert all(int(x)%pm==0 for x in w)
            B[:,i]=np.array([(int(x)//pm)%p for x in w],dtype=np.int64)
        big=np.concatenate([B,Abar],axis=1)
        ker=modp_kernel(big,p)
        newg=[]; red=[]
        for kv in ker:
            a=kv[:t]; u=kv[t:]
            if not a.any(): continue
            g=sum(int(a[i])*gens[i] for i in range(t))+pm*u.astype(object)
            newg.append(g); red.append(np.array([int(x)%p for x in g],dtype=np.int64))
        # generators: newg, p*g_i, and p^m * K1
        gens=newg+[p*g for g in gens]+[pm*v.astype(object) for v in K1]
        # prune generators that are 0 mod p^{m+1}
        gens=[g for g in gens if any(int(x)%(p**(m+1)) for x in g)]
        out.append(red)
    return out

def run(p,k,g0,X1,cmax=3,verbose=True):
    r=k-1-p; M=(p-1)*(k-1); D=M+r+1; N=D+cmax*M+2
    d=setup(p,k,g0,N,c=cmax); S=d['S']; m=d['m']; z=d['z']; phi=d['phi']; gam=d['gam']
    zc=int(z[p-1])%p
    X0=pow(zc,-1,p); X=X0+p*X1
    build_G(d,X); assert d['check']==[-1]*4
    G=d['G']; Xz=d['Xz']
    phipi=d['phipi']; gpi=d['gpi']
    pG11=phi(G[0][0]); ppG12=phi(phi(G[0][1])); pgXz=phi(gam(Xz)); gqk=gam(d['qk'])
    phi2pi=phi(phipi)
    cols=[]
    pp=S.one(); gp_=S.one()
    for i in range(D):
        col=(S.shift(pG11,i)-S.mul(pp,ppG12)-gp_)%m
        cols.append(col)
        pp=S.mul(pp,phipi); gp_=S.mul(gp_,gpi)
    p1=S.one(); p2=S.one()
    for j in range(1,N):
        p1=S.mul(p1,phipi); p2=S.mul(p2,phi2pi)
        col=(p2-S.mul(pgXz,p1)+S.shift(gqk,j))%m
        cols.append((-col)%m)
    A=np.array(cols).T
    Ks=lift_chain(A,p,cmax)
    # mod p data for NF
    wbar=Xz%p; Sp=R(p,N)
    beta=(-wbar)%p
    for it in range(60):
        fb=Sp.shift(_frob(Sp,beta,p),-p*(p-1)); t=Sp.shift(Sp.inv(fb),M-p*(p-1)); nb=(-wbar-t)%p
        if np.array_equal(nb,beta): break
        beta=nb
    mu=(-_frob(Sp,beta,p))%p; u=Sp.shift(mu,-p*(p-1)); top=p*p; T=R(p,top+1)
    ut=u[:top+1].copy(); phirho=T.one(); t=ut
    for n in range(1,4):
        t=_frob(T,t,p); phirho=T.mul(phirho,t)
    def nf_of(x):
        xx=np.zeros(top+1,dtype=np.int64); L=min(top+1,len(x)); xx[:L]=x[:L]
        return NF(T.mul(phirho,xx),p,top)
    pw_=_frob(Sp,wbar,p)
    cob=[]
    for j in range(0,r+1):
        y=(Sp.mono(p*p*j)-Sp.shift(pw_,p*j)+Sp.mono(M+j))%p
        cob.append(((-y)%p)[:D].copy())
    res=[]
    for mm,K in enumerate(Ks,1):
        xs=[v[:D] for v in K]
        dimx=rank_modp(xs+cob,p)-rank_modp(cob,p) if xs else 0
        nfs=[]
        for x in xs:
            n=nf_of(x)
            if n not in nfs: nfs.append(n)
        nz=[n for n in nfs if n]
        ratios=set((n.get(1,0)*pow(n.get(0,1),-1,p))%p if n.get(0,0) else 'inf' for n in nz)
        res.append((mm,len(K),dimx,len(nz),ratios,sorted(set(kk for n in nz for kk in n))))
    return dict(u0=int(u[0]),X=X,res=res)

if __name__=="__main__":
    p=int(sys.argv[1]); k=int(sys.argv[2]); cmax=int(sys.argv[3])
    g0=next(g for g in range(2,p) if all(pow(g,(p-1)//f,p)!=1 for f in range(2,p) if (p-1)%f==0) and pow(g,p-1,p*p)!=1)
    X1s=[int(a) for a in sys.argv[4:]] or list(range(p))
    for X1 in X1s:
        t=time.time()
        out=run(p,k,g0,X1,cmax)
        print("p",p,"k",k,"X",out['X'],"u0",out['u0'],"| (m, #gens, dim classes mod cob, #nonzero NF, slot1/slot0, slots):",out['res'],"%.1fs"%(time.time()-t)); sys.stdout.flush()

Overwriting N7c/integral.py


In [81]:
outs = [run("N7c", "integral.py", 7, k, 3, *N7C_INT_X1) for k in (10, 11, 12)]
if N7C_INT_FULL: outs.append(run("N7c", "integral.py", 11, 16, 3))
for o in outs:
    for line in re.findall(r"\| \(m, #gens.*?\): (\[.*\]) [\d.]+s", o):
        res = eval(line)
        assert [r[2] for r in res] == [2, 1, 1]
        assert all(r[4] == {2} and r[5] == [0, 1] for r in res)
record("N7c", "integral generator", "liftable classes: dims 2,1,1 mod l,l^2,l^3; surviving class in unit line", "l = 7, k = 10,11,12" + (" + l=11,k=16" if N7C_INT_FULL else ", X1 in {0,1,2}"), "PASS", "small l; one family")
record("N7", "pg:I, pg:II, pg:III, pg:loss, pg:odd", "-", "-", "NOT TESTED numerically", "hand tests in rank one only; rank-two test only for one family (above)")

p 7 k 10 X 4 u0 1 | (m, #gens, dim classes mod cob, #nonzero NF, slot1/slot0, slots): [(1, 59, 2, 2, {2}, [0, 1]), (2, 63, 1, 4, {2}, [0, 1]), (3, 185, 1, 6, {2}, [0, 1])] 0.5s
p 7 k 10 X 11 u0 1 | (m, #gens, dim classes mod cob, #nonzero NF, slot1/slot0, slots): [(1, 59, 2, 2, {2}, [0, 1]), (2, 63, 1, 4, {2}, [0, 1]), (3, 185, 1, 6, {2}, [0, 1])] 0.5s
p 7 k 10 X 18 u0 1 | (m, #gens, dim classes mod cob, #nonzero NF, slot1/slot0, slots): [(1, 59, 2, 2, {2}, [0, 1]), (2, 63, 1, 4, {2}, [0, 1]), (3, 185, 1, 5, {2}, [0, 1])] 0.5s
[N7c/integral.py 7 10 3 0 1 2: exit code 0, 1.5 s]


p 7 k 11 X 5 u0 1 | (m, #gens, dim classes mod cob, #nonzero NF, slot1/slot0, slots): [(1, 66, 2, 2, {2}, [0, 1]), (2, 71, 1, 4, {2}, [0, 1]), (3, 208, 1, 6, {2}, [0, 1])] 0.6s
p 7 k 11 X 12 u0 1 | (m, #gens, dim classes mod cob, #nonzero NF, slot1/slot0, slots): [(1, 66, 2, 2, {2}, [0, 1]), (2, 71, 1, 5, {2}, [0, 1]), (3, 208, 1, 6, {2}, [0, 1])] 0.6s
p 7 k 11 X 19 u0 1 | (m, #gens, dim classes mod cob, #nonzero NF, slot1/slot0, slots): [(1, 66, 2, 2, {2}, [0, 1]), (2, 71, 1, 5, {2}, [0, 1]), (3, 208, 1, 5, {2}, [0, 1])] 0.6s
[N7c/integral.py 7 11 3 0 1 2: exit code 0, 2.0 s]


p 7 k 12 X 2 u0 1 | (m, #gens, dim classes mod cob, #nonzero NF, slot1/slot0, slots): [(1, 73, 2, 5, {2}, [0, 1]), (2, 79, 1, 6, {2}, [0, 1]), (3, 231, 1, 6, {2}, [0, 1])] 0.8s
p 7 k 12 X 9 u0 1 | (m, #gens, dim classes mod cob, #nonzero NF, slot1/slot0, slots): [(1, 73, 2, 5, {2}, [0, 1]), (2, 79, 1, 4, {2}, [0, 1]), (3, 231, 1, 6, {2}, [0, 1])] 0.8s
p 7 k 12 X 16 u0 1 | (m, #gens, dim classes mod cob, #nonzero NF, slot1/slot0, slots): [(1, 73, 2, 5, {2}, [0, 1]), (2, 79, 1, 5, {2}, [0, 1]), (3, 231, 1, 6, {2}, [0, 1])] 0.8s
[N7c/integral.py 7 12 3 0 1 2: exit code 0, 2.5 s]
RECORDED: N7c | integral generator | PASS
RECORDED: N7 | pg:I, pg:II, pg:III, pg:loss, pg:odd | NOT TESTED numerically


### Node N8 — the test form of least valuation (`psu:small`, `psu:arch`) and the count `pa:count`

Model: $d=7$, $K=\mathbb Q(\sqrt{-7})$, normalised lattice $\Omega\mathcal O_K$ with admissible point $t=\Omega/2$; exact arithmetic in $\mathbb Q(\sqrt7)$ with valuations at all places above $p$. `psu:small` and `psu:arch` themselves were not tested (their transfer steps have no small instance); the tests check the one-curve and two-place count statements on which they rest.

**`t0_g2.py`:** numerical determination of $a_1^2=3\sqrt7$ and $g_2=1/\sqrt7$ (60 digits + PSLQ). *Refutation would be:* a different algebraic relation.

In [82]:
%%writefile N8/nf.py
# Q(s), s = sqrt(7): pairs of Fractions.  Series in z^2 (even series) as lists indexed by m (coefficient of z^{2m}).
from fractions import Fraction as Fr
from math import factorial
class S7:
    __slots__=("a","b")
    def __init__(s,a=0,b=0): s.a=Fr(a); s.b=Fr(b)
    def __add__(s,o):
        o=S7c(o); return S7(s.a+o.a,s.b+o.b)
    __radd__=__add__
    def __neg__(s): return S7(-s.a,-s.b)
    def __sub__(s,o): o=S7c(o); return S7(s.a-o.a,s.b-o.b)
    def __rsub__(s,o): return S7c(o)-s
    def __mul__(s,o):
        o=S7c(o); return S7(s.a*o.a+7*s.b*o.b, s.a*o.b+s.b*o.a)
    __rmul__=__mul__
    def inv(s):
        n=s.a*s.a-7*s.b*s.b; return S7(s.a/n,-s.b/n)
    def __truediv__(s,o): return s*S7c(o).inv()
    def iszero(s): return s.a==0 and s.b==0
    def __repr__(s): return "(%s+%s*s7)"%(s.a,s.b)
def S7c(o): return o if isinstance(o,S7) else S7(o,0)
def vq(r,p):
    if r==0: return float('inf')
    v=0; n=r.numerator; d=r.denominator
    while n%p==0: n//=p; v+=1
    while d%p==0: d//=p; v-=1
    return v
def vals(x,p):
    """valuations of x=a+b sqrt7 at the places above p (list; v(p)=1)"""
    if x.iszero(): return [float('inf')]
    if p==7:
        return [min(vq(x.a,7), vq(x.b,7)+Fr(1,2))]
    N=x.a*x.a-7*x.b*x.b
    if pow(7,(p-1)//2,p)!=1:      # inert in Q(sqrt7)
        return [Fr(vq(N,p),2)]
    va,vb=vq(x.a,p),vq(x.b,p)
    m=min(va,vb)
    if va!=vb: return [m,m]
    return [m, vq(N,p)-m]
def vmin(x,p): return min(vals(x,p))
def vmax(x,p): return max(vals(x,p))
S=S7(0,1)
A1SQ=3*S            # a1^2
G2=S/7              # g_2 = 1/sqrt7
def xseries(M):
    c=[S7(0)]*(M+1); c[1]=S7(1)
    for m in range(2,M+1):
        conv=S7(0)
        for i in range(1,m-1): conv=conv+c[i]*c[m-1-i]
        c[m]=(6*conv+A1SQ*c[m-1])/((2*m)*(2*m-1))
    return c
def gseries(M):
    c=xseries(M); g=[S7(0)]*(M+1); g[1]=G2
    for m in range(2,M+1): g[m]=c[m-1]/((2*m)*(2*m-1))
    return g
def mul(u,v,M):
    w=[S7(0)]*(M+1)
    for i,a in enumerate(u):
        if i>M: break
        if a.iszero(): continue
        for j,b in enumerate(v):
            if i+j>M: break
            if b.iszero(): continue
            w[i+j]=w[i+j]+a*b
    return w
def expser(h,M):
    # exp of a series with h[0]=0 : e' = h' e
    e=[S7(0)]*(M+1); e[0]=S7(1)
    for n in range(1,M+1):
        acc=S7(0)
        for k in range(1,n+1):
            if not h[k].iszero(): acc=acc+k*h[k]*e[n-k]
        e[n]=acc/n
    return e

Overwriting N8/nf.py


In [83]:
%%writefile N8/t0_g2.py
# numeric determination of a1^2 and g2 for the normalised lattice Omega*O_K, K=Q(sqrt(-7)), t=Omega/2
from mpmath import mp, mpf, mpc, pi, sqrt, exp, pslq, re, im
mp.dps = 60
tau = (1+1j*sqrt(7))/2
q = exp(1j*pi*tau)
th3 = sum(q**(n*n) for n in range(-40,41))
th4 = sum((-1)**n*q**(n*n) for n in range(-40,41))
Om4 = pi**4*th3**4*th4**4
print("Omega^4 =", Om4)
Om = pi*abs(th3)**2
e1 = (pi**2/3)*(th3**4+th4**4)
a1sq = 12*e1/Om**2
print("a1^2 =", a1sq, " 3*sqrt7 =", 3*sqrt(7))
Q = exp(2j*pi*tau)
def sig(n): return sum(d for d in range(1,n+1) if n%d==0)
E2 = 1-24*sum(sig(n)*Q**n for n in range(1,60))
s2 = (pi**2/3)*E2 - pi/im(tau)
print("s2(O_K) =", s2)
g2 = (s2/Om**2 + a1sq/12)/2
print("g2 =", g2)
print("pslq [g2,1,sqrt7]:", pslq([re(g2), mpf(1), sqrt(7)], maxcoeff=10**6, maxsteps=10**5))

Overwriting N8/t0_g2.py


In [84]:
o = run("N8", "t0_g2.py")
a1 = re.search(r"a1\^2 = \(([\d.]+) \+ 0\.0j\)  3\*sqrt7 = ([\d.]+)", o)
assert a1 and a1.group(1)[:50] == a1.group(2)[:50]
assert "pslq [g2,1,sqrt7]: [-7, 0, 1]" in o       # 7 g2 = sqrt 7, i.e. g2 = 1/sqrt 7
record("N8", "model constants (pf:two)", "a1^2 = 3 sqrt 7, g2 = 1/sqrt 7", "d = 7, 60 digits + PSLQ", "PASS", "one curve")

Omega^4 = (97.7925029688610691069619130403727974912003435348982712645378 + 4.44338605041230931972169075070055061978175127195230073132038e-61j)
a1^2 = (7.937253933193771771504847260917781277130777549247350541105 + 0.0j)  3*sqrt7 = 7.937253933193771771504847260917781277130777549247350541105
s2(O_K) = (0.934423537768746247884220102826182523448217099919421754175488 - 2.21112259282618998494844000910812011865933817566033418280051e-63j)
g2 = (0.377964473009227227214516536234180060815751311868921454338333 - 1.11796973184120331552236873735174352956480435344549065935138e-64j)
pslq [g2,1,sqrt7]: [-7, 0, 1]
[N8/t0_g2.py: exit code 0, 0.1 s]
RECORDED: N8 | model constants (pf:two) | PASS


**`t1_onecurve.py`: `pa:param`, `psm:param`(d), `psm:lambda`, `pa:units`(a), `pa:frob`.** T1a: $v(q!\,[q]T)=\gamma_p(q-1)$ for $p=3,5$ (inert), $7=d$, $11,23$ (split). T1b: $x$ has integral coefficients in degrees $<q$ and $x_{p-1}$ is a unit. T1c–e: $\exp(qg(z)-g(\pi z))$ is exactly a polynomial in $x$ of degree $(q-1)/2$ with all coefficients divisible by $\pi$, for $\pi=\sqrt{-7},3,5$. *Refutation would be:* any mismatch with the predicted valuations, a non-zero remainder, or a coefficient not divisible by $\pi$.

In [85]:
%%writefile N8/t1_onecurve.py
from nf import *
M=32
c=xseries(M); g=gseries(M)
print("x_4 =",c[2]," x_6 =",c[3])
# (a) psm:param(d)/pa:param: v((q^j)! [q^j]T) = gamma_p (q^j-1).  [q]T=[q]y=(q+1)/2 * x_{q+1} for odd q
print("== T1a: v_p(q! [q]y) vs gamma_p (q-1)")
for p,q,gam in [(3,9,Fr(3,8)),(5,25,Fr(5,24)),(7,7,Fr(1,12)),(11,11,Fr(0)),(23,23,Fr(0)),(13,169,None)]:
    if (q+1)//2>M: continue
    yq=Fr(q+1,2)*c[(q+1)//2]
    vv=[v+vq(Fr(factorial(q)),p) for v in vals(yq,p)]
    print(" p=%d q=%d  v(q![q]y)=%s  predicted %s"%(p,q,vv,gam*(q-1)))
# (b) integrality of x in degrees < q (pa:param(c)) and unit x_{p-1} (psm:lambda)
print("== T1b: min valuation of x_n for n<q ; v(x_{p-1})")
for p,q in [(3,9),(5,25),(7,7),(11,11),(13,60),(19,60)]:
    mv=min(vmin(c[m],p) for m in range(1,min(M,(q-1)//2)+1))
    print(" p=%d: min v(x_n), n<%d : %s ; v(x_{p-1})=%s"%(p,q,mv,vals(c[(p-1)//2],p)))
# (c) pa:frob for pi=sqrt(-7): exp(7g(z)-g(sqrt(-7)z)) = cubic in x, R==1 mod sqrt7
h=[S7(0)]*(M+1)
for m in range(1,M+1): h[m]=(7-(-7)**m)*g[m]
E=expser(h,M)
# fit cubic
xp=[ [S7(1)]+[S7(0)]*M ]
for i in range(3): xp.append(mul(xp[-1],c,M))
r=[S7(0)]*4; r[0]=S7(1)
res=E[:]
res=[res[i]-xp[0][i] for i in range(M+1)]
for i in range(1,4):
    r[i]=res[i]            # coefficient of z^{2i}; x^i = z^{2i}+...
    res=[res[k]-r[i]*xp[i][k] for k in range(M+1)]
print("== T1c: f_pi (pi=sqrt(-7)) = 1 + r1 x + r2 x^2 + r3 x^3 with r =",r[1:], " remainder zero up to z^%d: %s"%(2*M,all(t.iszero() for t in res)))
print("   v_7(r_i) =",[vals(t,7) for t in r[1:]])
# (d) pa:frob for p=3 (pi=3,q=9): exp(9g(z)-g(3z)) = quartic in x, == 1 mod 3
h=[S7(0)]*(M+1)
for m in range(1,M+1): h[m]=(9-9**m)*g[m]
E=expser(h,M)
xp=[ [S7(1)]+[S7(0)]*M ]
for i in range(4): xp.append(mul(xp[-1],c,M))
res=[E[i]-xp[0][i] for i in range(M+1)]; r=[S7(1)]
for i in range(1,5):
    r.append(res[i]); res=[res[k]-r[i]*xp[i][k] for k in range(M+1)]
print("== T1d: f_3 = quartic in x, remainder zero: %s ; v_3(r_i) = %s"%(all(t.iszero() for t in res),[vals(t,3) for t in r[1:]]))
# (e) alpha=2+sqrt(-7) not available in real field; use alpha=3 above. Also p=5: degree 12 polynomial
h=[S7(0)]*(M+1)
for m in range(1,M+1): h[m]=(25-25**m)*g[m]
E=expser(h,M)
xp=[ [S7(1)]+[S7(0)]*M ]
for i in range(12): xp.append(mul(xp[-1],c,M))
res=[E[i]-xp[0][i] for i in range(M+1)]; r=[S7(1)]
for i in range(1,13):
    r.append(res[i]); res=[res[k]-r[i]*xp[i][k] for k in range(M+1)]
print("== T1e: f_5 = degree-12 polynomial in x, remainder zero: %s ; v_5(r_i) = %s"%(all(t.iszero() for t in res),[vals(t,5)[0] for t in r[1:]]))

Overwriting N8/t1_onecurve.py


In [86]:
from fractions import Fraction
o = run("N8", "t1_onecurve.py")
t1a = re.findall(r"v\(q!\[q\]y\)=\[(.*?)\]  predicted (\S+)", o)
for vals, pred in t1a:
    vs = [Fraction(eval(v.strip())) if "Fraction" not in v else eval(v.strip()) for v in re.split(r",\s*(?![^()]*\))", vals)]
    assert all(v == Fraction(pred) for v in vs), (vals, pred)
assert len(t1a) == 5
assert all(x == "0" for x in re.findall(r"min v\(x_n\), n<\d+ : (\S+)", o))
assert o.count("remainder zero up to z^64: True") == 1 and o.count("remainder zero: True") == 2
record("N8", "pa:param, psm:param(d), psm:lambda, pa:units(a), pa:frob", "valuations of torsion coefficients; Frobenius polynomial identity", "p = 3,5,7,11,13,19,23", "PASS", "one CM curve (d = 7)")

x_4 = (0+1/4*s7)  x_6 = (3/8+0*s7)
== T1a: v_p(q! [q]y) vs gamma_p (q-1)
 p=3 q=9  v(q![q]y)=[3, 3]  predicted 3
 p=5 q=25  v(q![q]y)=[Fraction(5, 1)]  predicted 5
 p=7 q=7  v(q![q]y)=[Fraction(1, 2)]  predicted 1/2
 p=11 q=11  v(q![q]y)=[Fraction(0, 1)]  predicted 0
 p=23 q=23  v(q![q]y)=[Fraction(0, 1)]  predicted 0
== T1b: min valuation of x_n for n<q ; v(x_{p-1})
 p=3: min v(x_n), n<9 : 0 ; v(x_{p-1})=[0, 0]
 p=5: min v(x_n), n<25 : 0 ; v(x_{p-1})=[Fraction(0, 1)]
 p=7: min v(x_n), n<7 : 0 ; v(x_{p-1})=[0]
 p=11: min v(x_n), n<11 : 0 ; v(x_{p-1})=[Fraction(0, 1)]
 p=13: min v(x_n), n<60 : 0 ; v(x_{p-1})=[Fraction(0, 1)]
 p=19: min v(x_n), n<60 : 0 ; v(x_{p-1})=[0, 0]
== T1c: f_pi (pi=sqrt(-7)) = 1 + r1 x + r2 x^2 + r3 x^3 with r = [(0+2*s7), (7+0*s7), (0+1*s7)]  remainder zero up to z^64: True
   v_7(r_i) = [[Fraction(1, 2)], [1], [Fraction(1, 2)]]
== T1d: f_3 = quartic in x, remainder zero: True ; v_3(r_i) = [[inf], [1, 1], [1, 1], [1, 1]]
== T1e: f_5 = degree-12 polynomial in x, 

**`t2_coupling.py`: `psm:coupling`** (hence `pa:onecurve` + `psm:lambda` + `psm:six`) at one place, self-partner, $p$ inert: $v([z'^{pj}z''^{j}]G_0)=-j-v_p(j!)$ **exactly**, for $p=3$ ($j\le11$), $5$ ($j\le7$), $13$ ($j\le3$), $19$ ($j\le2$), plus a split control. *Refutation would be:* any valuation different from the prediction.

In [87]:
%%writefile N8/t2_coupling.py
# psm:coupling in the smallest case: E=K=Q(sqrt(-7)), one place, Theta=theta, self-partner, p inert.
# G_0(z',z'') = theta(z'+z'') * prod_{c>=1} theta(delta_c z'), sum_c delta_c^2 = 0 (delta_0=1).
# p=3,19: delta=(1,sqrt(-2),1,0,0,0); p=5,13: delta=(1,i,0,0,0,0).  theta(delta z)=exp(-sum g_{2m} delta^{2m} z^{2m}).
# Prediction (Step 3-4 of the proof): v([z'^{p^e j} z''^j] G_0) = j(1/(p-1)-1) - v(j!) - rho_p (p^e+1) j  with e=1,
# i.e. = -j - v(j!)   (exact).
from nf import *
from math import comb
import sys
def run(p,dsq,M2):   # dsq: list of delta_c^2 for c>=1 ; series up to z^{2*M2}
    g=gseries(M2)
    th=expser([-t for t in g],M2)          # theta, even series: th[m]=coeff z^{2m}
    hh=[S7(0)]*(M2+1)
    for m in range(1,M2+1):
        hh[m]=-g[m]*sum(Fr(d)**m for d in dsq)
    H=expser(hh,M2)                         # prod_{c>=1} theta(delta_c z')
    out=[]
    j=1
    while (p+1)*j<=2*M2:
        a,b=p*j,j
        # [z'^a z''^b] theta(z'+z'') H(z') = sum_k C(b+k,b) [b+k]theta [a-k]H
        tot=S7(0)
        for k in range(0,a+1):
            if (b+k)%2 or (a-k)%2: continue
            tot=tot+comb(b+k,b)*th[(b+k)//2]*H[(a-k)//2]
        vj=sum(j//p**i for i in range(1,10))
        out.append((j,vals(tot,p),-j-vj))
        j+=1
    return out
for p,dsq,M2 in [(3,[-2,1],22),(5,[-1],21),(13,[-1],21),(19,[-2,1],20)]:
    print("p=%d (inert in K): (j, v([z'^{pj} z''^j]G_0) at the places of Q(sqrt7), predicted -j-v(j!))"%p)
    print("  ",run(p,dsq,M2))
# control: split prime p=11 (gamma=0): no coupling, coefficients integral -> valuations >= 0 expected
print("control p=11 (split), delta=(1,sqrt(-2),1): -2 is a QR mod 11:", pow(-2%11,5,11)==1)
print("  ",run(11,[-2,1],18))

Overwriting N8/t2_coupling.py


In [88]:
o = run("N8", "t2_coupling.py")
tot = 0
for block in re.findall(r"^   (\[.*\])$", o, re.M):
    for j, vals, pred in eval(block, {"Fraction": Fraction}):
        assert all(v == pred for v in vals), (j, vals, pred); tot += 1
assert tot == 26
record("N8", "psm:coupling", "exact valuation -j - v(j!) of coupling coefficients", "p = 3 (j<=11), 5 (j<=7), 13 (j<=3), 19 (j<=2), split control p=11", f"PASS ({tot} coefficients)", "one place, Theta = theta")

p=3 (inert in K): (j, v([z'^{pj} z''^j]G_0) at the places of Q(sqrt7), predicted -j-v(j!))
   [(1, [-1, -1], -1), (2, [-2, -2], -2), (3, [-4, -4], -4), (4, [-5, -5], -5), (5, [-6, -6], -6), (6, [-8, -8], -8), (7, [-9, -9], -9), (8, [-10, -10], -10), (9, [-13, -13], -13), (10, [-14, -14], -14), (11, [-15, -15], -15)]
p=5 (inert in K): (j, v([z'^{pj} z''^j]G_0) at the places of Q(sqrt7), predicted -j-v(j!))
   [(1, [Fraction(-1, 1)], -1), (2, [Fraction(-2, 1)], -2), (3, [Fraction(-3, 1)], -3), (4, [Fraction(-4, 1)], -4), (5, [Fraction(-6, 1)], -6), (6, [Fraction(-7, 1)], -7), (7, [Fraction(-8, 1)], -8)]
p=13 (inert in K): (j, v([z'^{pj} z''^j]G_0) at the places of Q(sqrt7), predicted -j-v(j!))
   [(1, [Fraction(-1, 1)], -1), (2, [Fraction(-2, 1)], -2), (3, [Fraction(-3, 1)], -3)]
p=19 (inert in K): (j, v([z'^{pj} z''^j]G_0) at the places of Q(sqrt7), predicted -j-v(j!))
   [(1, [-1, -1], -1), (2, [-2, -2], -2)]
control p=11 (split), delta=(1,sqrt(-2),1): -2 is a QR mod 11: True
   [(1, [

**`t3_count.py`: `pa:product`, `pa:partner`, `pa:count`** with two places ($E=K(i)$, $F=\prod_k\theta((-i)^kz_1+i^kz_2)^4$, coefficients of $z_1^n$, $n\le160$). Predictions: for $p=5,13$ (inert, partner = other place) $v(n![z_1^n]F)\in[\gamma_pn-1,\gamma_pn+1]$ (the count and its sharpness); for $p=19$ (partner = same place) no divisibility ($v\in\{0,1\}$); for $p=3$ the lower bound 0; for $p=7$, $v\ge n/96-O(\log n)$. With the opposite partner rule the roles of $\{5,13\}$ and $\{19\}$ would be exchanged, so the test also confirms the sign.
*Refutation would be:* valuations outside these windows.

In [89]:
%%writefile N8/t3_count.py
# Two places: E=K(i), q_1=4, Phi={tau_1,tau_2}=Gal(E/K)=(Z/4)^x ; resolution u_k=i^k/4, multiplicity 4.
# F_0(z1,z2)=prod_{k<4} theta(((-i)^k z1 + i^k z2)/4)^4.  Scale c=4 (a unit at odd p): F(z)=F_0(4z)
#   F(z1,0)=theta(z1)^8 theta(i z1)^8 = exp(-16 sum_{m even} g_{2m} z^{2m}).
# Partner of tau at p^e is u(-p^e)tau: class of -p^e mod 4.
#  p inert in K, p=1 mod 4 (5,13): partner = other place for all odd e  -> count for degrees (n,0):
#       v(n! [z1^n]F) >= gamma_p n - O(log n),  gamma_p=p/(p^2-1)
#  p inert, p=3 mod 4 (3,19): partner = same place -> predicted divisibility 0.
#  p=7=d: partner: e odd same place, e even other place; Pred = (1/2) sum_{e even>=2} n/7^e = n/96.
# If the partner were u(+p^e) the roles of (5,13) and (3,19) would be exchanged.
from nf import *
M2=80
g=gseries(M2)
h=[S7(0)]*(M2+1)
for m in range(2,M2+1,2): h[m]=-16*g[m]
F=expser(h,M2)
def vfact(n,p): return sum(n//p**i for i in range(1,12))
for p,gam,label in [(5,Fr(5,24),"inert, partner=other"),(13,Fr(13,168),"inert, partner=other"),
                    (3,Fr(3,8),"inert, partner=same"),(19,Fr(19,360),"inert, partner=same"),(7,Fr(1,12),"ramified")]:
    rows=[]
    worst=None
    for m in range(2,M2+1,2):
        n=2*m
        if F[m].iszero(): continue
        v=vmin(F[m],p)+vfact(n,p)
        rows.append((n,v,float(gam*n)))
    print("p=%d (%s): (n, min_places v(n![z^n]F(z,0)), gamma_p*n)"%(p,label))
    print("   ",[(n,str(v),round(gn,2)) for n,v,gn in rows if n%16==0 or n<=24])
    print("    min over n of v - gamma*n = %.3f ; max over n of v = %s ; min of v/n over n>=40: %.4f (gamma=%.4f, 1/96=%.4f)"%(
        min(float(v)-gn for n,v,gn in rows), max(v for n,v,gn in rows), min(float(v)/n for n,v,gn in rows if n>=40), float(gam), 1/96))

Overwriting N8/t3_count.py


In [90]:
o = run("N8", "t3_count.py")
blocks = re.split(r"^p=", o, flags=re.M)[1:]
info = {}
for b in blocks:
    p = int(b.split()[0]); rows = eval(re.search(r"(\[\(.*\)\])", b).group(1))
    info[p] = [(n, int(v), g) for n, v, g in rows]
for p in (5, 13):
    assert all(g - 1 - 0.01 <= v <= g + 1 + 0.01 for n, v, g in info[p]), p
assert all(v in (0, 1) for n, v, g in info[19])
assert all(v >= 0 for n, v, g in info[3])
assert all(v >= n/96 - 2*__import__("math").log(n) for n, v, g in info[7])
record("N8", "pa:product, pa:partner, pa:count", "count of p-divisibility, sharp to +-1; partner sign", "E = K(i), n <= 160, p = 3,5,7,13,19", "PASS", "one variable direction; small degrees")

p=5 (inert, partner=other): (n, min_places v(n![z^n]F(z,0)), gamma_p*n)
    [(4, '0', 0.83), (8, '1', 1.67), (12, '2', 2.5), (16, '3', 3.33), (20, '5', 4.17), (24, '4', 5.0), (32, '6', 6.67), (48, '9', 10.0), (64, '13', 13.33), (80, '16', 16.67), (96, '19', 20.0), (112, '23', 23.33), (128, '26', 26.67), (144, '29', 30.0), (160, '33', 33.33)]
    min over n of v - gamma*n = -1.000 ; max over n of v = 33 ; min of v/n over n>=40: 0.1875 (gamma=0.2083, 1/96=0.0104)
p=13 (inert, partner=other): (n, min_places v(n![z^n]F(z,0)), gamma_p*n)
    [(4, '0', 0.31), (8, '0', 0.62), (12, '0', 0.93), (16, '1', 1.24), (20, '1', 1.55), (24, '1', 1.86), (32, '2', 2.48), (48, '4', 3.71), (64, '4', 4.95), (80, '6', 6.19), (96, '7', 7.43), (112, '8', 8.67), (128, '9', 9.9), (144, '11', 11.14), (160, '12', 12.38)]
    min over n of v - gamma*n = -0.976 ; max over n of v = 12 ; min of v/n over n>=40: 0.0625 (gamma=0.0774, 1/96=0.0104)
p=3 (inert, partner=same): (n, min_places v(n![z^n]F(z,0)), gamma_p*n)
   

**`t4_count2.py`: `pa:count` for two-variable coefficients** (total degree $\le44$): $\min_{(n_1,n_2)}[v(n_1!n_2![z_1^{n_1}z_2^{n_2}]F)-\mathrm{Pred}(n_1,n_2)]$ must be bounded below by minus the error term $f n'^3(1+\log_pB)$; observed $-4/5$ ($p=5$), $-12/13$ ($p=13$), $0$ ($p=3,19,7$; for $p=d=7$ the extra term $-\tfrac12\sum$ is included in Pred). *Refutation would be:* a deficit larger than the error term (the check below uses the stricter bound $1+\log_p D$ printed by the script).

In [91]:
%%writefile N8/t4_count2.py
# Two variables, same set-up as t3: F(z1,z2)=exp(-16 sum_m g_{2m} sum_{j=m mod 2} C(2m,j) z1^j z2^{2m-j}).
# Count (pa:count), shape = n_tau columns of height 1:  v(n1! n2! [z1^n1 z2^n2]F) >= Pred(n1,n2) - c(1+log_p B),
#   Pred = v(pi) sum_{e in I_p} sum_tau (n_tau/p^e - n_{pr_e tau})^+   (minus (1/2) sum_tau min(n_tau,n_{pr_0 tau}) for p=d)
from nf import *
from math import comb, log
import sys
D=int(sys.argv[1]) if len(sys.argv)>1 else 44     # total degree
g=gseries(D//2)
# exponent h[(a,b)]
h={}
for m in range(1,D//2+1):
    for j in range(0,2*m+1):
        if (j-m)%2==0:
            h[(j,2*m-j)]=-16*g[m]*comb(2*m,j)
# exp via  z1 dF/dz1 = (z1 dh/dz1) F ; do it by total degree using Euler: N F_N = sum (deg part of h)*...
F={(0,0):S7(1)}
hk=list(h.items())
for N in range(2,D+1,2):
    for a in range(0,N+1):
        b=N-a
        acc=S7(0)
        for (i,j),c in hk:
            if i<=a and j<=b and (i+j)<=N:
                f=F.get((a-i,b-j))
                if f is not None and not f.iszero():
                    acc=acc+(i+j)*c*f
        F[(a,b)]=acc/N
def vfact(n,p): return sum(n//p**i for i in range(1,12))
def pred(n1,n2,p):
    if p==7:
        s=Fr(0)
        e=1
        while 7**e<=n1+n2:
            if e%2==1: s+=max(Fr(n1,7**e)-n1,0)+max(Fr(n2,7**e)-n2,0)
            else: s+=max(Fr(n1,7**e)-n2,0)+max(Fr(n2,7**e)-n1,0)
            e+=1
        return s/2-Fr(min(n1,n2))        # e=0: partner = other place; (1/2)*(min(n1,n2)+min(n2,n1))
    swap=(p%4==1)
    s=Fr(0); e=1
    while p**e<=n1+n2:
        if swap: s+=max(Fr(n1,p**e)-n2,0)+max(Fr(n2,p**e)-n1,0)
        else: s+=max(Fr(n1,p**e)-n1,0)+max(Fr(n2,p**e)-n2,0)
        e+=2
    return s
for p in (5,13,3,19,7):
    lo=None; hi=None; arg=None; argh=None
    ex=[]
    for (a,b),c in F.items():
        if c.iszero() or a+b==0: continue
        v=vmin(c,p)+vfact(a,p)+vfact(b,p)
        d=v-pred(a,b,p)
        if lo is None or d<lo: lo=d; arg=(a,b,v)
        if hi is None or d>hi: hi=d; argh=(a,b,v)
        if a+b==D and a%6==0: ex.append((a,b,str(v),str(pred(a,b,p))))
    print("p=%d: min(v-Pred)=%s at %s ; max(v-Pred)=%s at %s ; 1+log_p(D)=%.2f"%(p,lo,arg,hi,argh,1+log(D)/log(p)))
    print("    samples (n1,n2,v,Pred):",ex)

Overwriting N8/t4_count2.py


In [92]:
o = run("N8", "t4_count2.py")
rows = re.findall(r"p=(\d+): min\(v-Pred\)=(-?[\d/]+) at .*?1\+log_p\(D\)=([\d.]+)", o)
print(rows)
assert len(rows) == 5 and all(float(Fraction(m)) >= -float(e) for p, m, e in rows)
assert all(float(Fraction(m)) > -1 for p, m, e in rows)
record("N8", "pa:count (two variables)", "v - Pred bounded below by the error term", "total degree <= 44, p = 3,5,7,13,19", "PASS (min deficit -12/13)", "two places, small degree")
record("N8", "psu:small, psu:arch", "-", "-", "NOT TESTED numerically", "transfer to the specific test form not testable in small cases")

p=5: min(v-Pred)=-4/5 at (1, 9, Fraction(0, 1)) ; max(v-Pred)=5 at (8, 28, Fraction(5, 1)) ; 1+log_p(D)=3.35
    samples (n1,n2,v,Pred): [(0, 44, '11', '44/5'), (6, 38, '2', '8/5'), (12, 32, '1', '0'), (18, 26, '2', '0'), (24, 20, '1', '0'), (30, 14, '2', '0'), (36, 8, '1', '0'), (42, 2, '6', '32/5')]
p=13: min(v-Pred)=-12/13 at (1, 25, Fraction(0, 1)) ; max(v-Pred)=1 at (2, 6, Fraction(1, 1)) ; 1+log_p(D)=2.48
    samples (n1,n2,v,Pred): [(0, 44, '3', '44/13'), (6, 38, '0', '0'), (12, 32, '0', '0'), (18, 26, '1', '0'), (24, 20, '0', '0'), (30, 14, '1', '0'), (36, 8, '0', '0'), (42, 2, '1', '16/13')]
p=3: min(v-Pred)=0 at (1, 1, 0) ; max(v-Pred)=11 at (6, 30, 11) ; 1+log_p(D)=4.44
    samples (n1,n2,v,Pred): [(0, 44, '4', '0'), (6, 38, '6', '0'), (12, 32, '5', '0'), (18, 26, '6', '0'), (24, 20, '4', '0'), (30, 14, '7', '0'), (36, 8, '4', '0'), (42, 2, '6', '0')]
p=19: min(v-Pred)=0 at (1, 1, 0) ; max(v-Pred)=2 at (19, 19, 2) ; 1+log_p(D)=2.29
    samples (n1,n2,v,Pred): [(0, 44, '0', '

## 5. Results table

The table is generated from the `record(...)` calls of the cells above: rows with outcome `PASS` were produced only by cells whose assertions held in this run. Rows marked `recorded only` or `NOT TESTED` are listed so that the coverage is visible.

In [93]:
from IPython.display import Markdown, display
extra = [
 dict(node="N1", label="ph:thm (in its range)", what="-", params="-", outcome="NOT TESTABLE", limit="hypothesis ph:eq-large requires astronomically large n; only the shape of the conclusion is tested far below the range"),
 dict(node="N2", label="au:seesaw", what="-", params="-", outcome="NOT TESTED (hand checks only)", limit="no numerical scripts"),
 dict(node="N3", label="ps:conj = pk:conj", what="-", params="-", outcome="partially (identities pinned for d <= 71)", limit="exact constant checked only for small imaginary quadratic fields"),
 dict(node="X1", label="pb:lattice (A/B junction)", what="analytic", params="-", outcome="hand verification", limit="interface only; Theorem B not verified"),
 dict(node="global", label="K1-K4", what="analytic knob tests", params="-", outcome="analytic (no false consequence found)", limit="not computations; evidence of design, not of correctness"),
]
rows = RESULTS + extra
hdr = "| Node | Statement | What was tested | Parameters | Outcome | Limit of the test |\n|---|---|---|---|---|---|\n"
body = "\n".join(f"| {r['node']} | `{r['label']}` | {r['what']} | {r['params']} | **{r['outcome']}** | {r['limit']} |" for r in rows)
display(Markdown(hdr + body))
npass = sum(r['outcome'].startswith('PASS') for r in RESULTS)
print(f"{npass} executed tests passed; {len(rows)-npass} rows are illustrations, recorded-only results, analytic arguments, or statements not tested.")
print(f"total runtime of the notebook up to here: {time.time()-T0:.0f} s")

| Node | Statement | What was tested | Parameters | Outcome | Limit of the test |
|---|---|---|---|---|---|
| global | `Euler-product falsifier` | Davenport–Heilbronn f: functional equation, zero off the line, non-multiplicative coefficients | f mod 5, zero near 0.8085+85.6993i | **illustration (computed)** | Illustrates the falsifier only; the proof's mechanism (L'/L prime sums, product formula) has no analogue for f, so the falsifier is VACUOUS, not passed |
| N1 | `ph:root` | least-weight monomials of largest stable subspace have def<=0 | k=2,N=2; p=5 deg<=8, p=3 deg<=7 | **PASS (791+966 cases, 0 violations)** | residue field only, k=2 |
| N1 | `ph:depth (k=2)` | sum of dims of depth subspaces <= sharp bound | k=2,N=2; p=5 deg<=8, p=3 deg<=7 | **PASS** | small n, far below ph:eq-large |
| N1 | `ph:root, ph:depth (full size)` | same | p=5 deg<=11, p=3 deg<=9 | **recorded only (2014+2077 cases, 0 violations; all depth sums OK; also re-run at full size when this notebook was prepared: same)** | set N1_T1_FULL=True to re-run |
| N1 | `ph:depth (k=3,4)` | depth sum <= sharp bound | (p,k,r,n)=(3,3,1,4-5),(5,3,1,5),(3,4,2,3),(3,4,1,3) | **PASS (5/5)** | n<=5, far below ph:eq-large |
| N1 | `pe:vol` | Monte Carlo cone share = prod (q-1)/(q^l-1) | 9 cases (k,N,P) up to (4,2,2), 4-8e6 samples | **PASS (max dev 2.0 s.e.)** | statistical; small k |
| N1 | `ph:thm (shape)` | mean elementary-divisor exponent of edge Hecke operator ~ n/P, below 3c_div | p=5,k=2,r=1,N=2, n in {1,2,4,8,12} | **PASS** | far below ph:eq-large where the theorem is asserted; one vertex; ph:thm itself is NOT testable in its range |
| N1 | `ph:thm (shape), large n` | same, n=16,20,24 | p=5, matrices up to 625x625 | **recorded only (means 2.64/3.08, 3.14/3.83, 3.81/4.93)** | far below ph:eq-large |
| N1 | `ph:count` | [G:K_r] dim S_c <= (1+theta)|B| at smallest admissible n | k=2,N=2; P in {5,7,13}, theta in {1,1/2,1/10}, n up to 48000 | **PASS (6/6; ratios 1.003-1.031)** | k=2 only (L = diagonal torus) |
| N2b | `par:eq-osc, par:casimir, par:relw, par:exist` | oscillator relations, Casimir identity, harmonic product of minors (sympy, exact) | 4 signatures; 60 matching pairs up to U(2,1)xU(3,2) | **PASS** | small signatures only |
| N2b | `par:unique` | space of pre-joint harmonics is 1-dimensional | 5 valid cases, degree <= deg Phi + 3 | **PASS (5/5)** | bounded degree; small signatures |
| N2b | `pco:eq-F` | matrix-coefficient formula on U(2,1): independence, equivariance, F(1)=1, decay, Casimir, closed form | 4 parameters + 2 holomorphic cases | **PASS** | U(2,1) only; quadrature |
| N2b | `pl:mir, pl:Uprop, pl:law, k_y` | level-N algebra at a split place | (n,N)=(2,3),(3,5),(3,7), 200 samples each | **PASS** | one place; small n |
| N3 | `pf:CS / (C4)` | Chowla–Selberg average equals c_CS - log(2 sqrt d) | d = 7, 23, 47, 71 | **PASS (diff <= 3e-40)** | class numbers 1-7 only |
| N3 | `(C4), pf:CS` | product of area*|Delta|^(1/6) over classes = (2 pi^2 e^c/sqrt d)^h | d = 7, 23, 47 | **PASS (diff <= 6e-44)** | small d |
| N3 | `pf:CS (unit)` | u^36 conjugates: symmetric functions 1222, 727653, -1; area*|u|^2 = target per class | d = 23 | **PASS** | one field |
| N3 | `pf:chi(b), pf:values, pf:recip` | torsion characters, units, Galois stability and integrality of values/jets, denominators | d = 7, 23; n = 3, 5 | **PASS** | two fields, n <= 5 |
| N3 | `pf:frame` | frame matrix is h*I with h from the formula | d = 7, q1 = 8 | **PASS (67*I)** | one case |
| N4a | `pbm masses, (C0), pbe:link(a), Ostrowski–Taussky, Hua, pbm:crude` | mass formula vs GHY, exponent identities, block determinant, inequalities | E=Q(zeta_12); random samples | **PASS** | one field for masses; identities on random samples |
| N4a | `pbm:pos(c), pbm:pair, pbm:first` | finite-group lemmas of (E4) by brute force vs hand values | GL_3(F_3) (m=1); GL_4(F_2) (m=2, partial check) | **PASS** | two tiny groups; hand values only for GL_3(F_3) |
| N4b | `pbo:section, pbo:sizes(c)` | section gamma_Z, formula M(Z,W), Gram determinant | 6 (rho,k) up to (3,4), ~190 random Z each | **PASS (1154 cases)** | small rho,k |
| N4b | `pff:siegelformula` | constant in Siegel's formula vs Gan–Hanke–Yu masses | E=Q(zeta_12), k=3,4,5 | **PASS (ratio 1 to 25+ digits)** | one field, full level |
| N4b | `pbr:degree, pbo:meanvalue, pbr:eq-unstable` | formal degree, c_G = k/pi, orbital identity on U(1,1), Beta(1,k) law | U(k,1), k<=5; U(1,1) | **PASS** | rank <= 6; scalar holomorphic series only |
| N4b | `prp:fibre, pfr:haar` | Haar-measure constant 4 pi^3 on U(1,1) | S = 1.5, 2, 4 | **PASS (ratios 1.00003-1.00009)** | one hyperbolic plane; quadrature |
| N4c | `pb:singular, pbm:chain, pbs:stab, pbe:scalar` | finite-field models: counting inequality, chain character, unique stabiliser element, descent | 5+3+3+2 cases over F_2..F_11 | **PASS** | split model; tiny groups |
| N4c | `pbo:sizes(c)` | isometry, frame matrix, norm bound 15P^6, derivative bound | (rho,k) = (1,1),(2,2),(3,2); 300 Z each | **PASS** | small rho,k |
| N4c | `pbv:haar, prp:heis, (Cv3)` | Jacobian 1 between eta_U parameters and row coordinates, left invariance | rho <= 3 | **PASS** | numerical Jacobians; rho <= 3 |
| N4c | `pbg:index` | - | - | **NOT TESTED** | no non-vacuous small case found; proof re-derived by hand only |
| N4 | `au:moment (Theorem B itself)` | - | - | **NOT TESTED** | rank r ~ x^5; no small instance |
| N5 | `ptr:match (a),(b); sign identity of squares` | combinatorics of the triangle | m <= 6, k <= 4 (signs); m <= 5, k in {3,5} (blocks) | **PASS** | small m,k |
| N5 | `pc:pointlaw, pc:next, ptv:fourier` | Fourier identities in a finite-field model | F_3, |N|,|T|,|T'| = 1 | **PASS** | q = 3, smallest block sizes |
| N5 | `au:factor, au:inj (constants with L-values)` | - | - | **NOT TESTED numerically** | checked by hand only (Rallis exponent, c_Q) |
| N6b | `pgl:gram (+ pgl:stat)` | exact Gram relations; w in U(J_Z) | 60 random configurations, m = 2,4,6 | **PASS (exact part)** | floating-point residuals of the second check are ~1e-2 absolute on entries up to ~1e6 (scale of the integers used) |
| N6b | `pbp:minorconj, pgl:rows, pgk:param` | minor expansion with error bound; row identity; composition law | 400 + 100 + 100 random cases | **PASS (max |r|/bound = 0.026)** | small matrices |
| N6b | `pgy:match, pgy:gap, pgy:T1, pgy:series` | block rule, gap, sign count of squares; local inequalities | (m,k) in {2,4}^2 | **PASS** | small m,k |
| N6 | `au:onelift` | - | - | **NOT TESTED (no numerical test)** | no small instance; hand consistency checks only |
| N7c | `Wach-module construction` | P phi(G) = G gamma(P) to precision; mod p vs mod p^c agreement | p=7, k=11 | **PASS (t1, t3, t4); t2 diagnostic** | construction sanity only |
| N7c | `pki:generic(i), rank two, small l` | crystalline extension classes span only the unit line; control detects tres ramifie | l = 7, 11, 13; k = l+3..l+5 | **PASS** | one family (V_{k,a}, v(a)=1); outside the paper's l range |
| N7c | `pki:generic(i), edges` | same, at the edges r = 2, 3, l-3, l-2 | l = 7, 11, 13 | **PASS (12/12)** | one family |
| N7c | `pki:generic(i), l = 41, remaining weights` | same | k in {62, 63} | **recorded only (NF rank 1, ratio 2)** | set N7C_KS to all four to re-run |
| N7c | `pki:generic(i), rank two, l = 41` | extension classes of a rank-two crystalline lattice lie in the unit line | l = 41, k in [61, 64] | **PASS** | rank two, one family (irreducible, v(a_l)=1); mod l |
| N7c | `integral generator` | liftable classes: dims 2,1,1 mod l,l^2,l^3; surviving class in unit line | l = 7, k = 10,11,12, X1 in {0,1,2} | **PASS** | small l; one family |
| N7 | `pg:I, pg:II, pg:III, pg:loss, pg:odd` | - | - | **NOT TESTED numerically** | hand tests in rank one only; rank-two test only for one family (above) |
| N8 | `model constants (pf:two)` | a1^2 = 3 sqrt 7, g2 = 1/sqrt 7 | d = 7, 60 digits + PSLQ | **PASS** | one curve |
| N8 | `pa:param, psm:param(d), psm:lambda, pa:units(a), pa:frob` | valuations of torsion coefficients; Frobenius polynomial identity | p = 3,5,7,11,13,19,23 | **PASS** | one CM curve (d = 7) |
| N8 | `psm:coupling` | exact valuation -j - v(j!) of coupling coefficients | p = 3 (j<=11), 5 (j<=7), 13 (j<=3), 19 (j<=2), split control p=11 | **PASS (26 coefficients)** | one place, Theta = theta |
| N8 | `pa:product, pa:partner, pa:count` | count of p-divisibility, sharp to +-1; partner sign | E = K(i), n <= 160, p = 3,5,7,13,19 | **PASS** | one variable direction; small degrees |
| N8 | `pa:count (two variables)` | v - Pred bounded below by the error term | total degree <= 44, p = 3,5,7,13,19 | **PASS (min deficit -12/13)** | two places, small degree |
| N8 | `psu:small, psu:arch` | - | - | **NOT TESTED numerically** | transfer to the specific test form not testable in small cases |
| N1 | `ph:thm (in its range)` | - | - | **NOT TESTABLE** | hypothesis ph:eq-large requires astronomically large n; only the shape of the conclusion is tested far below the range |
| N2 | `au:seesaw` | - | - | **NOT TESTED (hand checks only)** | no numerical scripts |
| N3 | `ps:conj = pk:conj` | - | - | **partially (identities pinned for d <= 71)** | exact constant checked only for small imaginary quadratic fields |
| X1 | `pb:lattice (A/B junction)` | analytic | - | **hand verification** | interface only; Theorem B not verified |
| global | `K1-K4` | analytic knob tests | - | **analytic (no false consequence found)** | not computations; evidence of design, not of correctness |

39 executed tests passed; 15 rows are illustrations, recorded-only results, analytic arguments, or statements not tested.
total runtime of the notebook up to here: 536 s


## 6. What these tests do not show

* **They do not prove anything.** Every numerical test checks a statement in a handful of small instances. A statement can hold in all small cases and fail in general.
* **They do not reach the theorems' range.** The load-bearing statements require ranks of order $x^5$, levels and moduli larger than $b^{20}$, degrees beyond `ph:eq-large`. None of these is reachable. For `ph:thm` (N1) only the shape of the conclusion was observed far below its range; the theorem itself, which the referee of the chain lists as the cheapest decisive point to settle, is untested where it is asserted.
* **Several load-bearing statements have no numerical test at all:** `au:seesaw` (N2), `au:moment` = Theorem B (N4; only auxiliary lemmas were tested), the constants of `au:factor`/`au:inj` (N5), `au:onelift` (N6: no numerical test was run), `pg:I`–`pg:III`, `pg:loss`, `pg:odd` (N7: hand tests in rank one, and a rank-two computer test for **one family** of representations only), `psu:small`/`psu:arch` (N8: only the counting statements beneath them), `pbg:index` (N4c: no non-vacuous small case).
* **The logical falsifiers are not numerical tests of the proof.** The Davenport–Heilbronn computation only illustrates the objection; the argument cannot be run on functions without Euler product, so that falsifier is vacuous. The knob tests K1–K4 are analytic arguments showing that some obvious variations do not yield false consequences — evidence of careful design, not of correctness.
* **They do not replace refereeing.** A decisive verification needs independent line-by-line refereeing by specialists of N1 (p-adic linear algebra and combinatorics), N4 (relative trace formula), N6–N8 (congruences, Selmer groups, integral p-adic Hodge theory at all primes), and a written lemma making explicit the compatibility of the two requirements on the level $N$ (K1).